In [2]:
import json
import os
import pandas as pd

RAW_DATA_PATH = "../data/raw"

files = [
    file for file in os.listdir(RAW_DATA_PATH)
    if file.endswith(".json")
]

print("Total JSON files:", len(files))

Total JSON files: 1243


In [3]:
from collections import defaultdict
import json
import os
import pandas as pd


def process_match(file_path):

    # -------------------------
    # LOAD MATCH
    # -------------------------

    with open(file_path, "r") as f:
        match = json.load(f)

    info = match["info"]

    # Match ID from filename
    match_id = os.path.splitext(os.path.basename(file_path))[0]

    # Basic match information
    date = str(info["dates"][0])
    season = info.get("season")
    venue = info.get("venue")
    teams = info["teams"]

    toss = info.get("toss", {})
    toss_winner = toss.get("winner")
    toss_decision = toss.get("decision")

    outcome = info.get("outcome", {})
    winner = outcome.get("winner")

    # -------------------------
    # PLAYER STATISTICS
    # -------------------------

    player_stats = defaultdict(lambda: {
        "runs": 0,
        "balls_faced": 0,
        "fours": 0,
        "sixes": 0,
        "runs_conceded": 0,
        "balls_bowled": 0,
        "wickets": 0,
        "dismissals": 0
    })

    # -------------------------
    # BATTER vs BOWLER
    # -------------------------

    matchup_stats = defaultdict(lambda: {
        "balls": 0,
        "runs": 0,
        "fours": 0,
        "sixes": 0,
        "dismissals": 0
    })

    # -------------------------
    # PROCESS BOTH INNINGS
    # -------------------------

    for innings in match["innings"]:

        batting_team = innings["team"]

        for over in innings["overs"]:

            for delivery in over["deliveries"]:

                batter = delivery["batter"]
                bowler = delivery["bowler"]
                runs = delivery["runs"]

                # -------------------------
                # BATTING
                # -------------------------

                player_stats[batter]["runs"] += runs["batter"]

                # Legal ball
                is_legal_ball = (
                    "extras" not in delivery
                    or (
                        "wides" not in delivery["extras"]
                        and "noballs" not in delivery["extras"]
                    )
                )

                if is_legal_ball:
                    player_stats[batter]["balls_faced"] += 1

                if runs["batter"] == 4:
                    player_stats[batter]["fours"] += 1

                if runs["batter"] == 6:
                    player_stats[batter]["sixes"] += 1

                # -------------------------
                # BOWLING
                # -------------------------

                if is_legal_ball:
                    player_stats[bowler]["balls_bowled"] += 1

                # Bowler runs conceded
                bowler_runs = runs["total"]

                if "extras" in delivery:
                    bowler_runs -= delivery["extras"].get("byes", 0)
                    bowler_runs -= delivery["extras"].get("legbyes", 0)

                player_stats[bowler]["runs_conceded"] += bowler_runs

                # -------------------------
                # WICKETS
                # -------------------------

                if "wickets" in delivery:

                    for wicket in delivery["wickets"]:

                        player_out = wicket["player_out"]
                        wicket_kind = wicket["kind"]

                        player_stats[player_out]["dismissals"] += 1

                        if wicket_kind not in [
                            "run out",
                            "retired hurt",
                            "retired out",
                            "obstructing the field"
                        ]:
                            player_stats[bowler]["wickets"] += 1

                            # Batter vs bowler dismissal
                            matchup_stats[
                                (batter, bowler)
                            ]["dismissals"] += 1

                # -------------------------
                # BATTER vs BOWLER
                # -------------------------

                if is_legal_ball:
                    matchup_stats[(batter, bowler)]["balls"] += 1

                matchup_stats[(batter, bowler)]["runs"] += runs["batter"]

                if runs["batter"] == 4:
                    matchup_stats[(batter, bowler)]["fours"] += 1

                if runs["batter"] == 6:
                    matchup_stats[(batter, bowler)]["sixes"] += 1

    # -------------------------
    # CREATE PLAYER ROWS
    # -------------------------

    player_rows = []

    for player, stats in player_stats.items():

        # Find player's team from Playing XI
        player_team = None

        for team in teams:
            if player in info["players"].get(team, []):
                player_team = team
                break

        player_rows.append({
            "match_id": match_id,
            "date": date,
            "season": season,
            "venue": venue,
            "player": player,
            "team": player_team,
            "runs": stats["runs"],
            "balls_faced": stats["balls_faced"],
            "fours": stats["fours"],
            "sixes": stats["sixes"],
            "runs_conceded": stats["runs_conceded"],
            "balls_bowled": stats["balls_bowled"],
            "wickets": stats["wickets"],
            "dismissals": stats["dismissals"],
            "winner": winner
        })

    # -------------------------
    # CREATE MATCHUP ROWS
    # -------------------------

    matchup_rows = []

    for (batter, bowler), stats in matchup_stats.items():

        matchup_rows.append({
            "match_id": match_id,
            "date": date,
            "season": season,
            "batter": batter,
            "bowler": bowler,
            "balls": stats["balls"],
            "runs": stats["runs"],
            "fours": stats["fours"],
            "sixes": stats["sixes"],
            "dismissals": stats["dismissals"]
        })

    return player_rows, matchup_rows

In [4]:
player_rows, matchup_rows = process_match(
    "../data/raw/" + files[0]
)

print("Player rows:", len(player_rows))
print("Matchup rows:", len(matchup_rows))

print(player_rows[:2])
print(matchup_rows[:2])

Player rows: 21
Matchup rows: 63
[{'match_id': '1082591', 'date': '2017-04-05', 'season': 2017, 'venue': 'Rajiv Gandhi International Stadium, Uppal', 'player': 'DA Warner', 'team': 'Sunrisers Hyderabad', 'runs': 14, 'balls_faced': 7, 'fours': 2, 'sixes': 1, 'runs_conceded': 0, 'balls_bowled': 0, 'wickets': 0, 'dismissals': 1, 'winner': 'Sunrisers Hyderabad'}, {'match_id': '1082591', 'date': '2017-04-05', 'season': 2017, 'venue': 'Rajiv Gandhi International Stadium, Uppal', 'player': 'TS Mills', 'team': 'Royal Challengers Bangalore', 'runs': 6, 'balls_faced': 3, 'fours': 0, 'sixes': 1, 'runs_conceded': 31, 'balls_bowled': 24, 'wickets': 1, 'dismissals': 1, 'winner': 'Sunrisers Hyderabad'}]
[{'match_id': '1082591', 'date': '2017-04-05', 'season': 2017, 'batter': 'DA Warner', 'bowler': 'TS Mills', 'balls': 4, 'runs': 4, 'fours': 1, 'sixes': 0, 'dismissals': 0}, {'match_id': '1082591', 'date': '2017-04-05', 'season': 2017, 'batter': 'S Dhawan', 'bowler': 'TS Mills', 'balls': 4, 'runs': 2, 

In [5]:
from collections import defaultdict
import json
import os
import pandas as pd


def process_match(file_path):

    # =========================
    # 1. LOAD MATCH
    # =========================

    with open(file_path, "r") as f:
        match = json.load(f)

    info = match["info"]

    # Match information
    match_id = os.path.splitext(os.path.basename(file_path))[0]
    date = str(info["dates"][0])
    season = info.get("season")
    venue = info.get("venue")

    teams = info["teams"]

    toss = info.get("toss", {})
    toss_winner = toss.get("winner")
    toss_decision = toss.get("decision")

    outcome = info.get("outcome", {})
    winner = outcome.get("winner")


    # =========================
    # 2. PLAYER STATISTICS
    # =========================

    player_stats = defaultdict(lambda: {
        "runs": 0,
        "balls_faced": 0,
        "fours": 0,
        "sixes": 0,
        "runs_conceded": 0,
        "balls_bowled": 0,
        "wickets": 0,
        "dismissals": 0
    })


    # =========================
    # 3. BATTER vs BOWLER
    # =========================

    matchup_stats = defaultdict(lambda: {
        "balls": 0,
        "runs": 0,
        "fours": 0,
        "sixes": 0,
        "dismissals": 0
    })


    # =========================
    # 4. PROCESS BOTH INNINGS
    # =========================

    for innings in match["innings"]:

        for over in innings["overs"]:

            for delivery in over["deliveries"]:

                batter = delivery["batter"]
                bowler = delivery["bowler"]

                runs = delivery["runs"]

                # -------------------------
                # Is this a legal delivery?
                # -------------------------

                is_legal_ball = (
                    "extras" not in delivery
                    or (
                        "wides" not in delivery["extras"]
                        and "noballs" not in delivery["extras"]
                    )
                )


                # =========================
                # BATTING
                # =========================

                # Batter runs
                player_stats[batter]["runs"] += runs["batter"]

                # Balls faced
                if is_legal_ball:
                    player_stats[batter]["balls_faced"] += 1

                # Fours
                if runs["batter"] == 4:
                    player_stats[batter]["fours"] += 1

                # Sixes
                if runs["batter"] == 6:
                    player_stats[batter]["sixes"] += 1


                # =========================
                # BOWLING
                # =========================

                # Balls bowled
                if is_legal_ball:
                    player_stats[bowler]["balls_bowled"] += 1

                # Runs conceded by bowler
                bowler_runs = runs["total"]

                if "extras" in delivery:

                    bowler_runs -= delivery["extras"].get("byes", 0)

                    bowler_runs -= delivery["extras"].get("legbyes", 0)

                player_stats[bowler]["runs_conceded"] += bowler_runs


                # =========================
                # WICKETS
                # =========================

                if "wickets" in delivery:

                    for wicket in delivery["wickets"]:

                        player_out = wicket["player_out"]

                        wicket_kind = wicket["kind"]

                        # Batter dismissed
                        player_stats[player_out]["dismissals"] += 1

                        # Bowler gets wicket
                        if wicket_kind not in [
                            "run out",
                            "retired hurt",
                            "retired out",
                            "obstructing the field"
                        ]:

                            player_stats[bowler]["wickets"] += 1

                            matchup_stats[
                                (batter, bowler)
                            ]["dismissals"] += 1


                # =========================
                # BATTER vs BOWLER
                # =========================

                matchup_stats[
                    (batter, bowler)
                ]["runs"] += runs["batter"]

                if is_legal_ball:

                    matchup_stats[
                        (batter, bowler)
                    ]["balls"] += 1

                if runs["batter"] == 4:

                    matchup_stats[
                        (batter, bowler)
                    ]["fours"] += 1

                if runs["batter"] == 6:

                    matchup_stats[
                        (batter, bowler)
                    ]["sixes"] += 1


    # =========================
    # 5. CREATE PLAYER ROWS
    # =========================

    player_rows = []

    # IMPORTANT:
    # Use Playing XI, not players who appeared
    # in deliveries.

    for team in teams:

        for player in info["players"][team]:

            stats = player_stats[player]

            player_rows.append({

                "match_id": match_id,

                "date": date,

                "season": season,

                "venue": venue,

                "player": player,

                "team": team,

                "runs": stats["runs"],

                "balls_faced": stats["balls_faced"],

                "fours": stats["fours"],

                "sixes": stats["sixes"],

                "runs_conceded": stats["runs_conceded"],

                "balls_bowled": stats["balls_bowled"],

                "wickets": stats["wickets"],

                "dismissals": stats["dismissals"],

                "winner": winner
            })


    # =========================
    # 6. CREATE MATCHUP ROWS
    # =========================

    matchup_rows = []

    for (batter, bowler), stats in matchup_stats.items():

        matchup_rows.append({

            "match_id": match_id,

            "date": date,

            "season": season,

            "batter": batter,

            "bowler": bowler,

            "balls": stats["balls"],

            "runs": stats["runs"],

            "fours": stats["fours"],

            "sixes": stats["sixes"],

            "dismissals": stats["dismissals"]
        })


    # =========================
    # 7. RETURN BOTH DATASETS
    # =========================

    return player_rows, matchup_rows

In [6]:
player_rows, matchup_rows = process_match(
    "../data/raw/" + files[0]
)

print("Player rows:", len(player_rows))
print("Matchup rows:", len(matchup_rows))

Player rows: 22
Matchup rows: 63


In [7]:
print(player_rows[:2])

[{'match_id': '1082591', 'date': '2017-04-05', 'season': 2017, 'venue': 'Rajiv Gandhi International Stadium, Uppal', 'player': 'DA Warner', 'team': 'Sunrisers Hyderabad', 'runs': 14, 'balls_faced': 7, 'fours': 2, 'sixes': 1, 'runs_conceded': 0, 'balls_bowled': 0, 'wickets': 0, 'dismissals': 1, 'winner': 'Sunrisers Hyderabad'}, {'match_id': '1082591', 'date': '2017-04-05', 'season': 2017, 'venue': 'Rajiv Gandhi International Stadium, Uppal', 'player': 'S Dhawan', 'team': 'Sunrisers Hyderabad', 'runs': 40, 'balls_faced': 31, 'fours': 5, 'sixes': 0, 'runs_conceded': 0, 'balls_bowled': 0, 'wickets': 0, 'dismissals': 1, 'winner': 'Sunrisers Hyderabad'}]


In [8]:
all_player_rows = []
all_matchup_rows = []

total_files = len(files)

for i, file in enumerate(files):

    file_path = os.path.join(RAW_DATA_PATH, file)

    try:
        player_rows, matchup_rows = process_match(file_path)

        all_player_rows.extend(player_rows)
        all_matchup_rows.extend(matchup_rows)

    except Exception as e:
        print(f"Error processing {file}: {e}")

    # Show progress every 100 matches
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{total_files} matches")


print("===================================")
print("Finished!")
print("Total player rows:", len(all_player_rows))
print("Total matchup rows:", len(all_matchup_rows))

Processed 100/1243 matches
Processed 200/1243 matches
Processed 300/1243 matches
Processed 400/1243 matches
Processed 500/1243 matches
Processed 600/1243 matches
Processed 700/1243 matches
Processed 800/1243 matches
Processed 900/1243 matches
Processed 1000/1243 matches
Processed 1100/1243 matches
Processed 1200/1243 matches
Finished!
Total player rows: 27909
Total matchup rows: 61429


In [9]:
player_df = pd.DataFrame(all_player_rows)

matchup_df = pd.DataFrame(all_matchup_rows)

print("Player dataset shape:", player_df.shape)
print("Matchup dataset shape:", matchup_df.shape)

Player dataset shape: (27909, 15)
Matchup dataset shape: (61429, 10)


In [10]:
os.makedirs("../data/processed", exist_ok=True)

player_df.to_csv(
    "../data/processed/player_match_stats.csv",
    index=False
)

matchup_df.to_csv(
    "../data/processed/batter_bowler_matchups.csv",
    index=False
)

print("Datasets saved successfully!")

Datasets saved successfully!


In [11]:
player_df = pd.read_csv(
    "../data/processed/player_match_stats.csv"
)

print(player_df.shape)
print(player_df.head())

(27909, 15)
   match_id        date season                                      venue  \
0   1082591  2017-04-05   2017  Rajiv Gandhi International Stadium, Uppal   
1   1082591  2017-04-05   2017  Rajiv Gandhi International Stadium, Uppal   
2   1082591  2017-04-05   2017  Rajiv Gandhi International Stadium, Uppal   
3   1082591  2017-04-05   2017  Rajiv Gandhi International Stadium, Uppal   
4   1082591  2017-04-05   2017  Rajiv Gandhi International Stadium, Uppal   

         player                 team  runs  balls_faced  fours  sixes  \
0     DA Warner  Sunrisers Hyderabad    14            7      2      1   
1      S Dhawan  Sunrisers Hyderabad    40           31      5      0   
2  MC Henriques  Sunrisers Hyderabad    52           37      3      2   
3  Yuvraj Singh  Sunrisers Hyderabad    62           27      7      3   
4      DJ Hooda  Sunrisers Hyderabad    16           12      0      1   

   runs_conceded  balls_bowled  wickets  dismissals               winner  
0          

In [12]:
player_df["date"] = pd.to_datetime(player_df["date"])

player_df = player_df.sort_values(
    ["player", "date", "match_id"]
).reset_index(drop=True)

player_df.head()

,match_id,date,season,venue,player,team,runs,balls_faced,fours,sixes,runs_conceded,balls_bowled,wickets,dismissals,winner
0,548341,2012-04-26,2012,Subrata Roy Sahara Stadium,A Ashish Reddy,Deccan Chargers,0,0,0,0,32,24,2,0,Deccan Chargers
1,548346,2012-04-29,2012,Wankhede Stadium,A Ashish Reddy,Deccan Chargers,10,10,0,1,11,13,1,1,Mumbai Indians
2,548348,2012-05-01,2012,Barabati Stadium,A Ashish Reddy,Deccan Chargers,0,0,0,0,32,18,1,0,Deccan Chargers
3,548352,2012-05-04,2012,"MA Chidambaram Stadium, Chepauk",A Ashish Reddy,Deccan Chargers,3,3,0,0,16,12,1,1,Chennai Super Kings
4,548356,2012-05-06,2012,M Chinnaswamy Stadium,A Ashish Reddy,Deccan Chargers,0,0,0,0,36,24,1,0,Royal Challengers Bangalore


In [13]:
player_df["previous_runs"] = (
    player_df.groupby("player")["runs"]
    .cumsum()
    .shift(1)
    .fillna(0)
)

In [14]:
player_df["previous_wickets"] = (
    player_df.groupby("player")["wickets"]
    .cumsum()
    .shift(1)
    .fillna(0)
)

In [15]:
player_df["previous_dismissals"] = (
    player_df.groupby("player")["dismissals"]
    .cumsum()
    .shift(1)
    .fillna(0)
)

In [16]:
print(
    player_df[
        player_df["player"] == "DA Warner"
    ][
        [
            "date",
            "runs",
            "previous_runs",
            "wickets",
            "previous_wickets"
        ]
    ].head(10)
)

           date  runs  previous_runs  wickets  previous_wickets
5020 2009-05-02    51            6.0        0               2.0
5021 2009-05-05    36           51.0        0               0.0
5022 2009-05-08    21           87.0        0               0.0
5023 2009-05-10    36          108.0        0               0.0
5024 2009-05-13     4          144.0        0               0.0
5025 2009-05-21    15          148.0        0               0.0
5026 2009-05-22     0          163.0        0               0.0
5027 2010-03-19     6          163.0        0               0.0
5028 2010-03-21    57          169.0        0               0.0
5029 2010-03-25    33          226.0        0               0.0


In [17]:
# Previous cumulative runs
player_df["previous_runs"] = (
    player_df.groupby("player")["runs"]
    .cumsum()
    .groupby(player_df["player"])
    .shift(1)
    .fillna(0)
)

# Previous cumulative wickets
player_df["previous_wickets"] = (
    player_df.groupby("player")["wickets"]
    .cumsum()
    .groupby(player_df["player"])
    .shift(1)
    .fillna(0)
)

# Previous cumulative dismissals
player_df["previous_dismissals"] = (
    player_df.groupby("player")["dismissals"]
    .cumsum()
    .groupby(player_df["player"])
    .shift(1)
    .fillna(0)
)

In [18]:
print(
    player_df[
        player_df["player"] == "A Ashish Reddy"
    ][
        [
            "date",
            "runs",
            "previous_runs",
            "wickets",
            "previous_wickets",
            "dismissals",
            "previous_dismissals"
        ]
    ].head(10)
)

        date  runs  previous_runs  wickets  previous_wickets  dismissals  \
0 2012-04-26     0            0.0        2               0.0           0   
1 2012-04-29    10            0.0        1               2.0           1   
2 2012-05-01     0           10.0        1               3.0           0   
3 2012-05-04     3           10.0        1               4.0           1   
4 2012-05-06     0           13.0        1               5.0           0   
5 2012-05-08     8           13.0        2               6.0           1   
6 2012-05-10     0           21.0        0               8.0           0   
7 2012-05-18    10           21.0        0               8.0           0   
8 2012-05-20     4           31.0        3               8.0           1   
9 2013-04-05     7           35.0        1              11.0           0   

   previous_dismissals  
0                  0.0  
1                  0.0  
2                  1.0  
3                  1.0  
4                  2.0  
5            

In [19]:
player_df["last_5_runs"] = (
    player_df.groupby("player")["runs"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .fillna(0)
)

In [20]:
player_df["last_10_runs"] = (
    player_df.groupby("player")["runs"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
    .fillna(0)
)

In [21]:
player_df["last_5_wickets"] = (
    player_df.groupby("player")["wickets"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
)

player_df["last_10_wickets"] = (
    player_df.groupby("player")["wickets"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).sum())
    .fillna(0)
)

In [22]:
print(
    player_df[
        player_df["player"] == "A Ashish Reddy"
    ][
        [
            "date",
            "runs",
            "previous_runs",
            "last_5_runs",
            "last_10_runs",
            "wickets",
            "last_5_wickets",
            "last_10_wickets"
        ]
    ].head(15)
)

         date  runs  previous_runs  last_5_runs  last_10_runs  wickets  \
0  2012-04-26     0            0.0     0.000000      0.000000        2   
1  2012-04-29    10            0.0     0.000000      0.000000        1   
2  2012-05-01     0           10.0     5.000000      5.000000        1   
3  2012-05-04     3           10.0     3.333333      3.333333        1   
4  2012-05-06     0           13.0     3.250000      3.250000        1   
5  2012-05-08     8           13.0     2.600000      2.600000        2   
6  2012-05-10     0           21.0     4.200000      3.500000        0   
7  2012-05-18    10           21.0     2.200000      3.000000        0   
8  2012-05-20     4           31.0     4.200000      3.875000        3   
9  2013-04-05     7           35.0     4.400000      3.888889        1   
10 2013-04-07    14           42.0     5.800000      4.200000        1   
11 2013-04-09     3           56.0     7.000000      5.600000        0   
12 2013-04-12    16           59.0    

In [23]:
# Previous cumulative balls faced
player_df["previous_balls_faced"] = (
    player_df.groupby("player")["balls_faced"]
    .cumsum()
    .groupby(player_df["player"])
    .shift(1)
    .fillna(0)
)

# Previous strike rate
player_df["previous_strike_rate"] = (
    player_df["previous_runs"]
    / player_df["previous_balls_faced"]
    * 100
)

# Replace infinity and missing values
player_df["previous_strike_rate"] = (
    player_df["previous_strike_rate"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

In [24]:
# Previous 5 matches total runs
player_df["last_5_total_runs"] = (
    player_df.groupby("player")["runs"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
)

# Previous 5 matches total balls
player_df["last_5_total_balls"] = (
    player_df.groupby("player")["balls_faced"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
)

# Last 5 strike rate
player_df["last_5_strike_rate"] = (
    player_df["last_5_total_runs"]
    / player_df["last_5_total_balls"]
    * 100
)

player_df["last_5_strike_rate"] = (
    player_df["last_5_strike_rate"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

In [25]:
print(
    player_df[
        player_df["player"] == "A Ashish Reddy"
    ][
        [
            "date",
            "runs",
            "balls_faced",
            "previous_runs",
            "previous_balls_faced",
            "previous_strike_rate",
            "last_5_runs",
            "last_5_strike_rate"
        ]
    ].head(15)
)

         date  runs  balls_faced  previous_runs  previous_balls_faced  \
0  2012-04-26     0            0            0.0                   0.0   
1  2012-04-29    10           10            0.0                   0.0   
2  2012-05-01     0            0           10.0                  10.0   
3  2012-05-04     3            3           10.0                  10.0   
4  2012-05-06     0            0           13.0                  13.0   
5  2012-05-08     8            8           13.0                  13.0   
6  2012-05-10     0            0           21.0                  21.0   
7  2012-05-18    10            4           21.0                  21.0   
8  2012-05-20     4            4           31.0                  25.0   
9  2013-04-05     7            4           35.0                  29.0   
10 2013-04-07    14           12           42.0                  33.0   
11 2013-04-09     3            4           56.0                  45.0   
12 2013-04-12    16            9           59.0    

In [26]:
# ==========================================
# PREVIOUS CUMULATIVE BOWLING STATISTICS
# ==========================================

# Previous total runs conceded
player_df["previous_runs_conceded"] = (
    player_df.groupby("player")["runs_conceded"]
    .cumsum()
    .groupby(player_df["player"])
    .shift(1)
    .fillna(0)
)

# Previous total balls bowled
player_df["previous_balls_bowled"] = (
    player_df.groupby("player")["balls_bowled"]
    .cumsum()
    .groupby(player_df["player"])
    .shift(1)
    .fillna(0)
)

# Previous economy
player_df["previous_economy"] = (
    player_df["previous_runs_conceded"]
    / player_df["previous_balls_bowled"]
    * 6
)

player_df["previous_economy"] = (
    player_df["previous_economy"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)


# ==========================================
# LAST 5 BOWLING STATISTICS
# ==========================================

# Last 5 matches - runs conceded
player_df["last_5_runs_conceded"] = (
    player_df.groupby("player")["runs_conceded"]
    .transform(
        lambda x: x.shift(1)
        .rolling(5, min_periods=1)
        .sum()
    )
    .fillna(0)
)

# Last 5 matches - balls bowled
player_df["last_5_balls_bowled"] = (
    player_df.groupby("player")["balls_bowled"]
    .transform(
        lambda x: x.shift(1)
        .rolling(5, min_periods=1)
        .sum()
    )
    .fillna(0)
)

# Last 5 economy
player_df["last_5_economy"] = (
    player_df["last_5_runs_conceded"]
    / player_df["last_5_balls_bowled"]
    * 6
)

player_df["last_5_economy"] = (
    player_df["last_5_economy"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

In [27]:
print(
    player_df[
        player_df["player"] == "TS Mills"
    ][
        [
            "date",
            "runs_conceded",
            "balls_bowled",
            "wickets",
            "previous_runs_conceded",
            "previous_balls_bowled",
            "previous_economy",
            "last_5_runs_conceded",
            "last_5_balls_bowled",
            "last_5_economy"
        ]
    ].head(10)
)

            date  runs_conceded  balls_bowled  wickets  \
25486 2017-04-05             31            24        1   
25487 2017-04-08             33            24        1   
25488 2017-04-10             22            12        1   
25489 2017-04-14             36            23        0   
25490 2017-04-23             31            24        2   
25491 2022-03-27             26            18        1   
25492 2022-04-02             35            24        3   
25493 2022-04-06             38            18        2   
25494 2022-04-13             37            24        0   
25495 2022-04-16             54            18        0   

       previous_runs_conceded  previous_balls_bowled  previous_economy  \
25486                     0.0                    0.0          0.000000   
25487                    31.0                   24.0          7.750000   
25488                    64.0                   48.0          8.000000   
25489                    86.0                   60.0          8.6

In [28]:
player_df = player_df.sort_values(
    ["player", "date", "match_id"]
).reset_index(drop=True)

In [29]:
player_df["venue_previous_runs"] = (
    player_df
    .groupby(["player", "venue"])["runs"]
    .cumsum()
    .groupby([
        player_df["player"],
        player_df["venue"]
    ])
    .shift(1)
    .fillna(0)
)

In [30]:
player_df["venue_previous_balls"] = (
    player_df
    .groupby(["player", "venue"])["balls_faced"]
    .cumsum()
    .groupby([
        player_df["player"],
        player_df["venue"]
    ])
    .shift(1)
    .fillna(0)
)

In [31]:
player_df["venue_previous_strike_rate"] = (
    player_df["venue_previous_runs"]
    / player_df["venue_previous_balls"]
    * 100
)

player_df["venue_previous_strike_rate"] = (
    player_df["venue_previous_strike_rate"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

In [32]:
player_df["venue_previous_wickets"] = (
    player_df
    .groupby(["player", "venue"])["wickets"]
    .cumsum()
    .groupby([
        player_df["player"],
        player_df["venue"]
    ])
    .shift(1)
    .fillna(0)
)

In [33]:
print(
    player_df[
        player_df["player"] == "DA Warner"
    ][
        [
            "date",
            "venue",
            "runs",
            "venue_previous_runs",
            "balls_faced",
            "venue_previous_balls",
            "venue_previous_strike_rate",
            "wickets",
            "venue_previous_wickets"
        ]
    ].head(15)
)

           date                  venue  runs  venue_previous_runs  \
5020 2009-05-02  New Wanderers Stadium    51                  0.0   
5021 2009-05-05              Kingsmead    36                  0.0   
5022 2009-05-08           Buffalo Park    21                  0.0   
5023 2009-05-10  New Wanderers Stadium    36                 51.0   
5024 2009-05-13              Kingsmead     4                 36.0   
5025 2009-05-21        SuperSport Park    15                  0.0   
5026 2009-05-22        SuperSport Park     0                 15.0   
5027 2010-03-19       Feroz Shah Kotla     6                  0.0   
5028 2010-03-21       Barabati Stadium    57                  0.0   
5029 2010-03-25  M Chinnaswamy Stadium    33                  0.0   
5030 2010-03-29       Feroz Shah Kotla   107                  6.0   
5031 2010-03-31       Feroz Shah Kotla     4                113.0   
5032 2010-04-04       Feroz Shah Kotla    33                117.0   
5033 2010-04-07           Eden Gar

In [35]:
matchup_df = pd.read_csv(
    "../data/processed/batter_bowler_matchups.csv"
)

print(matchup_df.shape)
print(matchup_df.head())

(61429, 10)
   match_id        date season        batter       bowler  balls  runs  fours  \
0   1082591  2017-04-05   2017     DA Warner     TS Mills      4     4      1   
1   1082591  2017-04-05   2017      S Dhawan     TS Mills      4     2      0   
2   1082591  2017-04-05   2017      S Dhawan  A Choudhary      5     3      0   
3   1082591  2017-04-05   2017     DA Warner  A Choudhary      3    10      1   
4   1082591  2017-04-05   2017  MC Henriques  A Choudhary      4     6      1   

   sixes  dismissals  
0      0           0  
1      0           0  
2      0           0  
3      1           1  
4      0           0  


In [36]:
matchup_df["date"] = pd.to_datetime(
    matchup_df["date"]
)

matchup_df = matchup_df.sort_values(
    ["batter", "bowler", "date", "match_id"]
).reset_index(drop=True)

In [37]:
matchup_df["previous_balls"] = (
    matchup_df
    .groupby(["batter", "bowler"])["balls"]
    .cumsum()
    .groupby([
        matchup_df["batter"],
        matchup_df["bowler"]
    ])
    .shift(1)
    .fillna(0)
)

In [38]:
matchup_df["previous_runs"] = (
    matchup_df
    .groupby(["batter", "bowler"])["runs"]
    .cumsum()
    .groupby([
        matchup_df["batter"],
        matchup_df["bowler"]
    ])
    .shift(1)
    .fillna(0)
)

In [39]:
matchup_df["previous_dismissals"] = (
    matchup_df
    .groupby(["batter", "bowler"])["dismissals"]
    .cumsum()
    .groupby([
        matchup_df["batter"],
        matchup_df["bowler"]
    ])
    .shift(1)
    .fillna(0)
)

In [40]:
matchup_df["previous_fours"] = (
    matchup_df
    .groupby(["batter", "bowler"])["fours"]
    .cumsum()
    .groupby([
        matchup_df["batter"],
        matchup_df["bowler"]
    ])
    .shift(1)
    .fillna(0)
)

matchup_df["previous_sixes"] = (
    matchup_df
    .groupby(["batter", "bowler"])["sixes"]
    .cumsum()
    .groupby([
        matchup_df["batter"],
        matchup_df["bowler"]
    ])
    .shift(1)
    .fillna(0)
)

In [41]:
matchup_df["previous_strike_rate"] = (
    matchup_df["previous_runs"]
    / matchup_df["previous_balls"]
    * 100
)

matchup_df["previous_strike_rate"] = (
    matchup_df["previous_strike_rate"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

In [42]:
print(
    matchup_df[
        matchup_df["previous_balls"] > 0
    ][
        [
            "date",
            "batter",
            "bowler",
            "balls",
            "runs",
            "previous_balls",
            "previous_runs",
            "previous_dismissals",
            "previous_strike_rate"
        ]
    ].head(20)
)

         date          batter              bowler  balls  runs  \
1  2015-05-02  A Ashish Reddy             A Nehra      4     2   
3  2013-04-17  A Ashish Reddy            AB Dinda      4     3   
5  2015-04-18  A Ashish Reddy          AD Mathews      4    12   
12 2015-04-16  A Ashish Reddy           CH Morris      1     1   
14 2013-04-25  A Ashish Reddy            DJ Bravo     10    19   
15 2015-05-02  A Ashish Reddy            DJ Bravo      2     4   
26 2016-04-16  A Ashish Reddy            M Morkel      1     1   
39 2013-04-07  A Ashish Reddy       R Vinay Kumar      4     8   
40 2013-04-09  A Ashish Reddy       R Vinay Kumar      2     2   
47 2016-04-30  A Ashish Reddy           SR Watson      3     2   
53 2016-04-16  A Ashish Reddy            UT Yadav      4     8   
57 2024-05-05        A Badoni          AD Russell      6    12   
59 2025-03-24        A Badoni            AR Patel      3     3   
60 2026-04-01        A Badoni            AR Patel      1     0   
63 2026-04

In [43]:
matchup_df["last_5_runs"] = (
    matchup_df
    .groupby(["batter", "bowler"])["runs"]
    .transform(
        lambda x: x.shift(1)
        .rolling(5, min_periods=1)
        .sum()
    )
    .fillna(0)
)

In [44]:
matchup_df["last_5_balls"] = (
    matchup_df
    .groupby(["batter", "bowler"])["balls"]
    .transform(
        lambda x: x.shift(1)
        .rolling(5, min_periods=1)
        .sum()
    )
    .fillna(0)
)

In [45]:
matchup_df["last_5_strike_rate"] = (
    matchup_df["last_5_runs"]
    / matchup_df["last_5_balls"]
    * 100
)

matchup_df["last_5_strike_rate"] = (
    matchup_df["last_5_strike_rate"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

In [46]:
matchup_df["last_5_dismissals"] = (
    matchup_df
    .groupby(["batter", "bowler"])["dismissals"]
    .transform(
        lambda x: x.shift(1)
        .rolling(5, min_periods=1)
        .sum()
    )
    .fillna(0)
)

In [47]:
print(
    matchup_df[
        matchup_df["batter"] == "A Ashish Reddy"
    ][
        [
            "date",
            "batter",
            "bowler",
            "balls",
            "runs",
            "previous_balls",
            "previous_runs",
            "previous_strike_rate",
            "last_5_runs",
            "last_5_balls",
            "last_5_strike_rate",
            "last_5_dismissals"
        ]
    ].head(20)
)

         date          batter           bowler  balls  runs  previous_balls  \
0  2013-05-04  A Ashish Reddy          A Nehra      4     5             0.0   
1  2015-05-02  A Ashish Reddy          A Nehra      4     2             4.0   
2  2013-04-05  A Ashish Reddy         AB Dinda      3     6             0.0   
3  2013-04-17  A Ashish Reddy         AB Dinda      4     3             3.0   
4  2013-04-17  A Ashish Reddy       AD Mathews      8    13             0.0   
5  2015-04-18  A Ashish Reddy       AD Mathews      4    12             8.0   
6  2016-04-16  A Ashish Reddy       AD Russell      3     4             0.0   
7  2015-04-27  A Ashish Reddy    Anureet Singh      2     2             0.0   
8  2013-04-19  A Ashish Reddy    Azhar Mahmood      3     2             0.0   
9  2013-04-05  A Ashish Reddy          B Kumar      1     1             0.0   
10 2012-05-04  A Ashish Reddy    BW Hilfenhaus      2     2             0.0   
11 2013-04-25  A Ashish Reddy        CH Morris      

In [48]:
player_df["batting_experience"] = player_df["previous_runs"]

player_df["batting_form"] = player_df["last_5_runs"]

player_df["batting_strike_rate"] = player_df["previous_strike_rate"]

player_df["recent_strike_rate"] = player_df["last_5_strike_rate"]

player_df["bowling_experience"] = player_df["previous_wickets"]

player_df["bowling_form"] = player_df["last_5_wickets"]

player_df["bowling_economy"] = player_df["previous_economy"]

player_df["recent_bowling_economy"] = player_df["last_5_economy"]

player_df["venue_runs"] = player_df["venue_previous_runs"]

player_df["venue_strike_rate"] = player_df["venue_previous_strike_rate"]

player_df["venue_wickets"] = player_df["venue_previous_wickets"]

In [49]:
team_features = (
    player_df
    .groupby(["match_id", "team"])
    .agg(
        batting_experience=("batting_experience", "sum"),
        batting_form=("batting_form", "mean"),
        batting_strike_rate=("batting_strike_rate", "mean"),
        recent_strike_rate=("recent_strike_rate", "mean"),
        
        bowling_experience=("bowling_experience", "sum"),
        bowling_form=("bowling_form", "sum"),
        bowling_economy=("bowling_economy", "mean"),
        recent_bowling_economy=("recent_bowling_economy", "mean"),
        
        venue_runs=("venue_runs", "sum"),
        venue_strike_rate=("venue_strike_rate", "mean"),
        venue_wickets=("venue_wickets", "sum")
    )
    .reset_index()
)

In [50]:
print(team_features.shape)
print(team_features.head(10))

(2486, 13)
   match_id                         team  batting_experience  batting_form  \
0    335982        Kolkata Knight Riders                 0.0      0.000000   
1    335982  Royal Challengers Bangalore                 0.0      0.000000   
2    335983          Chennai Super Kings                 0.0      0.000000   
3    335983              Kings XI Punjab                 0.0      0.000000   
4    335984             Delhi Daredevils                 0.0      0.000000   
5    335984             Rajasthan Royals                 0.0      0.000000   
6    335985               Mumbai Indians                 0.0      0.000000   
7    335985  Royal Challengers Bangalore                42.0      3.818182   
8    335986              Deccan Chargers                 0.0      0.000000   
9    335986        Kolkata Knight Riders               205.0     18.636364   

   batting_strike_rate  recent_strike_rate  bowling_experience  bowling_form  \
0             0.000000            0.000000        

In [51]:
match_rows = []

for file in files:

    file_path = os.path.join(RAW_DATA_PATH, file)

    try:
        with open(file_path, "r") as f:
            match = json.load(f)

        info = match["info"]

        teams = info["teams"]
        outcome = info["outcome"]

        match_rows.append({
            "match_id": str(os.path.splitext(file)[0]),
            "date": info["dates"][0],
            "season": info["season"],
            "venue": info["venue"],
            "team_1": teams[0],
            "team_2": teams[1],
            "toss_winner": info["toss"]["winner"],
            "toss_decision": info["toss"]["decision"],
            "winner": outcome.get("winner")
        })

    except Exception as e:
        print("Error:", file, e)

match_info_df = pd.DataFrame(match_rows)

print(match_info_df.shape)
print(match_info_df.head())

(1243, 9)
  match_id        date season                                      venue  \
0  1082591  2017-04-05   2017  Rajiv Gandhi International Stadium, Uppal   
1  1082592  2017-04-06   2017    Maharashtra Cricket Association Stadium   
2  1082593  2017-04-07   2017     Saurashtra Cricket Association Stadium   
3  1082594  2017-04-08   2017                     Holkar Cricket Stadium   
4  1082595  2017-04-08   2017                      M.Chinnaswamy Stadium   

                        team_1                       team_2  \
0          Sunrisers Hyderabad  Royal Challengers Bangalore   
1       Rising Pune Supergiant               Mumbai Indians   
2                Gujarat Lions        Kolkata Knight Riders   
3              Kings XI Punjab       Rising Pune Supergiant   
4  Royal Challengers Bangalore             Delhi Daredevils   

                   toss_winner toss_decision                       winner  
0  Royal Challengers Bangalore         field          Sunrisers Hyderabad  
1 

In [52]:
team1_features = team_features.rename(
    columns={
        "team": "team_1",
        "batting_experience": "team1_batting_experience",
        "batting_form": "team1_batting_form",
        "batting_strike_rate": "team1_batting_strike_rate",
        "recent_strike_rate": "team1_recent_strike_rate",
        "bowling_experience": "team1_bowling_experience",
        "bowling_form": "team1_bowling_form",
        "bowling_economy": "team1_bowling_economy",
        "recent_bowling_economy": "team1_recent_bowling_economy",
        "venue_runs": "team1_venue_runs",
        "venue_strike_rate": "team1_venue_strike_rate",
        "venue_wickets": "team1_venue_wickets"
    }
)

team2_features = team_features.rename(
    columns={
        "team": "team_2",
        "batting_experience": "team2_batting_experience",
        "batting_form": "team2_batting_form",
        "batting_strike_rate": "team2_batting_strike_rate",
        "recent_strike_rate": "team2_recent_strike_rate",
        "bowling_experience": "team2_bowling_experience",
        "bowling_form": "team2_bowling_form",
        "bowling_economy": "team2_bowling_economy",
        "recent_bowling_economy": "team2_recent_bowling_economy",
        "venue_runs": "team2_venue_runs",
        "venue_strike_rate": "team2_venue_strike_rate",
        "venue_wickets": "team2_venue_wickets"
    }
)

In [54]:
match_ml = match_info_df.merge(
    team1_features,
    on=["match_id", "team_1"],
    how="left"
)

ValueError: You are trying to merge on str and int64 columns for key 'match_id'. If you wish to proceed you should use pd.concat

In [55]:
# Make match_id the same type everywhere
match_info_df["match_id"] = match_info_df["match_id"].astype(str)

team_features["match_id"] = team_features["match_id"].astype(str)

matchup_df["match_id"] = matchup_df["match_id"].astype(str)

print(match_info_df["match_id"].dtype)
print(team_features["match_id"].dtype)
print(matchup_df["match_id"].dtype)

str
str
str


In [56]:
# ==========================================
# BUILD MATCH-LEVEL ML DATASET
# ==========================================

# Team 1 features
team1_features = team_features.rename(
    columns={
        "team": "team_1",
        "batting_experience": "team1_batting_experience",
        "batting_form": "team1_batting_form",
        "batting_strike_rate": "team1_batting_strike_rate",
        "recent_strike_rate": "team1_recent_strike_rate",
        "bowling_experience": "team1_bowling_experience",
        "bowling_form": "team1_bowling_form",
        "bowling_economy": "team1_bowling_economy",
        "recent_bowling_economy": "team1_recent_bowling_economy",
        "venue_runs": "team1_venue_runs",
        "venue_strike_rate": "team1_venue_strike_rate",
        "venue_wickets": "team1_venue_wickets"
    }
)

# Team 2 features
team2_features = team_features.rename(
    columns={
        "team": "team_2",
        "batting_experience": "team2_batting_experience",
        "batting_form": "team2_batting_form",
        "batting_strike_rate": "team2_batting_strike_rate",
        "recent_strike_rate": "team2_recent_strike_rate",
        "bowling_experience": "team2_bowling_experience",
        "bowling_form": "team2_bowling_form",
        "bowling_economy": "team2_bowling_economy",
        "recent_bowling_economy": "team2_recent_bowling_economy",
        "venue_runs": "team2_venue_runs",
        "venue_strike_rate": "team2_venue_strike_rate",
        "venue_wickets": "team2_venue_wickets"
    }
)

# Make sure match_id types are consistent
team1_features["match_id"] = team1_features["match_id"].astype(str)
team2_features["match_id"] = team2_features["match_id"].astype(str)
match_info_df["match_id"] = match_info_df["match_id"].astype(str)

# Merge Team 1
match_ml = match_info_df.merge(
    team1_features,
    on=["match_id", "team_1"],
    how="left"
)

# Merge Team 2
match_ml = match_ml.merge(
    team2_features,
    on=["match_id", "team_2"],
    how="left"
)

# Target: 1 if Team 1 won, otherwise 0
match_ml["team1_win"] = (
    match_ml["winner"] == match_ml["team_1"]
).astype(int)

print("Shape:", match_ml.shape)

print(
    match_ml[
        [
            "match_id",
            "team_1",
            "team_2",
            "winner",
            "team1_win"
        ]
    ].head(10)
)

Shape: (1243, 32)
  match_id                       team_1                       team_2  \
0  1082591          Sunrisers Hyderabad  Royal Challengers Bangalore   
1  1082592       Rising Pune Supergiant               Mumbai Indians   
2  1082593                Gujarat Lions        Kolkata Knight Riders   
3  1082594              Kings XI Punjab       Rising Pune Supergiant   
4  1082595  Royal Challengers Bangalore             Delhi Daredevils   
5  1082596          Sunrisers Hyderabad                Gujarat Lions   
6  1082597               Mumbai Indians        Kolkata Knight Riders   
7  1082598              Kings XI Punjab  Royal Challengers Bangalore   
8  1082599       Rising Pune Supergiant             Delhi Daredevils   
9  1082600               Mumbai Indians          Sunrisers Hyderabad   

                        winner  team1_win  
0          Sunrisers Hyderabad          1  
1       Rising Pune Supergiant          1  
2        Kolkata Knight Riders          0  
3            

In [57]:
playing_xi = (
    player_df
    .groupby(["match_id", "team"])["player"]
    .apply(list)
    .reset_index()
)

print(playing_xi.head())

   match_id                         team  \
0    335982        Kolkata Knight Riders   
1    335982  Royal Challengers Bangalore   
2    335983          Chennai Super Kings   
3    335983              Kings XI Punjab   
4    335984             Delhi Daredevils   

                                              player  
0  [AB Agarkar, AB Dinda, BB McCullum, DJ Hussey,...  
1  [AA Noffke, B Akhil, CL White, JH Kallis, MV B...  
2  [JDP Oram, Joginder Sharma, M Muralitharan, ME...  
3  [B Lee, IK Pathan, JR Hopes, K Goel, KC Sangak...  
4  [B Geeves, DL Vettori, G Gambhir, GD McGrath, ...  


In [58]:
print(
    playing_xi[
        playing_xi["match_id"] == "1082591"
    ]
)

Empty DataFrame
Columns: [match_id, team, player]
Index: []


In [59]:
player_df["match_id"] = player_df["match_id"].astype(str)

print(player_df["match_id"].dtype)

str


In [60]:
playing_xi = (
    player_df
    .groupby(["match_id", "team"])["player"]
    .apply(list)
    .reset_index()
)

print(playing_xi.head())

  match_id                         team  \
0  1082591  Royal Challengers Bangalore   
1  1082591          Sunrisers Hyderabad   
2  1082592               Mumbai Indians   
3  1082592       Rising Pune Supergiant   
4  1082593                Gujarat Lions   

                                              player  
0  [A Choudhary, CH Gayle, KM Jadhav, Mandeep Sin...  
1  [A Nehra, B Kumar, BCJ Cutting, Bipul Sharma, ...  
2  [AT Rayudu, HH Pandya, JC Buttler, JJ Bumrah, ...  
3  [A Zampa, AB Dinda, AM Rahane, BA Stokes, DL C...  
4  [AJ Finch, BB McCullum, DR Smith, DS Kulkarni,...  


In [61]:
print(
    playing_xi[
        playing_xi["match_id"] == "1082591"
    ]
)

  match_id                         team  \
0  1082591  Royal Challengers Bangalore   
1  1082591          Sunrisers Hyderabad   

                                              player  
0  [A Choudhary, CH Gayle, KM Jadhav, Mandeep Sin...  
1  [A Nehra, B Kumar, BCJ Cutting, Bipul Sharma, ...  


In [62]:
def get_match_roles(match):

    batting_players = set()
    bowling_players = set()

    for innings in match["innings"]:

        # Players who batted
        for over in innings["overs"]:
            for delivery in over["deliveries"]:

                batting_players.add(delivery["batter"])
                batting_players.add(delivery["non_striker"])

                # Player who bowled
                bowling_players.add(delivery["bowler"])

    return batting_players, bowling_players

In [63]:
for file in files:

    if os.path.splitext(file)[0] == "1082591":

        file_path = os.path.join(RAW_DATA_PATH, file)

        with open(file_path, "r") as f:
            test_match = json.load(f)

        break

In [64]:
batting_players, bowling_players = get_match_roles(test_match)

print("Batters:", batting_players)
print()
print("Bowlers:", bowling_players)

Batters: {'DJ Hooda', 'DA Warner', 'TM Head', 'SR Watson', 'Mandeep Singh', 'KM Jadhav', 'A Choudhary', 'S Dhawan', 'MC Henriques', 'CH Gayle', 'Sachin Baby', 'TS Mills', 'YS Chahal', 'S Aravind', 'BCJ Cutting', 'STR Binny', 'Yuvraj Singh'}

Bowlers: {'DJ Hooda', 'SR Watson', 'TM Head', 'Bipul Sharma', 'TS Mills', 'B Kumar', 'Rashid Khan', 'S Aravind', 'MC Henriques', 'YS Chahal', 'A Choudhary', 'BCJ Cutting', 'STR Binny', 'A Nehra'}


In [65]:
def get_team_roles(match):

    batting_players, bowling_players = get_match_roles(match)

    team_roles = {}

    for team in match["info"]["teams"]:

        players = match["info"]["players"][team]

        batters = [
            player
            for player in players
            if player in batting_players
        ]

        bowlers = [
            player
            for player in players
            if player in bowling_players
        ]

        team_roles[team] = {
            "batters": batters,
            "bowlers": bowlers
        }

    return team_roles

In [66]:
roles = get_team_roles(test_match)

for team, role_data in roles.items():

    print("\nTEAM:", team)

    print("Batters:")
    print(role_data["batters"])

    print("Bowlers:")
    print(role_data["bowlers"])


TEAM: Sunrisers Hyderabad
Batters:
['DA Warner', 'S Dhawan', 'MC Henriques', 'Yuvraj Singh', 'DJ Hooda', 'BCJ Cutting']
Bowlers:
['MC Henriques', 'DJ Hooda', 'BCJ Cutting', 'Bipul Sharma', 'B Kumar', 'A Nehra', 'Rashid Khan']

TEAM: Royal Challengers Bangalore
Batters:
['CH Gayle', 'Mandeep Singh', 'TM Head', 'KM Jadhav', 'SR Watson', 'Sachin Baby', 'STR Binny', 'S Aravind', 'TS Mills', 'YS Chahal', 'A Choudhary']
Bowlers:
['TM Head', 'SR Watson', 'STR Binny', 'S Aravind', 'TS Mills', 'YS Chahal', 'A Choudhary']


In [67]:
# ==========================================
# PLAYER ROLE PROFILE
# ==========================================

player_profile = (
    player_df
    .groupby("player")
    .agg(
        total_runs=("runs", "sum"),
        total_balls_faced=("balls_faced", "sum"),
        total_fours=("fours", "sum"),
        total_sixes=("sixes", "sum"),

        total_runs_conceded=("runs_conceded", "sum"),
        total_balls_bowled=("balls_bowled", "sum"),
        total_wickets=("wickets", "sum"),

        matches=("match_id", "nunique")
    )
    .reset_index()
)

# Batting strike rate
player_profile["career_strike_rate"] = (
    player_profile["total_runs"]
    / player_profile["total_balls_faced"]
    * 100
)

# Bowling economy
player_profile["career_economy"] = (
    player_profile["total_runs_conceded"]
    / player_profile["total_balls_bowled"]
    * 6
)

# Replace invalid values
player_profile = player_profile.replace(
    [float("inf"), -float("inf")],
    0
).fillna(0)

# Simple role classification
def classify_role(row):

    batting = row["total_balls_faced"]
    bowling = row["total_balls_bowled"]

    if batting > 0 and bowling > 0:
        return "all_rounder"

    elif batting > 0:
        return "batter"

    elif bowling > 0:
        return "bowler"

    else:
        return "unknown"


player_profile["role"] = player_profile.apply(
    classify_role,
    axis=1
)

print("Players:", len(player_profile))

print(
    player_profile[
        [
            "player",
            "matches",
            "total_runs",
            "career_strike_rate",
            "total_wickets",
            "career_economy",
            "role"
        ]
    ].head(20)
)

Players: 811
            player  matches  total_runs  career_strike_rate  total_wickets  \
0   A Ashish Reddy       31         280          145.077720             18   
1         A Badoni       66        1178          141.247002              4   
2       A Chandila       12           4           57.142857             11   
3         A Chopra        7          53           74.647887              0   
4      A Choudhary        5          25          125.000000              5   
5      A Dananjaya        1           4           80.000000              0   
6       A Flintoff        3          62          116.981132              2   
7         A Kamboj       25          74          148.000000             31   
8         A Kumble       42          35           74.468085             45   
9        A Manohar       27         292          124.255319              0   
10        A Mhatre       13         441          184.518828              0   
11        A Mishra      162         381           9

In [68]:
# ==========================================
# XI vs XI MATCHUP STRENGTH
# ==========================================

# Make sure IDs are strings
matchup_df["match_id"] = matchup_df["match_id"].astype(str)
player_df["match_id"] = player_df["match_id"].astype(str)

# Historical matchup strength for each batter-bowler pair
matchup_df["matchup_sr"] = (
    matchup_df["previous_runs"]
    / matchup_df["previous_balls"]
    * 100
)

matchup_df["matchup_sr"] = (
    matchup_df["matchup_sr"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

# Create a simple matchup score
matchup_df["matchup_score"] = (
    matchup_df["previous_runs"]
    + matchup_df["last_5_runs"]
    + matchup_df["previous_strike_rate"] / 10
    - matchup_df["previous_dismissals"] * 10
)

print(matchup_df[
    [
        "batter",
        "bowler",
        "previous_runs",
        "previous_strike_rate",
        "previous_dismissals",
        "matchup_score"
    ]
].head(10))

           batter         bowler  previous_runs  previous_strike_rate  \
0  A Ashish Reddy        A Nehra            0.0                   0.0   
1  A Ashish Reddy        A Nehra            5.0                 125.0   
2  A Ashish Reddy       AB Dinda            0.0                   0.0   
3  A Ashish Reddy       AB Dinda            6.0                 200.0   
4  A Ashish Reddy     AD Mathews            0.0                   0.0   
5  A Ashish Reddy     AD Mathews           13.0                 162.5   
6  A Ashish Reddy     AD Russell            0.0                   0.0   
7  A Ashish Reddy  Anureet Singh            0.0                   0.0   
8  A Ashish Reddy  Azhar Mahmood            0.0                   0.0   
9  A Ashish Reddy        B Kumar            0.0                   0.0   

   previous_dismissals  matchup_score  
0                  0.0           0.00  
1                  1.0          12.50  
2                  0.0           0.00  
3                  0.0          32.0

In [69]:
# ==========================================
# CREATE MATCHUP STRENGTH FOR EACH MATCH
# ==========================================

# Player team in each match
player_team = (
    player_df[
        ["match_id", "player", "team"]
    ]
    .drop_duplicates()
)

# Batter's team
matchup_with_teams = matchup_df.merge(
    player_team.rename(
        columns={
            "player": "batter",
            "team": "batter_team"
        }
    ),
    on=["match_id", "batter"],
    how="left"
)

# Bowler's team
matchup_with_teams = matchup_with_teams.merge(
    player_team.rename(
        columns={
            "player": "bowler",
            "team": "bowler_team"
        }
    ),
    on=["match_id", "bowler"],
    how="left"
)

print(matchup_with_teams[
    [
        "match_id",
        "batter",
        "bowler",
        "batter_team",
        "bowler_team",
        "matchup_score"
    ]
].head(10))

  match_id          batter         bowler          batter_team  \
0   598044  A Ashish Reddy        A Nehra  Sunrisers Hyderabad   
1   829773  A Ashish Reddy        A Nehra  Sunrisers Hyderabad   
2   598000  A Ashish Reddy       AB Dinda  Sunrisers Hyderabad   
3   598018  A Ashish Reddy       AB Dinda  Sunrisers Hyderabad   
4   598018  A Ashish Reddy     AD Mathews  Sunrisers Hyderabad   
5   829731  A Ashish Reddy     AD Mathews  Sunrisers Hyderabad   
6   980915  A Ashish Reddy     AD Russell  Sunrisers Hyderabad   
7   829759  A Ashish Reddy  Anureet Singh  Sunrisers Hyderabad   
8   598021  A Ashish Reddy  Azhar Mahmood  Sunrisers Hyderabad   
9   598000  A Ashish Reddy        B Kumar  Sunrisers Hyderabad   

             bowler_team  matchup_score  
0       Delhi Daredevils           0.00  
1    Chennai Super Kings          12.50  
2          Pune Warriors           0.00  
3          Pune Warriors          32.00  
4          Pune Warriors           0.00  
5       Delhi Daredev

In [70]:
matchup_strength = (
    matchup_with_teams
    .groupby(
        [
            "match_id",
            "batter_team",
            "bowler_team"
        ]
    )
    .agg(
        matchup_strength=("matchup_score", "mean"),
        matchup_count=("matchup_score", "count")
    )
    .reset_index()
)

print(matchup_strength.head(20))
print("Rows:", len(matchup_strength))

   match_id                  batter_team                  bowler_team  \
0   1082591  Royal Challengers Bangalore          Sunrisers Hyderabad   
1   1082591          Sunrisers Hyderabad  Royal Challengers Bangalore   
2   1082592               Mumbai Indians       Rising Pune Supergiant   
3   1082592       Rising Pune Supergiant               Mumbai Indians   
4   1082593                Gujarat Lions        Kolkata Knight Riders   
5   1082593        Kolkata Knight Riders                Gujarat Lions   
6   1082594              Kings XI Punjab       Rising Pune Supergiant   
7   1082594       Rising Pune Supergiant              Kings XI Punjab   
8   1082595             Delhi Daredevils  Royal Challengers Bangalore   
9   1082595  Royal Challengers Bangalore             Delhi Daredevils   
10  1082596                Gujarat Lions          Sunrisers Hyderabad   
11  1082596          Sunrisers Hyderabad                Gujarat Lions   
12  1082597        Kolkata Knight Riders           

In [71]:
# Team 1 batting vs Team 2 bowling
team1_matchup = matchup_strength.rename(
    columns={
        "batter_team": "team_1",
        "bowler_team": "team_2",
        "matchup_strength": "team1_batting_vs_team2_bowling"
    }
)[
    [
        "match_id",
        "team_1",
        "team_2",
        "team1_batting_vs_team2_bowling"
    ]
]

# Team 2 batting vs Team 1 bowling
team2_matchup = matchup_strength.rename(
    columns={
        "batter_team": "team_2",
        "bowler_team": "team_1",
        "matchup_strength": "team2_batting_vs_team1_bowling"
    }
)[
    [
        "match_id",
        "team_1",
        "team_2",
        "team2_batting_vs_team1_bowling"
    ]
]

# Merge
match_ml = match_ml.merge(
    team1_matchup,
    on=["match_id", "team_1", "team_2"],
    how="left"
)

match_ml = match_ml.merge(
    team2_matchup,
    on=["match_id", "team_1", "team_2"],
    how="left"
)

# No historical matchup = 0
match_ml[
    [
        "team1_batting_vs_team2_bowling",
        "team2_batting_vs_team1_bowling"
    ]
] = match_ml[
    [
        "team1_batting_vs_team2_bowling",
        "team2_batting_vs_team1_bowling"
    ]
].fillna(0)

print("Final shape:", match_ml.shape)

print(
    match_ml[
        [
            "team_1",
            "team_2",
            "team1_batting_vs_team2_bowling",
            "team2_batting_vs_team1_bowling",
            "team1_win"
        ]
    ].head(10)
)

Final shape: (1243, 34)
                        team_1                       team_2  \
0          Sunrisers Hyderabad  Royal Challengers Bangalore   
1       Rising Pune Supergiant               Mumbai Indians   
2                Gujarat Lions        Kolkata Knight Riders   
3              Kings XI Punjab       Rising Pune Supergiant   
4  Royal Challengers Bangalore             Delhi Daredevils   
5          Sunrisers Hyderabad                Gujarat Lions   
6               Mumbai Indians        Kolkata Knight Riders   
7              Kings XI Punjab  Royal Challengers Bangalore   
8       Rising Pune Supergiant             Delhi Daredevils   
9               Mumbai Indians          Sunrisers Hyderabad   

   team1_batting_vs_team2_bowling  team2_batting_vs_team1_bowling  team1_win  
0                       21.221008                       21.252073          1  
1                       28.857946                       14.919698          1  
2                       28.319279            

In [72]:
# ==========================================
# PREPARE ML DATA
# ==========================================

# Sort chronologically
match_ml["date"] = pd.to_datetime(match_ml["date"])

match_ml = match_ml.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)

# Remove columns that should NOT be features
drop_columns = [
    "match_id",
    "date",
    "venue",
    "team_1",
    "team_2",
    "winner",
    "team1_win"
]

X = match_ml.drop(columns=drop_columns)
y = match_ml["team1_win"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

X shape: (1243, 27)
y shape: (1243,)

Features:
['season', 'toss_winner', 'toss_decision', 'team1_batting_experience', 'team1_batting_form', 'team1_batting_strike_rate', 'team1_recent_strike_rate', 'team1_bowling_experience', 'team1_bowling_form', 'team1_bowling_economy', 'team1_recent_bowling_economy', 'team1_venue_runs', 'team1_venue_strike_rate', 'team1_venue_wickets', 'team2_batting_experience', 'team2_batting_form', 'team2_batting_strike_rate', 'team2_recent_strike_rate', 'team2_bowling_experience', 'team2_bowling_form', 'team2_bowling_economy', 'team2_recent_bowling_economy', 'team2_venue_runs', 'team2_venue_strike_rate', 'team2_venue_wickets', 'team1_batting_vs_team2_bowling', 'team2_batting_vs_team1_bowling']


In [73]:
# ==========================================
# CHRONOLOGICAL TRAIN / TEST SPLIT
# ==========================================

split_index = int(len(match_ml) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training matches:", len(X_train))
print("Testing matches:", len(X_test))

print(
    "\nTraining period:",
    match_ml.iloc[0]["date"],
    "to",
    match_ml.iloc[split_index - 1]["date"]
)

print(
    "Testing period:",
    match_ml.iloc[split_index]["date"],
    "to",
    match_ml.iloc[-1]["date"]
)

Training matches: 994
Testing matches: 249

Training period: 2008-04-18 00:00:00 to 2023-05-02 00:00:00
Testing period: 2023-05-03 00:00:00 to 2026-05-31 00:00:00


In [76]:
# ==========================================
# FIX SEASON + TRAIN LOGISTIC REGRESSION
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# ------------------------------------------
# 1. Convert season to numeric
# ------------------------------------------

def season_to_year(value):
    try:
        return int(str(value).split("/")[0])
    except:
        return 0

X_train = X_train.copy()
X_test = X_test.copy()

X_train["season"] = X_train["season"].apply(season_to_year)
X_test["season"] = X_test["season"].apply(season_to_year)

# ------------------------------------------
# 2. Feature types
# ------------------------------------------

categorical_features = [
    "toss_winner",
    "toss_decision"
]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)

# ------------------------------------------
# 3. Preprocessor
# ------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        )
    ]
)

# ------------------------------------------
# 4. Model
# ------------------------------------------

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000
            )
        )
    ]
)

# ------------------------------------------
# 5. Train
# ------------------------------------------

model.fit(X_train, y_train)

# ------------------------------------------
# 6. Predict
# ------------------------------------------

y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

# ------------------------------------------
# 7. Metrics
# ------------------------------------------

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\n======================================")
print("LOGISTIC REGRESSION RESULTS")
print("======================================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\n======================================")
print("CLASSIFICATION REPORT")
print("======================================")

print(classification_report(y_test, y_pred))

Numeric features: 25
Categorical features: ['toss_winner', 'toss_decision']

LOGISTIC REGRESSION RESULTS
Accuracy : 0.6104
Precision: 0.5500
Recall   : 0.6055
F1 Score : 0.5764
ROC-AUC  : 0.6558

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.67      0.61      0.64       140
           1       0.55      0.61      0.58       109

    accuracy                           0.61       249
   macro avg       0.61      0.61      0.61       249
weighted avg       0.62      0.61      0.61       249



c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [77]:
# ==========================================
# RANDOM FOREST MODEL
# ==========================================

from sklearn.ensemble import RandomForestClassifier

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# ------------------------------------------
# 1. Feature types
# ------------------------------------------

categorical_features = [
    "toss_winner",
    "toss_decision"
]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

# ------------------------------------------
# 2. Preprocessor
# ------------------------------------------

preprocessor_rf = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        )
    ]
)

# ------------------------------------------
# 3. Random Forest
# ------------------------------------------

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),

        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

# ------------------------------------------
# 4. Train
# ------------------------------------------

rf_model.fit(X_train, y_train)

# ------------------------------------------
# 5. Predictions
# ------------------------------------------

rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(X_test)[:, 1]

# ------------------------------------------
# 6. Evaluation
# ------------------------------------------

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("======================================")
print("RANDOM FOREST RESULTS")
print("======================================")

print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_auc:.4f}")

print("\n======================================")
print("CLASSIFICATION REPORT")
print("======================================")

print(
    classification_report(
        y_test,
        rf_pred
    )
)

RANDOM FOREST RESULTS
Accuracy : 0.5944
Precision: 0.5392
Recall   : 0.5046
F1 Score : 0.5213
ROC-AUC  : 0.6115

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.63      0.66      0.65       140
           1       0.54      0.50      0.52       109

    accuracy                           0.59       249
   macro avg       0.59      0.58      0.58       249
weighted avg       0.59      0.59      0.59       249



In [78]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

ModuleNotFoundError: No module named 'xgboost'

In [79]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "xgboost"
])

0

In [80]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [81]:
# ==========================================
# XGBOOST MODEL
# ==========================================

import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# ------------------------------------------
# 1. Feature types
# ------------------------------------------

categorical_features = [
    "toss_winner",
    "toss_decision"
]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

# ------------------------------------------
# 2. Preprocessor
# ------------------------------------------

preprocessor_xgb = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        )
    ]
)

# ------------------------------------------
# 3. XGBoost
# ------------------------------------------

xgb_classifier = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------
# 4. Pipeline
# ------------------------------------------

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_xgb),
        ("classifier", xgb_classifier)
    ]
)

# ------------------------------------------
# 5. Train
# ------------------------------------------

print("Training XGBoost...")

xgb_model.fit(
    X_train,
    y_train
)

print("Training complete!")

# ------------------------------------------
# 6. Predictions
# ------------------------------------------

xgb_pred = xgb_model.predict(X_test)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# ------------------------------------------
# 7. Evaluation
# ------------------------------------------

xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

xgb_precision = precision_score(
    y_test,
    xgb_pred
)

xgb_recall = recall_score(
    y_test,
    xgb_pred
)

xgb_f1 = f1_score(
    y_test,
    xgb_pred
)

xgb_auc = roc_auc_score(
    y_test,
    xgb_prob
)

print("\n======================================")
print("XGBOOST RESULTS")
print("======================================")

print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1 Score : {xgb_f1:.4f}")
print(f"ROC-AUC  : {xgb_auc:.4f}")

print("\n======================================")
print("CLASSIFICATION REPORT")
print("======================================")

print(
    classification_report(
        y_test,
        xgb_pred
    )
)

Training XGBoost...
Training complete!

XGBOOST RESULTS
Accuracy : 0.5622
Precision: 0.5000
Recall   : 0.5780
F1 Score : 0.5362
ROC-AUC  : 0.5971

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.63      0.55      0.59       140
           1       0.50      0.58      0.54       109

    accuracy                           0.56       249
   macro avg       0.56      0.56      0.56       249
weighted avg       0.57      0.56      0.56       249



In [82]:
# ==========================================
# STEP 35 - FEATURE LEAKAGE AUDIT
# ==========================================

feature_groups = {
    "Basic": [
        "season",
        "toss_winner",
        "toss_decision"
    ],

    "Team 1 batting": [
        "team1_batting_experience",
        "team1_batting_form",
        "team1_batting_strike_rate",
        "team1_recent_strike_rate"
    ],

    "Team 1 bowling": [
        "team1_bowling_experience",
        "team1_bowling_form",
        "team1_bowling_economy",
        "team1_recent_bowling_economy"
    ],

    "Team 1 venue": [
        "team1_venue_runs",
        "team1_venue_strike_rate",
        "team1_venue_wickets"
    ],

    "Team 2 batting": [
        "team2_batting_experience",
        "team2_batting_form",
        "team2_batting_strike_rate",
        "team2_recent_strike_rate"
    ],

    "Team 2 bowling": [
        "team2_bowling_experience",
        "team2_bowling_form",
        "team2_bowling_economy",
        "team2_recent_bowling_economy"
    ],

    "Team 2 venue": [
        "team2_venue_runs",
        "team2_venue_strike_rate",
        "team2_venue_wickets"
    ],

    "Head-to-head matchup": [
        "team1_batting_vs_team2_bowling",
        "team2_batting_vs_team1_bowling"
    ]
}

print("======================================")
print("FEATURE LEAKAGE AUDIT")
print("======================================")

for group, features in feature_groups.items():

    print(f"\n{group}")

    for feature in features:

        if feature in match_ml.columns:
            print(f"  ✓ {feature}")
        else:
            print(f"  ✗ {feature} NOT FOUND")

print("\nTotal features checked:",
      sum(len(v) for v in feature_groups.values()))

FEATURE LEAKAGE AUDIT

Basic
  ✓ season
  ✓ toss_winner
  ✓ toss_decision

Team 1 batting
  ✓ team1_batting_experience
  ✓ team1_batting_form
  ✓ team1_batting_strike_rate
  ✓ team1_recent_strike_rate

Team 1 bowling
  ✓ team1_bowling_experience
  ✓ team1_bowling_form
  ✓ team1_bowling_economy
  ✓ team1_recent_bowling_economy

Team 1 venue
  ✓ team1_venue_runs
  ✓ team1_venue_strike_rate
  ✓ team1_venue_wickets

Team 2 batting
  ✓ team2_batting_experience
  ✓ team2_batting_form
  ✓ team2_batting_strike_rate
  ✓ team2_recent_strike_rate

Team 2 bowling
  ✓ team2_bowling_experience
  ✓ team2_bowling_form
  ✓ team2_bowling_economy
  ✓ team2_recent_bowling_economy

Team 2 venue
  ✓ team2_venue_runs
  ✓ team2_venue_strike_rate
  ✓ team2_venue_wickets

Head-to-head matchup
  ✓ team1_batting_vs_team2_bowling
  ✓ team2_batting_vs_team1_bowling

Total features checked: 27


In [83]:
# ==========================================
# STEP 36 - CHECK FEATURE VALUES OVER TIME
# ==========================================

# Sort matches chronologically
audit_df = match_ml.copy()

audit_df["date"] = pd.to_datetime(audit_df["date"])

audit_df = audit_df.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)

# ------------------------------------------
# Check whether feature values are changing
# logically over time
# ------------------------------------------

features_to_check = [
    "team1_batting_experience",
    "team1_batting_form",
    "team1_batting_strike_rate",

    "team1_bowling_experience",
    "team1_bowling_form",
    "team1_bowling_economy",

    "team1_venue_runs",
    "team1_venue_strike_rate",
    "team1_venue_wickets",

    "team2_batting_experience",
    "team2_batting_form",
    "team2_batting_strike_rate",

    "team2_bowling_experience",
    "team2_bowling_form",
    "team2_bowling_economy",

    "team2_venue_runs",
    "team2_venue_strike_rate",
    "team2_venue_wickets",

    "team1_batting_vs_team2_bowling",
    "team2_batting_vs_team1_bowling"
]

print("======================================")
print("FEATURE TIME AUDIT")
print("======================================")

for feature in features_to_check:

    # Number of non-zero values
    non_zero = (audit_df[feature] != 0).sum()

    # Number of missing values
    missing = audit_df[feature].isna().sum()

    print(
        f"{feature:45s} "
        f"non-zero={non_zero:4d} "
        f"missing={missing:4d}"
    )

# ------------------------------------------
# Show earliest matches
# ------------------------------------------

print("\n======================================")
print("EARLIEST MATCHES")
print("======================================")

print(
    audit_df[
        [
            "date",
            "team_1",
            "team_2",
            "team1_batting_experience",
            "team1_batting_form",
            "team1_venue_runs",
            "team1_batting_vs_team2_bowling"
        ]
    ].head(15)
)

FEATURE TIME AUDIT
team1_batting_experience                      non-zero=1239 missing=   0
team1_batting_form                            non-zero=1239 missing=   0
team1_batting_strike_rate                     non-zero=1239 missing=   0
team1_bowling_experience                      non-zero=1239 missing=   0
team1_bowling_form                            non-zero=1239 missing=   0
team1_bowling_economy                         non-zero=1239 missing=   0
team1_venue_runs                              non-zero=1087 missing=   0
team1_venue_strike_rate                       non-zero=1087 missing=   0
team1_venue_wickets                           non-zero=1065 missing=   0
team2_batting_experience                      non-zero=1239 missing=   0
team2_batting_form                            non-zero=1239 missing=   0
team2_batting_strike_rate                     non-zero=1239 missing=   0
team2_bowling_experience                      non-zero=1239 missing=   0
team2_bowling_form              

In [84]:
# ==========================================
# STEP 37
# LEAKAGE-FREE TEAM FEATURES
# ==========================================

# Make a clean copy
clean_matches = match_info_df.copy()

clean_matches["date"] = pd.to_datetime(
    clean_matches["date"]
)

clean_matches["match_id"] = clean_matches[
    "match_id"
].astype(str)

# Sort chronologically
clean_matches = clean_matches.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)


# ------------------------------------------
# Storage for historical team statistics
# ------------------------------------------

team_history = {}

team_features = []


# ------------------------------------------
# Process matches one by one
# ------------------------------------------

for _, match_row in clean_matches.iterrows():

    match_id = match_row["match_id"]
    team1 = match_row["team_1"]
    team2 = match_row["team_2"]

    # Create empty history if team is new
    if team1 not in team_history:
        team_history[team1] = {
            "matches": 0,
            "runs": 0,
            "balls": 0,
            "wickets": 0,
            "runs_conceded": 0,
            "balls_bowled": 0
        }

    if team2 not in team_history:
        team_history[team2] = {
            "matches": 0,
            "runs": 0,
            "balls": 0,
            "wickets": 0,
            "runs_conceded": 0,
            "balls_bowled": 0
        }

    h1 = team_history[team1]
    h2 = team_history[team2]

    # --------------------------------------
    # FEATURES BEFORE CURRENT MATCH
    # --------------------------------------

    def batting_average(history):
        if history["matches"] == 0:
            return 0

        return history["runs"] / history["matches"]

    def batting_strike_rate(history):
        if history["balls"] == 0:
            return 0

        return history["runs"] / history["balls"] * 100

    def bowling_economy(history):
        if history["balls_bowled"] == 0:
            return 0

        return (
            history["runs_conceded"]
            / history["balls_bowled"]
            * 6
        )

    # --------------------------------------
    # Store PRE-MATCH features
    # --------------------------------------

    team_features.append({

        "match_id": match_id,

        "team_1": team1,
        "team_2": team2,

        # Team 1
        "team1_previous_matches":
            h1["matches"],

        "team1_batting_experience":
            h1["balls"],

        "team1_batting_form":
            batting_average(h1),

        "team1_batting_strike_rate":
            batting_strike_rate(h1),

        "team1_bowling_experience":
            h1["balls_bowled"],

        "team1_bowling_form":
            h1["wickets"],

        "team1_bowling_economy":
            bowling_economy(h1),

        # Team 2
        "team2_previous_matches":
            h2["matches"],

        "team2_batting_experience":
            h2["balls"],

        "team2_batting_form":
            batting_average(h2),

        "team2_batting_strike_rate":
            batting_strike_rate(h2),

        "team2_bowling_experience":
            h2["balls_bowled"],

        "team2_bowling_form":
            h2["wickets"],

        "team2_bowling_economy":
            bowling_economy(h2)
    })


    # --------------------------------------
    # UPDATE HISTORY AFTER CURRENT MATCH
    # --------------------------------------

    # Find players from this match
    current_players = player_df[
        player_df["match_id"] == match_id
    ]

    # Update each team
    for team in [team1, team2]:

        team_players = current_players[
            current_players["team"] == team
        ]

        if len(team_players) == 0:
            continue

        runs = team_players["runs"].sum()

        balls = team_players["balls_faced"].sum()

        wickets = team_players["wickets"].sum()

        runs_conceded = team_players[
            "runs_conceded"
        ].sum()

        balls_bowled = team_players[
            "balls_bowled"
        ].sum()

        team_history[team]["matches"] += 1
        team_history[team]["runs"] += runs
        team_history[team]["balls"] += balls
        team_history[team]["wickets"] += wickets
        team_history[team]["runs_conceded"] += runs_conceded
        team_history[team]["balls_bowled"] += balls_bowled


# ------------------------------------------
# Convert to DataFrame
# ------------------------------------------

clean_team_features = pd.DataFrame(
    team_features
)

print("======================================")
print("LEAKAGE-FREE TEAM FEATURES")
print("======================================")

print(
    "Shape:",
    clean_team_features.shape
)

print(
    clean_team_features.head(10)
)

LEAKAGE-FREE TEAM FEATURES
Shape: (1243, 17)
  match_id                       team_1                       team_2  \
0   335982  Royal Challengers Bangalore        Kolkata Knight Riders   
1   335983              Kings XI Punjab          Chennai Super Kings   
2   335984             Delhi Daredevils             Rajasthan Royals   
3   335985               Mumbai Indians  Royal Challengers Bangalore   
4   335986        Kolkata Knight Riders              Deccan Chargers   
5   335987             Rajasthan Royals              Kings XI Punjab   
6   335988              Deccan Chargers             Delhi Daredevils   
7   335989          Chennai Super Kings               Mumbai Indians   
8   335990              Deccan Chargers             Rajasthan Royals   
9   335991              Kings XI Punjab               Mumbai Indians   

   team1_previous_matches  team1_batting_experience  team1_batting_form  \
0                       0                         0                 0.0   
1           

In [85]:
# ==========================================
# STEP 38
# LEAKAGE-FREE RECENT TEAM FORM
# ==========================================

# Match-level team performance
team_match_stats = []

for _, row in player_df.iterrows():

    team_match_stats.append({
        "match_id": str(row["match_id"]),
        "team": row["team"],
        "runs": row["runs"],
        "wickets": row["wickets"],
        "runs_conceded": row["runs_conceded"],
        "balls_bowled": row["balls_bowled"]
    })

team_match_stats = pd.DataFrame(team_match_stats)

# Aggregate players into team-match totals
team_match_stats = (
    team_match_stats
    .groupby(
        ["match_id", "team"],
        as_index=False
    )
    .agg(
        runs=("runs", "sum"),
        wickets=("wickets", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        balls_bowled=("balls_bowled", "sum")
    )
)

# Get match dates
match_dates = match_info_df[
    ["match_id", "date"]
].copy()

match_dates["match_id"] = (
    match_dates["match_id"].astype(str)
)

match_dates["date"] = pd.to_datetime(
    match_dates["date"]
)

team_match_stats = team_match_stats.merge(
    match_dates,
    on="match_id",
    how="left"
)

team_match_stats = team_match_stats.sort_values(
    ["team", "date", "match_id"]
)

# ------------------------------------------
# Create PREVIOUS match statistics
# ------------------------------------------

team_match_stats["previous_runs"] = (
    team_match_stats
    .groupby("team")["runs"]
    .shift(1)
)

team_match_stats["previous_wickets"] = (
    team_match_stats
    .groupby("team")["wickets"]
    .shift(1)
)

team_match_stats["last_5_runs"] = (
    team_match_stats
    .groupby("team")["runs"]
    .transform(
        lambda x: x.shift(1).rolling(
            5,
            min_periods=1
        ).mean()
    )
)

team_match_stats["last_5_wickets"] = (
    team_match_stats
    .groupby("team")["wickets"]
    .transform(
        lambda x: x.shift(1).rolling(
            5,
            min_periods=1
        ).mean()
    )
)

team_match_stats["last_5_runs_conceded"] = (
    team_match_stats
    .groupby("team")["runs_conceded"]
    .transform(
        lambda x: x.shift(1).rolling(
            5,
            min_periods=1
        ).mean()
    )
)

team_match_stats["last_5_balls_bowled"] = (
    team_match_stats
    .groupby("team")["balls_bowled"]
    .transform(
        lambda x: x.shift(1).rolling(
            5,
            min_periods=1
        ).mean()
    )
)

# Replace missing values
recent_columns = [
    "previous_runs",
    "previous_wickets",
    "last_5_runs",
    "last_5_wickets",
    "last_5_runs_conceded",
    "last_5_balls_bowled"
]

team_match_stats[recent_columns] = (
    team_match_stats[recent_columns]
    .fillna(0)
)

print("======================================")
print("RECENT FORM CREATED")
print("======================================")

print(
    team_match_stats[
        [
            "match_id",
            "team",
            "runs",
            "previous_runs",
            "last_5_runs",
            "wickets",
            "last_5_wickets"
        ]
    ].head(15)
)

RECENT FORM CREATED
     match_id                 team  runs  previous_runs  last_5_runs  wickets  \
1334   335983  Chennai Super Kings   234            0.0          0.0        4   
1346   335989  Chennai Super Kings   190          234.0        234.0        6   
1354   335993  Chennai Super Kings   134          190.0        212.0        8   
1360   335996  Chennai Super Kings   174          134.0        186.0        8   
1370   336001  Chennai Super Kings   162          174.0        183.0        2   
1378   336005  Chennai Super Kings   102          162.0        178.8        2   
1382   336007  Chennai Super Kings   134          102.0        152.4        3   
1386   336009  Chennai Super Kings   177          134.0        141.2        5   
1394   336013  Chennai Super Kings   166          177.0        149.8        9   
1404   336018  Chennai Super Kings   144          166.0        148.2        1   
1418   336025  Chennai Super Kings    51          144.0        144.6        5   
1426   3

In [86]:
# ==========================================
# STEP 39
# COMBINE LEAKAGE-FREE FEATURES
# ==========================================

# ------------------------------------------
# Prepare recent form table
# ------------------------------------------

recent_form = team_match_stats[
    [
        "match_id",
        "team",
        "last_5_runs",
        "last_5_wickets",
        "last_5_runs_conceded",
        "last_5_balls_bowled"
    ]
].copy()

# ------------------------------------------
# Team 1 recent form
# ------------------------------------------

team1_recent = recent_form.rename(
    columns={
        "team": "team_1",
        "last_5_runs": "team1_recent_runs",
        "last_5_wickets": "team1_recent_wickets",
        "last_5_runs_conceded": "team1_recent_runs_conceded",
        "last_5_balls_bowled": "team1_recent_balls_bowled"
    }
)

team1_recent = team1_recent[
    [
        "match_id",
        "team_1",
        "team1_recent_runs",
        "team1_recent_wickets",
        "team1_recent_runs_conceded",
        "team1_recent_balls_bowled"
    ]
]

# ------------------------------------------
# Team 2 recent form
# ------------------------------------------

team2_recent = recent_form.rename(
    columns={
        "team": "team_2",
        "last_5_runs": "team2_recent_runs",
        "last_5_wickets": "team2_recent_wickets",
        "last_5_runs_conceded": "team2_recent_runs_conceded",
        "last_5_balls_bowled": "team2_recent_balls_bowled"
    }
)

team2_recent = team2_recent[
    [
        "match_id",
        "team_2",
        "team2_recent_runs",
        "team2_recent_wickets",
        "team2_recent_runs_conceded",
        "team2_recent_balls_bowled"
    ]
]

# ------------------------------------------
# Start with original match information
# ------------------------------------------

clean_match_ml = clean_matches[
    [
        "match_id",
        "date",
        "season",
        "team_1",
        "team_2",
        "toss_winner",
        "toss_decision",
        "winner"
    ]
].copy()

# ------------------------------------------
# Create target
# ------------------------------------------

clean_match_ml["team1_win"] = (
    clean_match_ml["winner"]
    == clean_match_ml["team_1"]
).astype(int)

# ------------------------------------------
# Merge leakage-free team features
# ------------------------------------------

clean_match_ml = clean_match_ml.merge(
    clean_team_features,
    on=[
        "match_id",
        "team_1",
        "team_2"
    ],
    how="left"
)

# ------------------------------------------
# Merge Team 1 recent form
# ------------------------------------------

clean_match_ml = clean_match_ml.merge(
    team1_recent,
    on=[
        "match_id",
        "team_1"
    ],
    how="left"
)

# ------------------------------------------
# Merge Team 2 recent form
# ------------------------------------------

clean_match_ml = clean_match_ml.merge(
    team2_recent,
    on=[
        "match_id",
        "team_2"
    ],
    how="left"
)

# ------------------------------------------
# Fill missing historical values
# ------------------------------------------

clean_match_ml = clean_match_ml.fillna(0)

print("======================================")
print("CLEAN MATCH DATASET")
print("======================================")

print(
    "Shape:",
    clean_match_ml.shape
)

print("\nColumns:")
print(clean_match_ml.columns.tolist())

print("\nFirst 5 rows:")
print(clean_match_ml.head())

CLEAN MATCH DATASET
Shape: (1243, 31)

Columns:
['match_id', 'date', 'season', 'team_1', 'team_2', 'toss_winner', 'toss_decision', 'winner', 'team1_win', 'team1_previous_matches', 'team1_batting_experience', 'team1_batting_form', 'team1_batting_strike_rate', 'team1_bowling_experience', 'team1_bowling_form', 'team1_bowling_economy', 'team2_previous_matches', 'team2_batting_experience', 'team2_batting_form', 'team2_batting_strike_rate', 'team2_bowling_experience', 'team2_bowling_form', 'team2_bowling_economy', 'team1_recent_runs', 'team1_recent_wickets', 'team1_recent_runs_conceded', 'team1_recent_balls_bowled', 'team2_recent_runs', 'team2_recent_wickets', 'team2_recent_runs_conceded', 'team2_recent_balls_bowled']

First 5 rows:
  match_id       date   season                       team_1  \
0   335982 2008-04-18  2007/08  Royal Challengers Bangalore   
1   335983 2008-04-19  2007/08              Kings XI Punjab   
2   335984 2008-04-19  2007/08             Delhi Daredevils   
3   335985 

In [88]:
# ==========================================
# STEP 40 - LEAKAGE-FREE PLAYER MATCHUPS
# CORRECTED VERSION
# ==========================================

# ------------------------------------------
# 1. Prepare match information
# ------------------------------------------

match_info_temp = match_info_df[
    [
        "match_id",
        "date",
        "team_1",
        "team_2"
    ]
].copy()

match_info_temp["match_id"] = (
    match_info_temp["match_id"].astype(str)
)

match_info_temp["date"] = pd.to_datetime(
    match_info_temp["date"]
)


# ------------------------------------------
# 2. Prepare matchup data
# ------------------------------------------

matchup_history = matchup_df.copy()

matchup_history["match_id"] = (
    matchup_history["match_id"].astype(str)
)


# ------------------------------------------
# 3. Remove existing date if present
# ------------------------------------------

if "date" in matchup_history.columns:
    matchup_history = matchup_history.drop(
        columns=["date"]
    )

if "date_x" in matchup_history.columns:
    matchup_history = matchup_history.drop(
        columns=["date_x"]
    )

if "date_y" in matchup_history.columns:
    matchup_history = matchup_history.drop(
        columns=["date_y"]
    )


# ------------------------------------------
# 4. Add match date
# ------------------------------------------

matchup_history = matchup_history.merge(
    match_info_temp[
        ["match_id", "date"]
    ],
    on="match_id",
    how="left"
)


# ------------------------------------------
# 5. Sort chronologically
# ------------------------------------------

matchup_history = matchup_history.sort_values(
    [
        "batter",
        "bowler",
        "date",
        "match_id"
    ]
).reset_index(drop=True)


# ------------------------------------------
# 6. Previous runs
# ------------------------------------------

matchup_history["previous_runs"] = (
    matchup_history
    .groupby(
        ["batter", "bowler"]
    )["runs"]
    .cumsum()
    - matchup_history["runs"]
)


# ------------------------------------------
# 7. Previous balls
# ------------------------------------------

matchup_history["previous_balls"] = (
    matchup_history
    .groupby(
        ["batter", "bowler"]
    )["balls"]
    .cumsum()
    - matchup_history["balls"]
)


# ------------------------------------------
# 8. Previous dismissals
# ------------------------------------------

matchup_history["previous_dismissals"] = (
    matchup_history
    .groupby(
        ["batter", "bowler"]
    )["dismissals"]
    .cumsum()
    - matchup_history["dismissals"]
)


# ------------------------------------------
# 9. Previous strike rate
# ------------------------------------------

matchup_history["previous_strike_rate"] = (
    matchup_history["previous_runs"]
    / matchup_history["previous_balls"]
    * 100
)

matchup_history["previous_strike_rate"] = (
    matchup_history["previous_strike_rate"]
    .replace(
        [float("inf"), -float("inf")],
        0
    )
    .fillna(0)
)


# ------------------------------------------
# 10. Number of previous encounters
# ------------------------------------------

matchup_history["previous_matchups"] = (
    matchup_history
    .groupby(
        ["batter", "bowler"]
    )
    .cumcount()
)


# ------------------------------------------
# 11. Matchup strength
# ------------------------------------------

matchup_history["matchup_strength"] = (
    matchup_history["previous_runs"]
    + (
        matchup_history["previous_strike_rate"]
        / 10
    )
    - (
        matchup_history["previous_dismissals"]
        * 10
    )
)


# ------------------------------------------
# 12. Check result
# ------------------------------------------

print("======================================")
print("LEAKAGE-FREE MATCHUP DATA")
print("======================================")

print(
    "Shape:",
    matchup_history.shape
)

print("\nColumns:")

print(
    matchup_history.columns.tolist()
)

print("\nFirst 20 rows:")

print(
    matchup_history[
        [
            "match_id",
            "date",
            "batter",
            "bowler",
            "balls",
            "runs",
            "previous_runs",
            "previous_balls",
            "previous_dismissals",
            "previous_strike_rate",
            "previous_matchups",
            "matchup_strength"
        ]
    ].head(20)
)

LEAKAGE-FREE MATCHUP DATA
Shape: (61429, 24)

Columns:
['match_id', 'season', 'batter', 'bowler', 'balls', 'runs', 'fours', 'sixes', 'dismissals', 'previous_balls', 'previous_runs', 'previous_dismissals', 'previous_fours', 'previous_sixes', 'previous_strike_rate', 'last_5_runs', 'last_5_balls', 'last_5_strike_rate', 'last_5_dismissals', 'matchup_sr', 'matchup_score', 'date', 'previous_matchups', 'matchup_strength']

First 20 rows:
   match_id       date          batter           bowler  balls  runs  \
0    598044 2013-05-04  A Ashish Reddy          A Nehra      4     5   
1    829773 2015-05-02  A Ashish Reddy          A Nehra      4     2   
2    598000 2013-04-05  A Ashish Reddy         AB Dinda      3     6   
3    598018 2013-04-17  A Ashish Reddy         AB Dinda      4     3   
4    598018 2013-04-17  A Ashish Reddy       AD Mathews      8    13   
5    829731 2015-04-18  A Ashish Reddy       AD Mathews      4    12   
6    980915 2016-04-16  A Ashish Reddy       AD Russell      

In [89]:
# ==========================================
# STEP 41 - TEAM VS TEAM MATCHUP STRENGTH
# ==========================================

# Matchup data already contains historical
# matchup_strength for every batter-bowler pair.

# ------------------------------------------
# Get teams for each batter
# ------------------------------------------

player_teams = player_df[
    ["match_id", "player", "team"]
].copy()

player_teams["match_id"] = (
    player_teams["match_id"].astype(str)
)

player_teams = player_teams.rename(
    columns={
        "player": "batter"
    }
)

# ------------------------------------------
# Add batter team
# ------------------------------------------

m = matchup_history.merge(
    player_teams[
        ["match_id", "batter", "team"]
    ],
    on=["match_id", "batter"],
    how="left"
)

m = m.rename(
    columns={
        "team": "batter_team"
    }
)

# ------------------------------------------
# Add bowler team
# ------------------------------------------

bowler_teams = player_df[
    ["match_id", "player", "team"]
].copy()

bowler_teams["match_id"] = (
    bowler_teams["match_id"].astype(str)
)

bowler_teams = bowler_teams.rename(
    columns={
        "player": "bowler",
        "team": "bowler_team"
    }
)

m = m.merge(
    bowler_teams[
        ["match_id", "bowler", "bowler_team"]
    ],
    on=["match_id", "bowler"],
    how="left"
)

# ------------------------------------------
# Aggregate matchup strength by match
# ------------------------------------------

team_matchup = (
    m.groupby(
        [
            "match_id",
            "batter_team",
            "bowler_team"
        ],
        as_index=False
    )
    .agg(
        matchup_strength=(
            "matchup_strength",
            "mean"
        ),
        matchup_count=(
            "matchup_strength",
            "count"
        )
    )
)

print("======================================")
print("TEAM VS TEAM MATCHUP")
print("======================================")

print(
    "Rows:",
    len(team_matchup)
)

print(
    team_matchup.head(20)
)

TEAM VS TEAM MATCHUP
Rows: 2480
   match_id                  batter_team                  bowler_team  \
0   1082591  Royal Challengers Bangalore          Sunrisers Hyderabad   
1   1082591          Sunrisers Hyderabad  Royal Challengers Bangalore   
2   1082592               Mumbai Indians       Rising Pune Supergiant   
3   1082592       Rising Pune Supergiant               Mumbai Indians   
4   1082593                Gujarat Lions        Kolkata Knight Riders   
5   1082593        Kolkata Knight Riders                Gujarat Lions   
6   1082594              Kings XI Punjab       Rising Pune Supergiant   
7   1082594       Rising Pune Supergiant              Kings XI Punjab   
8   1082595             Delhi Daredevils  Royal Challengers Bangalore   
9   1082595  Royal Challengers Bangalore             Delhi Daredevils   
10  1082596                Gujarat Lions          Sunrisers Hyderabad   
11  1082596          Sunrisers Hyderabad                Gujarat Lions   
12  1082597        

In [90]:
# ==========================================
# STEP 42 - ADD MATCHUP FEATURES
# ==========================================

# Team 1 batting vs Team 2 bowling
t1_matchup = team_matchup.rename(
    columns={
        "batter_team": "team_1",
        "bowler_team": "team_2",
        "matchup_strength":
            "team1_batting_vs_team2_bowling",
        "matchup_count":
            "team1_matchup_count"
    }
)

t1_matchup = t1_matchup[
    [
        "match_id",
        "team_1",
        "team_2",
        "team1_batting_vs_team2_bowling",
        "team1_matchup_count"
    ]
]


# Team 2 batting vs Team 1 bowling
t2_matchup = team_matchup.rename(
    columns={
        "batter_team": "team_2",
        "bowler_team": "team_1",
        "matchup_strength":
            "team2_batting_vs_team1_bowling",
        "matchup_count":
            "team2_matchup_count"
    }
)

t2_matchup = t2_matchup[
    [
        "match_id",
        "team_1",
        "team_2",
        "team2_batting_vs_team1_bowling",
        "team2_matchup_count"
    ]
]


# ------------------------------------------
# Merge both matchup directions
# ------------------------------------------

final_ml = clean_match_ml.copy()

final_ml["match_id"] = (
    final_ml["match_id"].astype(str)
)

final_ml = final_ml.merge(
    t1_matchup,
    on=["match_id", "team_1", "team_2"],
    how="left"
)

final_ml = final_ml.merge(
    t2_matchup,
    on=["match_id", "team_1", "team_2"],
    how="left"
)


# ------------------------------------------
# Missing matchup history = 0
# ------------------------------------------

final_ml[
    [
        "team1_batting_vs_team2_bowling",
        "team2_batting_vs_team1_bowling",
        "team1_matchup_count",
        "team2_matchup_count"
    ]
] = final_ml[
    [
        "team1_batting_vs_team2_bowling",
        "team2_batting_vs_team1_bowling",
        "team1_matchup_count",
        "team2_matchup_count"
    ]
].fillna(0)


print("======================================")
print("FINAL ML DATASET")
print("======================================")

print(
    "Shape:",
    final_ml.shape
)

print("\nMatchup features:")

print(
    final_ml[
        [
            "team_1",
            "team_2",
            "team1_batting_vs_team2_bowling",
            "team2_batting_vs_team1_bowling",
            "team1_matchup_count",
            "team2_matchup_count",
            "team1_win"
        ]
    ].head(10)
)

FINAL ML DATASET
Shape: (1243, 35)

Matchup features:
                        team_1                       team_2  \
0  Royal Challengers Bangalore        Kolkata Knight Riders   
1              Kings XI Punjab          Chennai Super Kings   
2             Delhi Daredevils             Rajasthan Royals   
3               Mumbai Indians  Royal Challengers Bangalore   
4        Kolkata Knight Riders              Deccan Chargers   
5             Rajasthan Royals              Kings XI Punjab   
6              Deccan Chargers             Delhi Daredevils   
7          Chennai Super Kings               Mumbai Indians   
8              Deccan Chargers             Rajasthan Royals   
9              Kings XI Punjab               Mumbai Indians   

   team1_batting_vs_team2_bowling  team2_batting_vs_team1_bowling  \
0                             0.0                             0.0   
1                             0.0                             0.0   
2                             0.0            

In [91]:
# ==========================================
# STEP 43 - FINAL X AND y
# ==========================================

import pandas as pd

# ------------------------------------------
# Make a copy
# ------------------------------------------

df = final_ml.copy()

# Make sure date is datetime
df["date"] = pd.to_datetime(df["date"])

# Sort chronologically
df = df.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)


# ------------------------------------------
# Target
# ------------------------------------------

y = df["team1_win"].astype(int)


# ------------------------------------------
# Features
# ------------------------------------------

feature_columns = [

    # Basic match information
    "season",
    "toss_winner",
    "toss_decision",

    # Team 1 historical batting
    "team1_batting_experience",
    "team1_batting_form",
    "team1_batting_strike_rate",

    # Team 1 historical bowling
    "team1_bowling_experience",
    "team1_bowling_form",
    "team1_bowling_economy",

    # Team 1 recent form
    "team1_recent_runs",
    "team1_recent_wickets",
    "team1_recent_runs_conceded",

    # Team 2 historical batting
    "team2_batting_experience",
    "team2_batting_form",
    "team2_batting_strike_rate",

    # Team 2 historical bowling
    "team2_bowling_experience",
    "team2_bowling_form",
    "team2_bowling_economy",

    # Team 2 recent form
    "team2_recent_runs",
    "team2_recent_wickets",
    "team2_recent_runs_conceded",

    # Matchup
    "team1_batting_vs_team2_bowling",
    "team2_batting_vs_team1_bowling",

    # Matchup sample size
    "team1_matchup_count",
    "team2_matchup_count"
]


X = df[feature_columns].copy()


# ------------------------------------------
# Remove problematic season format
# ------------------------------------------

# Example:
# 2007/08 -> 2007
#
# Convert season to numeric year

X["season"] = (
    X["season"]
    .astype(str)
    .str[:4]
    .astype(int)
)


# ------------------------------------------
# Identify categorical features
# ------------------------------------------

categorical_features = [
    "toss_winner",
    "toss_decision"
]


numeric_features = [
    col for col in feature_columns
    if col not in categorical_features
]


# ------------------------------------------
# Fill missing values
# ------------------------------------------

X[numeric_features] = (
    X[numeric_features]
    .fillna(0)
)

X[categorical_features] = (
    X[categorical_features]
    .fillna("Unknown")
)


# ------------------------------------------
# Final check
# ------------------------------------------

print("======================================")
print("FINAL X / y")
print("======================================")

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "\nNumeric features:",
    len(numeric_features)
)

print(
    "Categorical features:",
    categorical_features
)

print(
    "\nTarget distribution:"
)

print(
    y.value_counts()
)

print(
    "\nFirst 5 rows:"
)

print(
    X.head()
)

FINAL X / y
X shape: (1243, 25)
y shape: (1243,)

Numeric features: 23
Categorical features: ['toss_winner', 'toss_decision']

Target distribution:
team1_win
0    635
1    608
Name: count, dtype: int64

First 5 rows:
   season                  toss_winner toss_decision  \
0    2007  Royal Challengers Bangalore         field   
1    2007          Chennai Super Kings           bat   
2    2007             Rajasthan Royals           bat   
3    2007               Mumbai Indians           bat   
4    2007              Deccan Chargers           bat   

   team1_batting_experience  team1_batting_form  team1_batting_strike_rate  \
0                         0                 0.0                   0.000000   
1                         0                 0.0                   0.000000   
2                         0                 0.0                   0.000000   
3                         0                 0.0                   0.000000   
4                       120               205.0         

In [92]:
# ==========================================
# STEP 44 - CHRONOLOGICAL TRAIN / TEST SPLIT
# ==========================================

# Number of matches
split_date = pd.Timestamp("2023-05-03")

# Use the already sorted df
train_mask = df["date"] < split_date
test_mask = df["date"] >= split_date

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("======================================")
print("CHRONOLOGICAL TRAIN / TEST SPLIT")
print("======================================")

print(
    "Training matches:",
    len(X_train)
)

print(
    "Testing matches:",
    len(X_test)
)

print(
    "\nTraining period:",
    df.loc[train_mask, "date"].min(),
    "to",
    df.loc[train_mask, "date"].max()
)

print(
    "Testing period:",
    df.loc[test_mask, "date"].min(),
    "to",
    df.loc[test_mask, "date"].max()
)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())

CHRONOLOGICAL TRAIN / TEST SPLIT
Training matches: 994
Testing matches: 249

Training period: 2008-04-18 00:00:00 to 2023-05-02 00:00:00
Testing period: 2023-05-03 00:00:00 to 2026-05-31 00:00:00

Training target:
team1_win
1    499
0    495
Name: count, dtype: int64

Testing target:
team1_win
0    140
1    109
Name: count, dtype: int64


In [93]:
# ==========================================
# STEP 45 - TRAIN ALL MODELS
# ==========================================

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import xgboost as xgb


# ==========================================
# 1. PREPROCESSOR
# ==========================================

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# ==========================================
# 2. MODELS
# ==========================================

models = {

    "Logistic Regression": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=5000,
                    C=0.5
                )
            )
        ]
    ),

    "Random Forest": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=8,
                    min_samples_leaf=5,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    ),

    "XGBoost": Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                xgb.XGBClassifier(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.03,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    )
}


# ==========================================
# 3. TRAIN + EVALUATE
# ==========================================

results = []

trained_models = {}

for name, model in models.items():

    print("\n" + "=" * 50)
    print("TRAINING:", name)
    print("=" * 50)

    model.fit(
        X_train,
        y_train
    )

    trained_models[name] = model

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities
    y_prob = model.predict_proba(
        X_test
    )[:, 1]

    # Metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc
    })

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4))


# ==========================================
# 4. MODEL COMPARISON
# ==========================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)

print("\n")
print("=" * 60)
print("FINAL MODEL COMPARISON")
print("=" * 60)

print(
    results_df.to_string(
        index=False
    )
)


# ==========================================
# 5. BEST MODEL
# ==========================================

best_model_name = results_df.loc[
    0,
    "Model"
]

best_model = trained_models[
    best_model_name
]

print("\n")
print("=" * 60)
print("BEST MODEL")
print("=" * 60)

print(
    "Best model:",
    best_model_name
)

print(
    "Best ROC-AUC:",
    round(
        results_df.loc[0, "ROC-AUC"],
        4
    )
)


TRAINING: Logistic Regression
Accuracy : 0.7952
Precision: 0.7589
Recall   : 0.7798
F1 Score : 0.7692
ROC-AUC  : 0.8878

TRAINING: Random Forest
Accuracy : 0.7711
Precision: 0.7364
Recall   : 0.7431
F1 Score : 0.7397
ROC-AUC  : 0.8469

TRAINING: XGBoost
Accuracy : 0.751
Precision: 0.7423
Recall   : 0.6606
F1 Score : 0.699
ROC-AUC  : 0.8429


FINAL MODEL COMPARISON
              Model  Accuracy  Precision   Recall       F1  ROC-AUC
Logistic Regression  0.795181   0.758929 0.779817 0.769231 0.887811
      Random Forest  0.771084   0.736364 0.743119 0.739726 0.846920
            XGBoost  0.751004   0.742268 0.660550 0.699029 0.842857


BEST MODEL
Best model: Logistic Regression
Best ROC-AUC: 0.8878


In [94]:
# ==========================================
# STEP 46 - BEST MODEL ANALYSIS
# ==========================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

# ------------------------------------------
# Use best model
# ------------------------------------------

best_model = trained_models["Logistic Regression"]

# Predictions
y_pred = best_model.predict(X_test)

# Probabilities
y_prob = best_model.predict_proba(X_test)[:, 1]


# ------------------------------------------
# Confusion Matrix
# ------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred
)

print("======================================")
print("CONFUSION MATRIX")
print("======================================")

print(cm)


# ------------------------------------------
# Classification Report
# ------------------------------------------

print("\n======================================")
print("CLASSIFICATION REPORT")
print("======================================")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ------------------------------------------
# Prediction probabilities
# ------------------------------------------

prediction_results = df.loc[
    test_mask,
    [
        "match_id",
        "date",
        "team_1",
        "team_2",
        "winner"
    ]
].copy()

prediction_results["team1_probability"] = y_prob

prediction_results["team2_probability"] = (
    1 - y_prob
)

prediction_results["predicted_winner"] = (
    prediction_results.apply(
        lambda row:
        row["team_1"]
        if row["team1_probability"] >= 0.5
        else row["team_2"],
        axis=1
    )
)

prediction_results["correct"] = (
    prediction_results["predicted_winner"]
    == prediction_results["winner"]
)


# ------------------------------------------
# Display predictions
# ------------------------------------------

print("\n======================================")
print("SAMPLE PREDICTIONS")
print("======================================")

print(
    prediction_results[
        [
            "date",
            "team_1",
            "team_2",
            "winner",
            "predicted_winner",
            "team1_probability",
            "team2_probability",
            "correct"
        ]
    ].head(20).to_string(index=False)
)


# ------------------------------------------
# Overall prediction accuracy
# ------------------------------------------

print("\n======================================")
print("PREDICTION SUMMARY")
print("======================================")

print(
    "Correct predictions:",
    prediction_results["correct"].sum()
)

print(
    "Total predictions:",
    len(prediction_results)
)

print(
    "Accuracy:",
    prediction_results["correct"].mean()
)

CONFUSION MATRIX
[[113  27]
 [ 24  85]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.82      0.81      0.82       140
           1       0.76      0.78      0.77       109

    accuracy                           0.80       249
   macro avg       0.79      0.79      0.79       249
weighted avg       0.80      0.80      0.80       249


SAMPLE PREDICTIONS
      date                      team_1                team_2                      winner            predicted_winner  team1_probability  team2_probability  correct
2023-05-03        Lucknow Super Giants   Chennai Super Kings                           0         Chennai Super Kings           0.001841           0.998159    False
2023-05-03                Punjab Kings        Mumbai Indians              Mumbai Indians              Mumbai Indians           0.241054           0.758946     True
2023-05-04       Kolkata Knight Riders   Sunrisers Hyderabad       Kolkata Knight Riders       Kol

In [95]:
# ==========================================
# STEP 47 - VERIFY TEST PREDICTION ALIGNMENT
# ==========================================

# Create a clean test dataframe using the
# SAME indices as X_test

test_check = df.loc[X_test.index].copy()

# Model predictions
test_check["actual"] = y_test.values
test_check["predicted"] = y_pred
test_check["team1_probability"] = y_prob
test_check["team2_probability"] = 1 - y_prob

# Predicted winner
test_check["predicted_winner"] = test_check.apply(
    lambda row:
        row["team_1"]
        if row["predicted"] == 1
        else row["team_2"],
    axis=1
)

# Check winner
test_check["correct"] = (
    test_check["predicted_winner"]
    == test_check["winner"]
)

print("======================================")
print("ALIGNMENT CHECK")
print("======================================")

print("X_test rows:", len(X_test))
print("y_test rows:", len(y_test))
print("Predictions:", len(y_pred))
print("Probabilities:", len(y_prob))

print("\nCorrect predictions:")
print(test_check["correct"].sum())

print("\nTotal predictions:")
print(len(test_check))

print("\nActual accuracy:")
print(test_check["correct"].mean())

print("\nModel accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nDifference:")
print(
    abs(
        test_check["correct"].mean()
        - accuracy_score(y_test, y_pred)
    )
)

print("\nFirst 10 predictions:")

print(
    test_check[
        [
            "date",
            "team_1",
            "team_2",
            "winner",
            "predicted_winner",
            "team1_probability",
            "team2_probability",
            "correct"
        ]
    ].head(10).to_string(index=False)
)

ALIGNMENT CHECK
X_test rows: 249
y_test rows: 249
Predictions: 249
Probabilities: 249

Correct predictions:
193

Total predictions:
249

Actual accuracy:
0.7751004016064257

Model accuracy:
0.7951807228915663

Difference:
0.02008032128514059

First 10 predictions:
      date                      team_1                team_2                winner            predicted_winner  team1_probability  team2_probability  correct
2023-05-03        Lucknow Super Giants   Chennai Super Kings                     0         Chennai Super Kings           0.001841           0.998159    False
2023-05-03                Punjab Kings        Mumbai Indians        Mumbai Indians              Mumbai Indians           0.241054           0.758946     True
2023-05-04       Kolkata Knight Riders   Sunrisers Hyderabad Kolkata Knight Riders       Kolkata Knight Riders           0.751077           0.248923     True
2023-05-05            Rajasthan Royals        Gujarat Titans        Gujarat Titans              Gujarat

In [96]:
# ==========================================
# STEP 48 - FINAL ACCURACY VERIFICATION
# ==========================================

# Correct method: compare binary target directly
correct_binary = (
    y_test.values == y_pred
)

print("======================================")
print("FINAL ACCURACY CHECK")
print("======================================")

print(
    "Correct predictions:",
    correct_binary.sum()
)

print(
    "Total predictions:",
    len(correct_binary)
)

print(
    "Final Accuracy:",
    correct_binary.mean()
)

print(
    "Sklearn Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

print(
    "\nROC-AUC:",
    roc_auc_score(
        y_test,
        y_prob
    )
)

FINAL ACCURACY CHECK
Correct predictions: 198
Total predictions: 249
Final Accuracy: 0.7951807228915663
Sklearn Accuracy: 0.7951807228915663

ROC-AUC: 0.8878112712975098


In [97]:
# ==========================================
# STEP 49 - SAVE BEST MODEL
# ==========================================

import joblib

best_model = trained_models["Logistic Regression"]

joblib.dump(
    best_model,
    "versus_logistic_regression.pkl"
)

print("======================================")
print("MODEL SAVED")
print("======================================")

print(
    "File: versus_logistic_regression.pkl"
)

MODEL SAVED
File: versus_logistic_regression.pkl


In [98]:
# ==========================================
# STEP 50 - VERSUS PREDICTION FUNCTION
# ==========================================

import pandas as pd
import numpy as np
import joblib


# ------------------------------------------
# Load saved model
# ------------------------------------------

versus_model = joblib.load(
    "versus_logistic_regression.pkl"
)


# ------------------------------------------
# Features used during training
# ------------------------------------------

MODEL_FEATURES = list(X.columns)

print("Model features:", len(MODEL_FEATURES))


# ------------------------------------------
# Prediction function
# ------------------------------------------

def predict_match(team1, team2, toss_winner=None, toss_decision=None):

    # --------------------------------------
    # Check teams
    # --------------------------------------

    if team1 == team2:
        raise ValueError(
            "Team 1 and Team 2 cannot be the same."
        )

    # --------------------------------------
    # Find latest match involving each team
    # --------------------------------------

    team1_matches = df[
        (df["team_1"] == team1) |
        (df["team_2"] == team1)
    ].sort_values("date")

    team2_matches = df[
        (df["team_1"] == team2) |
        (df["team_2"] == team2)
    ].sort_values("date")

    if len(team1_matches) == 0:
        raise ValueError(
            f"No historical data found for {team1}"
        )

    if len(team2_matches) == 0:
        raise ValueError(
            f"No historical data found for {team2}"
        )

    # --------------------------------------
    # Use latest available match features
    # --------------------------------------

    latest_team1 = team1_matches.iloc[-1]
    latest_team2 = team2_matches.iloc[-1]

    # --------------------------------------
    # Helper function
    # --------------------------------------

    def get_team_value(row, team, feature_name):

        if row["team_1"] == team:
            return row.get(
                "team1_" + feature_name,
                0
            )

        return row.get(
            "team2_" + feature_name,
            0
        )

    # --------------------------------------
    # Build feature dictionary
    # --------------------------------------

    features = {}

    # Season
    features["season"] = int(
        pd.to_numeric(
            df["season"],
            errors="coerce"
        ).max()
    )

    # Toss
    features["toss_winner"] = (
        toss_winner
        if toss_winner is not None
        else team1
    )

    features["toss_decision"] = (
        toss_decision
        if toss_decision is not None
        else "field"
    )

    # --------------------------------------
    # Team 1
    # --------------------------------------

    features[
        "team1_batting_experience"
    ] = get_team_value(
        latest_team1,
        team1,
        "batting_experience"
    )

    features[
        "team1_batting_form"
    ] = get_team_value(
        latest_team1,
        team1,
        "batting_form"
    )

    features[
        "team1_batting_strike_rate"
    ] = get_team_value(
        latest_team1,
        team1,
        "batting_strike_rate"
    )

    features[
        "team1_bowling_experience"
    ] = get_team_value(
        latest_team1,
        team1,
        "bowling_experience"
    )

    features[
        "team1_bowling_form"
    ] = get_team_value(
        latest_team1,
        team1,
        "bowling_form"
    )

    features[
        "team1_bowling_economy"
    ] = get_team_value(
        latest_team1,
        team1,
        "bowling_economy"
    )

    # --------------------------------------
    # Team 2
    # --------------------------------------

    features[
        "team2_batting_experience"
    ] = get_team_value(
        latest_team2,
        team2,
        "batting_experience"
    )

    features[
        "team2_batting_form"
    ] = get_team_value(
        latest_team2,
        team2,
        "batting_form"
    )

    features[
        "team2_batting_strike_rate"
    ] = get_team_value(
        latest_team2,
        team2,
        "batting_strike_rate"
    )

    features[
        "team2_bowling_experience"
    ] = get_team_value(
        latest_team2,
        team2,
        "bowling_experience"
    )

    features[
        "team2_bowling_form"
    ] = get_team_value(
        latest_team2,
        team2,
        "bowling_form"
    )

    features[
        "team2_bowling_economy"
    ] = get_team_value(
        latest_team2,
        team2,
        "bowling_economy"
    )


    # --------------------------------------
    # Recent form
    # --------------------------------------

    recent_columns = [
        "recent_runs",
        "recent_wickets",
        "recent_runs_conceded",
        "recent_balls_bowled"
    ]

    for col in recent_columns:

        features[
            "team1_" + col
        ] = get_team_value(
            latest_team1,
            team1,
            col
        )

        features[
            "team2_" + col
        ] = get_team_value(
            latest_team2,
            team2,
            col
        )


    # --------------------------------------
    # Matchup features
    #
    # For a first version, use historical
    # team-vs-team matchup strength.
    # --------------------------------------

    matchup_1 = team_matchup_df[
        (
            team_matchup_df["batter_team"]
            == team1
        )
        &
        (
            team_matchup_df["bowler_team"]
            == team2
        )
    ]

    matchup_2 = team_matchup_df[
        (
            team_matchup_df["batter_team"]
            == team2
        )
        &
        (
            team_matchup_df["bowler_team"]
            == team1
        )
    ]

    if len(matchup_1) > 0:

        features[
            "team1_batting_vs_team2_bowling"
        ] = matchup_1.iloc[-1][
            "matchup_strength"
        ]

    else:

        features[
            "team1_batting_vs_team2_bowling"
        ] = 0


    if len(matchup_2) > 0:

        features[
            "team2_batting_vs_team1_bowling"
        ] = matchup_2.iloc[-1][
            "matchup_strength"
        ]

    else:

        features[
            "team2_batting_vs_team1_bowling"
        ] = 0


    # --------------------------------------
    # Create dataframe
    # --------------------------------------

    prediction_df = pd.DataFrame(
        [features]
    )

    # --------------------------------------
    # Add missing columns as 0
    # --------------------------------------

    for col in MODEL_FEATURES:

        if col not in prediction_df.columns:

            prediction_df[col] = 0


    # --------------------------------------
    # Keep EXACT training order
    # --------------------------------------

    prediction_df = prediction_df[
        MODEL_FEATURES
    ]


    # --------------------------------------
    # Prediction
    # --------------------------------------

    team1_probability = versus_model.predict_proba(
        prediction_df
    )[0][1]

    team2_probability = (
        1 - team1_probability
    )

    if team1_probability >= 0.5:

        predicted_winner = team1

    else:

        predicted_winner = team2


    # --------------------------------------
    # Result
    # --------------------------------------

    result = {

        "team1": team1,

        "team2": team2,

        "team1_probability":
            round(
                team1_probability * 100,
                2
            ),

        "team2_probability":
            round(
                team2_probability * 100,
                2
            ),

        "predicted_winner":
            predicted_winner
    }

    return result


print("\nPrediction function created successfully!")

Model features: 25

Prediction function created successfully!


In [99]:
# ==========================================
# STEP 51 - PLAYER STRENGTH DATASET
# ==========================================

import pandas as pd
import numpy as np

# Make a copy
player_strength_df = player_df.copy()

# ------------------------------------------
# Check required columns
# ------------------------------------------

print("Available columns:")
print(player_strength_df.columns.tolist())


# ------------------------------------------
# Create useful player features
# ------------------------------------------

# Batting average
player_strength_df["batting_average"] = (
    player_strength_df["total_runs"] /
    player_strength_df["matches"].replace(0, np.nan)
).fillna(0)


# ------------------------------------------
# Normalize important statistics
# ------------------------------------------

def min_max(series):

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            0.0,
            index=series.index
        )

    return (
        (series - minimum) /
        (maximum - minimum)
    )


player_strength_df["runs_score"] = min_max(
    player_strength_df["total_runs"]
)

player_strength_df["strike_rate_score"] = min_max(
    player_strength_df["career_strike_rate"]
)

player_strength_df["wickets_score"] = min_max(
    player_strength_df["total_wickets"]
)

player_strength_df["economy_score"] = 1 - min_max(
    player_strength_df["career_economy"]
)

player_strength_df["experience_score"] = min_max(
    player_strength_df["matches"]
)


# ------------------------------------------
# Overall player strength
# ------------------------------------------

player_strength_df["player_strength"] = (

    0.30 *
    player_strength_df["runs_score"]

    +

    0.20 *
    player_strength_df["strike_rate_score"]

    +

    0.30 *
    player_strength_df["wickets_score"]

    +

    0.10 *
    player_strength_df["economy_score"]

    +

    0.10 *
    player_strength_df["experience_score"]
)


# ------------------------------------------
# Sort strongest players first
# ------------------------------------------

player_strength_df = (
    player_strength_df
    .sort_values(
        "player_strength",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------
# Display result
# ------------------------------------------

print()
print("======================================")
print("PLAYER STRENGTH DATASET")
print("======================================")

print(
    "Players:",
    len(player_strength_df)
)

print()

print(
    player_strength_df[
        [
            "player",
            "matches",
            "total_runs",
            "career_strike_rate",
            "total_wickets",
            "career_economy",
            "role",
            "player_strength"
        ]
    ].head(20)
)

Available columns:
['match_id', 'date', 'season', 'venue', 'player', 'team', 'runs', 'balls_faced', 'fours', 'sixes', 'runs_conceded', 'balls_bowled', 'wickets', 'dismissals', 'winner', 'previous_runs', 'previous_wickets', 'previous_dismissals', 'last_5_runs', 'last_10_runs', 'last_5_wickets', 'last_10_wickets', 'previous_balls_faced', 'previous_strike_rate', 'last_5_total_runs', 'last_5_total_balls', 'last_5_strike_rate', 'previous_runs_conceded', 'previous_balls_bowled', 'previous_economy', 'last_5_runs_conceded', 'last_5_balls_bowled', 'last_5_economy', 'venue_previous_runs', 'venue_previous_balls', 'venue_previous_strike_rate', 'venue_previous_wickets', 'batting_experience', 'batting_form', 'batting_strike_rate', 'recent_strike_rate', 'bowling_experience', 'bowling_form', 'bowling_economy', 'recent_bowling_economy', 'venue_runs', 'venue_strike_rate', 'venue_wickets']


KeyError: 'total_runs'

In [100]:
# ==========================================
# STEP 51 - PLAYER STRENGTH DATASET
# FIXED VERSION
# ==========================================

import pandas as pd
import numpy as np

# Copy player statistics
player_strength_df = player_df.copy()

print("Columns:")
print(player_strength_df.columns.tolist())


# ------------------------------------------
# Helper: find column
# ------------------------------------------

def find_column(possible_names):

    for name in possible_names:
        if name in player_strength_df.columns:
            return name

    return None


runs_col = find_column([
    "total_runs",
    "runs",
    "career_runs"
])

matches_col = find_column([
    "matches",
    "match_count"
])

sr_col = find_column([
    "career_strike_rate",
    "strike_rate"
])

wickets_col = find_column([
    "total_wickets",
    "wickets",
    "career_wickets"
])

economy_col = find_column([
    "career_economy",
    "economy"
])


print("\nDetected columns:")
print("Runs      :", runs_col)
print("Matches   :", matches_col)
print("StrikeRate:", sr_col)
print("Wickets   :", wickets_col)
print("Economy   :", economy_col)


# ------------------------------------------
# Check
# ------------------------------------------

required = [
    runs_col,
    matches_col,
    sr_col,
    wickets_col,
    economy_col
]

if any(x is None for x in required):

    raise ValueError(
        "Could not find one or more required columns. "
        "See the detected columns printed above."
    )


# ------------------------------------------
# Convert numeric columns
# ------------------------------------------

for col in required:

    player_strength_df[col] = pd.to_numeric(
        player_strength_df[col],
        errors="coerce"
    ).fillna(0)


# ------------------------------------------
# Min-Max normalization
# ------------------------------------------

def min_max(series):

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            0.0,
            index=series.index
        )

    return (
        (series - minimum) /
        (maximum - minimum)
    )


player_strength_df["runs_score"] = min_max(
    player_strength_df[runs_col]
)

player_strength_df["strike_rate_score"] = min_max(
    player_strength_df[sr_col]
)

player_strength_df["wickets_score"] = min_max(
    player_strength_df[wickets_col]
)

player_strength_df["economy_score"] = (
    1 - min_max(
        player_strength_df[economy_col]
    )
)

player_strength_df["experience_score"] = min_max(
    player_strength_df[matches_col]
)


# ------------------------------------------
# Player strength
# ------------------------------------------

player_strength_df["player_strength"] = (

    0.30 *
    player_strength_df["runs_score"]

    +

    0.20 *
    player_strength_df["strike_rate_score"]

    +

    0.30 *
    player_strength_df["wickets_score"]

    +

    0.10 *
    player_strength_df["economy_score"]

    +

    0.10 *
    player_strength_df["experience_score"]
)


# ------------------------------------------
# Sort
# ------------------------------------------

player_strength_df = (
    player_strength_df
    .sort_values(
        "player_strength",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------
# Result
# ------------------------------------------

print("\n======================================")
print("PLAYER STRENGTH DATASET")
print("======================================")

print("Players:", len(player_strength_df))

print()

display(
    player_strength_df[
        [
            "player",
            matches_col,
            runs_col,
            sr_col,
            wickets_col,
            economy_col,
            "role",
            "player_strength"
        ]
    ].head(20)
)

Columns:
['match_id', 'date', 'season', 'venue', 'player', 'team', 'runs', 'balls_faced', 'fours', 'sixes', 'runs_conceded', 'balls_bowled', 'wickets', 'dismissals', 'winner', 'previous_runs', 'previous_wickets', 'previous_dismissals', 'last_5_runs', 'last_10_runs', 'last_5_wickets', 'last_10_wickets', 'previous_balls_faced', 'previous_strike_rate', 'last_5_total_runs', 'last_5_total_balls', 'last_5_strike_rate', 'previous_runs_conceded', 'previous_balls_bowled', 'previous_economy', 'last_5_runs_conceded', 'last_5_balls_bowled', 'last_5_economy', 'venue_previous_runs', 'venue_previous_balls', 'venue_previous_strike_rate', 'venue_previous_wickets', 'batting_experience', 'batting_form', 'batting_strike_rate', 'recent_strike_rate', 'bowling_experience', 'bowling_form', 'bowling_economy', 'recent_bowling_economy', 'venue_runs', 'venue_strike_rate', 'venue_wickets']

Detected columns:
Runs      : runs
Matches   : None
StrikeRate: None
Wickets   : wickets
Economy   : None


ValueError: Could not find one or more required columns. See the detected columns printed above.

In [101]:
print(player_df.columns.tolist())

['match_id', 'date', 'season', 'venue', 'player', 'team', 'runs', 'balls_faced', 'fours', 'sixes', 'runs_conceded', 'balls_bowled', 'wickets', 'dismissals', 'winner', 'previous_runs', 'previous_wickets', 'previous_dismissals', 'last_5_runs', 'last_10_runs', 'last_5_wickets', 'last_10_wickets', 'previous_balls_faced', 'previous_strike_rate', 'last_5_total_runs', 'last_5_total_balls', 'last_5_strike_rate', 'previous_runs_conceded', 'previous_balls_bowled', 'previous_economy', 'last_5_runs_conceded', 'last_5_balls_bowled', 'last_5_economy', 'venue_previous_runs', 'venue_previous_balls', 'venue_previous_strike_rate', 'venue_previous_wickets', 'batting_experience', 'batting_form', 'batting_strike_rate', 'recent_strike_rate', 'bowling_experience', 'bowling_form', 'bowling_economy', 'recent_bowling_economy', 'venue_runs', 'venue_strike_rate', 'venue_wickets']


In [103]:
# ==========================================
# STEP 51 - PLAYER STRENGTH DATASET
# FINAL FIX
# ==========================================

import pandas as pd
import numpy as np

# Sort data
player_df["date"] = pd.to_datetime(player_df["date"])

player_df = player_df.sort_values(
    ["player", "date", "match_id"]
).reset_index(drop=True)


# ------------------------------------------
# Latest record for every player
# ------------------------------------------

latest = (
    player_df
    .groupby("player", as_index=False)
    .tail(1)
    .copy()
)


# ------------------------------------------
# Career statistics
# ------------------------------------------

career = (
    player_df
    .groupby("player")
    .agg(
        matches=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        total_balls_faced=("balls_faced", "sum"),
        total_wickets=("wickets", "sum"),
        total_runs_conceded=("runs_conceded", "sum"),
        total_balls_bowled=("balls_bowled", "sum")
    )
    .reset_index()
)


# ------------------------------------------
# Merge
# ------------------------------------------

player_strength_df = latest.merge(
    career,
    on="player",
    how="left",
    suffixes=("", "_career")
)


# ------------------------------------------
# Career strike rate
# ------------------------------------------

player_strength_df["career_strike_rate"] = np.where(
    player_strength_df["total_balls_faced"] > 0,
    player_strength_df["total_runs"]
    / player_strength_df["total_balls_faced"] * 100,
    0
)


# ------------------------------------------
# Career economy
# ------------------------------------------

player_strength_df["career_economy"] = np.where(
    player_strength_df["total_balls_bowled"] > 0,
    player_strength_df["total_runs_conceded"]
    / player_strength_df["total_balls_bowled"] * 6,
    0
)


# ------------------------------------------
# Normalization
# ------------------------------------------

def normalize(s):

    if s.max() == s.min():
        return pd.Series(0.0, index=s.index)

    return (s - s.min()) / (s.max() - s.min())


player_strength_df["runs_score"] = normalize(
    player_strength_df["total_runs"]
)

player_strength_df["strike_rate_score"] = normalize(
    player_strength_df["career_strike_rate"]
)

player_strength_df["wickets_score"] = normalize(
    player_strength_df["total_wickets"]
)

player_strength_df["experience_score"] = normalize(
    player_strength_df["matches"]
)

player_strength_df["economy_score"] = (
    1 - normalize(
        player_strength_df["career_economy"]
    )
)


# ------------------------------------------
# Overall player strength
# ------------------------------------------

player_strength_df["player_strength"] = (

    0.30 * player_strength_df["runs_score"]

    + 0.20 *
    player_strength_df["strike_rate_score"]

    + 0.30 *
    player_strength_df["wickets_score"]

    + 0.10 *
    player_strength_df["economy_score"]

    + 0.10 *
    player_strength_df["experience_score"]
)


# ------------------------------------------
# Create role automatically
# ------------------------------------------

def determine_role(row):

    batting = row["total_balls_faced"]
    bowling = row["total_balls_bowled"]
    wickets = row["total_wickets"]

    if batting > 0 and bowling > 0:
        return "all_rounder"

    elif bowling > 0 or wickets > 0:
        return "bowler"

    else:
        return "batter"


player_strength_df["role"] = (
    player_strength_df
    .apply(determine_role, axis=1)
)


# ------------------------------------------
# Sort
# ------------------------------------------

player_strength_df = (
    player_strength_df
    .sort_values(
        "player_strength",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------
# Result
# ------------------------------------------

print("======================================")
print("PLAYER STRENGTH DATASET")
print("======================================")

print("Players:", len(player_strength_df))

display(
    player_strength_df[
        [
            "player",
            "matches",
            "total_runs",
            "career_strike_rate",
            "total_wickets",
            "career_economy",
            "role",
            "player_strength"
        ]
    ].head(20)
)

PLAYER STRENGTH DATASET
Players: 811


,player,matches,total_runs,career_strike_rate,total_wickets,career_economy,role,player_strength
0,RA Jadeja,267,3526,131.322160,180,7.708039,all_rounder,0.574473
1,SP Narine,201,1820,165.605096,209,6.789270,all_rounder,0.550808
2,V Kohli,282,9346,135.233685,4,8.796813,all_rounder,0.538672
3,B Kumar,206,354,94.148936,226,7.712609,all_rounder,0.494230
4,RG Sharma,281,7331,133.387918,15,8.017699,all_rounder,0.489172
5,R Ashwin,219,833,118.492176,187,7.200000,all_rounder,0.475874
6,DJ Bravo,160,1560,130.434783,183,8.384615,all_rounder,0.474891
7,YS Chahal,189,37,43.023256,233,8.051028,all_rounder,0.464166
8,PP Chawla,192,624,111.229947,192,7.960519,all_rounder,0.460769
9,Rashid Khan,153,635,158.750000,179,7.276884,all_rounder,0.452770


In [104]:
# ==========================================
# STEP 52 - CUSTOM TEAM BUILDER
# ==========================================

def build_custom_team(player_names, team_name="Custom Team"):

    # --------------------------------------
    # Remove duplicate players
    # --------------------------------------

    player_names = list(dict.fromkeys(player_names))

    # --------------------------------------
    # Check players exist
    # --------------------------------------

    available_players = set(
        player_strength_df["player"]
    )

    missing_players = [
        p for p in player_names
        if p not in available_players
    ]

    if missing_players:
        raise ValueError(
            f"Players not found: {missing_players}"
        )

    # --------------------------------------
    # Get player data
    # --------------------------------------

    team = player_strength_df[
        player_strength_df["player"].isin(
            player_names
        )
    ].copy()

    # Keep requested order
    team["player_order"] = (
        team["player"]
        .apply(player_names.index)
    )

    team = (
        team
        .sort_values("player_order")
        .drop(columns="player_order")
    )

    # --------------------------------------
    # Team statistics
    # --------------------------------------

    batting_strength = (
        team["runs_score"].mean()
        + team["strike_rate_score"].mean()
    ) / 2

    bowling_strength = (
        team["wickets_score"].mean()
        + team["economy_score"].mean()
    ) / 2

    experience = (
        team["experience_score"].mean()
    )

    recent_form = (
        team["batting_form"].mean()
        if "batting_form" in team.columns
        else 0
    )

    overall_strength = (
        0.40 * batting_strength
        +
        0.40 * bowling_strength
        +
        0.10 * experience
        +
        0.10 * min(
            recent_form / 100,
            1
        )
    )

    # --------------------------------------
    # Result
    # --------------------------------------

    result = {

        "team_name": team_name,

        "players": player_names,

        "player_data": team,

        "batting_strength":
            round(
                batting_strength * 100,
                2
            ),

        "bowling_strength":
            round(
                bowling_strength * 100,
                2
            ),

        "experience":
            round(
                experience * 100,
                2
            ),

        "overall_strength":
            round(
                overall_strength * 100,
                2
            )
    }

    return result


# ==========================================
# TEST CUSTOM TEAMS
# ==========================================

team1_players = [
    "DA Warner",
    "S Dhawan",
    "Yuvraj Singh",
    "DJ Hooda",
    "B Kumar"
]

team2_players = [
    "CH Gayle",
    "Mandeep Singh",
    "SR Watson",
    "TS Mills",
    "YS Chahal"
]


team1 = build_custom_team(
    team1_players,
    "Team 1"
)

team2 = build_custom_team(
    team2_players,
    "Team 2"
)


# ==========================================
# DISPLAY
# ==========================================

print("======================================")
print("CUSTOM TEAM 1")
print("======================================")

print("Players:")
for player in team1["players"]:
    print("-", player)

print(
    "\nBatting strength:",
    team1["batting_strength"]
)

print(
    "Bowling strength:",
    team1["bowling_strength"]
)

print(
    "Overall strength:",
    team1["overall_strength"]
)


print("\n======================================")
print("CUSTOM TEAM 2")
print("======================================")

print("Players:")
for player in team2["players"]:
    print("-", player)

print(
    "\nBatting strength:",
    team2["batting_strength"]
)

print(
    "Bowling strength:",
    team2["bowling_strength"]
)

print(
    "Overall strength:",
    team2["overall_strength"]
)

CUSTOM TEAM 1
Players:
- DA Warner
- S Dhawan
- Yuvraj Singh
- DJ Hooda
- B Kumar

Batting strength: 32.49
Bowling strength: 49.61
Overall strength: 40.54

CUSTOM TEAM 2
Players:
- CH Gayle
- Mandeep Singh
- SR Watson
- TS Mills
- YS Chahal

Batting strength: 22.34
Bowling strength: 52.21
Overall strength: 35.18


In [105]:
# ==========================================
# STEP 53 - CUSTOM TEAM MATCHUP STRENGTH
# ==========================================

def calculate_team_matchup(team1_players, team2_players):

    # --------------------------------------
    # Team 1 batting vs Team 2 bowling
    # --------------------------------------

    t1_vs_t2 = matchup_history[
        matchup_history["batter"].isin(team1_players)
        &
        matchup_history["bowler"].isin(team2_players)
    ].copy()

    # --------------------------------------
    # Team 2 batting vs Team 1 bowling
    # --------------------------------------

    t2_vs_t1 = matchup_history[
        matchup_history["batter"].isin(team2_players)
        &
        matchup_history["bowler"].isin(team1_players)
    ].copy()


    # --------------------------------------
    # Calculate strength
    # --------------------------------------

    if len(t1_vs_t2) > 0:

        team1_matchup_strength = (
            t1_vs_t2["matchup_score"].mean()
        )

        team1_matchup_count = len(t1_vs_t2)

    else:

        team1_matchup_strength = 0
        team1_matchup_count = 0


    if len(t2_vs_t1) > 0:

        team2_matchup_strength = (
            t2_vs_t1["matchup_score"].mean()
        )

        team2_matchup_count = len(t2_vs_t1)

    else:

        team2_matchup_strength = 0
        team2_matchup_count = 0


    # --------------------------------------
    # Result
    # --------------------------------------

    result = {

        "team1_vs_team2_strength":
            round(
                team1_matchup_strength,
                2
            ),

        "team2_vs_team1_strength":
            round(
                team2_matchup_strength,
                2
            ),

        "team1_matchup_count":
            team1_matchup_count,

        "team2_matchup_count":
            team2_matchup_count
    }

    return result


# ==========================================
# TEST
# ==========================================

matchup_result = calculate_team_matchup(
    team1_players,
    team2_players
)

print("======================================")
print("CUSTOM TEAM MATCHUP")
print("======================================")

print(
    "Team 1 batting vs Team 2 bowling:",
    matchup_result[
        "team1_vs_team2_strength"
    ]
)

print(
    "Team 2 batting vs Team 1 bowling:",
    matchup_result[
        "team2_vs_team1_strength"
    ]
)

print(
    "Team 1 matchup records:",
    matchup_result[
        "team1_matchup_count"
    ]
)

print(
    "Team 2 matchup records:",
    matchup_result[
        "team2_matchup_count"
    ]
)

CUSTOM TEAM MATCHUP
Team 1 batting vs Team 2 bowling: 47.22
Team 2 batting vs Team 1 bowling: 63.89
Team 1 matchup records: 79
Team 2 matchup records: 45


In [107]:
# ==========================================
# STEP 54 - CUSTOM TEAM COMPARISON ENGINE
# ==========================================

def predict_custom_match(team1_players, team2_players):

    # --------------------------------------
    # Build both teams
    # --------------------------------------

    team1 = build_custom_team(
        team1_players,
        "Team 1"
    )

    team2 = build_custom_team(
        team2_players,
        "Team 2"
    )

    # --------------------------------------
    # Calculate player matchup strength
    # --------------------------------------

    matchup = calculate_team_matchup(
        team1_players,
        team2_players
    )

    # --------------------------------------
    # Get values
    # --------------------------------------

    team1_strength = team1["overall_strength"]
    team2_strength = team2["overall_strength"]

    team1_matchup = (
        matchup["team1_vs_team2_strength"]
    )

    team2_matchup = (
        matchup["team2_vs_team1_strength"]
    )

    # --------------------------------------
    # Normalize matchup difference
    # --------------------------------------

    matchup_total = (
        team1_matchup +
        team2_matchup
    )

    if matchup_total > 0:

        team1_matchup_score = (
            team1_matchup /
            matchup_total
        )

        team2_matchup_score = (
            team2_matchup /
            matchup_total
        )

    else:

        team1_matchup_score = 0.5
        team2_matchup_score = 0.5


    # --------------------------------------
    # Normalize team strength
    # --------------------------------------

    strength_total = (
        team1_strength +
        team2_strength
    )

    if strength_total > 0:

        team1_base = (
            team1_strength /
            strength_total
        )

        team2_base = (
            team2_strength /
            strength_total
        )

    else:

        team1_base = 0.5
        team2_base = 0.5


    # --------------------------------------
    # Combine strength + matchup
    # --------------------------------------

    team1_score = (
        0.65 * team1_base
        +
        0.35 * team1_matchup_score
    )

    team2_score = (
        0.65 * team2_base
        +
        0.35 * team2_matchup_score
    )


    # --------------------------------------
    # Convert to probabilities
    # --------------------------------------

    total_score = (
        team1_score +
        team2_score
    )

    team1_probability = (
        team1_score /
        total_score
    )

    team2_probability = (
        team2_score /
        total_score
    )


    # --------------------------------------
    # Winner
    # --------------------------------------

    if team1_probability >= team2_probability:

        predicted_winner = "Team 1"

    else:

        predicted_winner = "Team 2"


    # --------------------------------------
    # Final result
    # --------------------------------------

    result = {

        "team1_players":
            team1_players,

        "team2_players":
            team2_players,

        "team1_strength":
            round(
                team1_strength * 100,
                2
            ),

        "team2_strength":
            round(
                team2_strength * 100,
                2
            ),

        "team1_matchup":
            round(
                team1_matchup,
                2
            ),

        "team2_matchup":
            round(
                team2_matchup,
                2
            ),

        "team1_probability":
            round(
                team1_probability * 100,
                2
            ),

        "team2_probability":
            round(
                team2_probability * 100,
                2
            ),

        "predicted_winner":
            predicted_winner
    }

    return result


# ==========================================
# TEST
# ==========================================

result = predict_custom_match(
    team1_players,
    team2_players
)


# ==========================================
# DISPLAY
# ==========================================

print("======================================")
print("VERSUS - CUSTOM TEAM PREDICTION")
print("======================================")

print()

print("TEAM 1")
print("------")

for player in team1_players:
    print("-", player)

print()

print(
    "Team strength:",
    result["team1_strength"]
)

print(
    "Matchup strength:",
    result["team1_matchup"]
)

print(
    "Win probability:",
    result["team1_probability"],
    "%"
)


print("\nTEAM 2")
print("------")

for player in team2_players:
    print("-", player)

print()

print(
    "Team strength:",
    result["team2_strength"]
)

print(
    "Matchup strength:",
    result["team2_matchup"]
)

print(
    "Win probability:",
    result["team2_probability"],
    "%"
)


print("\n======================================")

print(
    "PREDICTED WINNER:",
    result["predicted_winner"]
)

print("======================================")

VERSUS - CUSTOM TEAM PREDICTION

TEAM 1
------
- DA Warner
- S Dhawan
- Yuvraj Singh
- DJ Hooda
- B Kumar

Team strength: 4054.0
Matchup strength: 47.22
Win probability: 49.68 %

TEAM 2
------
- CH Gayle
- Mandeep Singh
- SR Watson
- TS Mills
- YS Chahal

Team strength: 3518.0
Matchup strength: 63.89
Win probability: 50.32 %

PREDICTED WINNER: Team 2


In [108]:
print("======================================")
print("FINAL CUSTOM TEAM PREDICTION")
print("======================================")

print("\nTEAM 1")
print("Players:", ", ".join(team1_players))
print("Strength:", round(result["team1_strength"] / 100, 2))
print("Matchup:", result["team1_matchup"])
print("Win Probability:", result["team1_probability"], "%")

print("\nTEAM 2")
print("Players:", ", ".join(team2_players))
print("Strength:", round(result["team2_strength"] / 100, 2))
print("Matchup:", result["team2_matchup"])
print("Win Probability:", result["team2_probability"], "%")

print("\n======================================")
print("PREDICTED WINNER:", result["predicted_winner"])
print("======================================")

FINAL CUSTOM TEAM PREDICTION

TEAM 1
Players: DA Warner, S Dhawan, Yuvraj Singh, DJ Hooda, B Kumar
Strength: 40.54
Matchup: 47.22
Win Probability: 49.68 %

TEAM 2
Players: CH Gayle, Mandeep Singh, SR Watson, TS Mills, YS Chahal
Strength: 35.18
Matchup: 63.89
Win Probability: 50.32 %

PREDICTED WINNER: Team 2


In [109]:
# ==========================================
# STEP 55 - CUSTOM TEAM ML PREDICTION
# ==========================================

import joblib
import pandas as pd
import numpy as np


# ------------------------------------------
# 1. Load saved Logistic Regression model
# ------------------------------------------

model = joblib.load(
    "versus_logistic_regression.pkl"
)

print("Model loaded successfully!")


# ------------------------------------------
# 2. Create custom team features
# ------------------------------------------

def create_custom_team_features(
    team1_players,
    team2_players
):

    team1 = build_custom_team(
        team1_players,
        "Team 1"
    )

    team2 = build_custom_team(
        team2_players,
        "Team 2"
    )

    matchup = calculate_team_matchup(
        team1_players,
        team2_players
    )


    # --------------------------------------
    # Matchup strengths
    # --------------------------------------

    t1_matchup = matchup[
        "team1_vs_team2_strength"
    ]

    t2_matchup = matchup[
        "team2_vs_team1_strength"
    ]


    # --------------------------------------
    # Recent form
    # --------------------------------------

    def get_average(players, column):

        values = player_strength_df[
            player_strength_df["player"].isin(players)
        ][column]

        if len(values) == 0:
            return 0

        return values.mean()


    team1_recent_runs = get_average(
        team1_players,
        "last_5_runs"
    )

    team2_recent_runs = get_average(
        team2_players,
        "last_5_runs"
    )


    team1_recent_wickets = get_average(
        team1_players,
        "last_5_wickets"
    )

    team2_recent_wickets = get_average(
        team2_players,
        "last_5_wickets"
    )


    # --------------------------------------
    # Build feature dictionary
    # --------------------------------------

    features = {

        "team1_batting_experience":
            team1["batting_strength"],

        "team1_batting_form":
            team1_recent_runs,

        "team1_batting_strike_rate":
            get_average(
                team1_players,
                "batting_strike_rate"
            ),

        "team1_bowling_experience":
            team1["bowling_strength"],

        "team1_bowling_form":
            team1_recent_wickets,

        "team1_bowling_economy":
            get_average(
                team1_players,
                "bowling_economy"
            ),

        "team2_batting_experience":
            team2["batting_strength"],

        "team2_batting_form":
            team2_recent_runs,

        "team2_batting_strike_rate":
            get_average(
                team2_players,
                "batting_strike_rate"
            ),

        "team2_bowling_experience":
            team2["bowling_strength"],

        "team2_bowling_form":
            team2_recent_wickets,

        "team2_bowling_economy":
            get_average(
                team2_players,
                "bowling_economy"
            ),

        "team1_recent_runs":
            team1_recent_runs,

        "team1_recent_wickets":
            team1_recent_wickets,

        "team2_recent_runs":
            team2_recent_runs,

        "team2_recent_wickets":
            team2_recent_wickets,

        "team1_batting_vs_team2_bowling":
            t1_matchup,

        "team2_batting_vs_team1_bowling":
            t2_matchup
    }


    return pd.DataFrame([features])


# ------------------------------------------
# 3. Create features
# ------------------------------------------

custom_features = create_custom_team_features(
    team1_players,
    team2_players
)


print()
print("======================================")
print("CUSTOM TEAM ML FEATURES")
print("======================================")

print(
    "Shape:",
    custom_features.shape
)

display(custom_features)

Model loaded successfully!

CUSTOM TEAM ML FEATURES
Shape: (1, 18)


,team1_batting_experience,team1_batting_form,team1_batting_strike_rate,team1_bowling_experience,team1_bowling_form,team1_bowling_economy,team2_batting_experience,team2_batting_form,team2_batting_strike_rate,team2_bowling_experience,team2_bowling_form,team2_bowling_economy,team1_recent_runs,team1_recent_wickets,team2_recent_runs,team2_recent_wickets,team1_batting_vs_team2_bowling,team2_batting_vs_team1_bowling
0,32.49,15.52,124.184262,49.61,1.8,8.788545,22.34,11.52,103.396294,52.21,2.2,9.193328,15.52,1.8,11.52,2.2,47.22,63.89


In [110]:
# ==========================================
# STEP 56 - CHECK EXACT MODEL FEATURES
# ==========================================

print("======================================")
print("MODEL FEATURE CHECK")
print("======================================")

print("Model type:")
print(type(model))

print()

# Check pipeline steps
if hasattr(model, "named_steps"):

    print("Pipeline steps:")
    print(model.named_steps.keys())

    print()

    for name, step in model.named_steps.items():

        print(
            name,
            "->",
            type(step)
        )

else:

    print("Model does not contain named steps.")


print()
print("Custom feature count:")
print(len(custom_features.columns))

print()
print("Custom features:")
print(list(custom_features.columns))

MODEL FEATURE CHECK
Model type:
<class 'sklearn.pipeline.Pipeline'>

Pipeline steps:
dict_keys(['preprocessor', 'classifier'])

preprocessor -> <class 'sklearn.compose._column_transformer.ColumnTransformer'>
classifier -> <class 'sklearn.linear_model._logistic.LogisticRegression'>

Custom feature count:
18

Custom features:
['team1_batting_experience', 'team1_batting_form', 'team1_batting_strike_rate', 'team1_bowling_experience', 'team1_bowling_form', 'team1_bowling_economy', 'team2_batting_experience', 'team2_batting_form', 'team2_batting_strike_rate', 'team2_bowling_experience', 'team2_bowling_form', 'team2_bowling_economy', 'team1_recent_runs', 'team1_recent_wickets', 'team2_recent_runs', 'team2_recent_wickets', 'team1_batting_vs_team2_bowling', 'team2_batting_vs_team1_bowling']


In [111]:
# ==========================================
# STEP 56 - EXACT TRAINED FEATURES
# ==========================================

preprocessor = model.named_steps["preprocessor"]

print("======================================")
print("TRAINED MODEL FEATURES")
print("======================================")

print("Numeric features:")
print(preprocessor.transformers_[0][2])

print()

print("Categorical features:")
print(preprocessor.transformers_[1][2])

print()

print("Total features expected by preprocessor:")

numeric_features = preprocessor.transformers_[0][2]
categorical_features = preprocessor.transformers_[1][2]

print(
    len(numeric_features) +
    len(categorical_features)
)

TRAINED MODEL FEATURES
Numeric features:
['season', 'team1_batting_experience', 'team1_batting_form', 'team1_batting_strike_rate', 'team1_bowling_experience', 'team1_bowling_form', 'team1_bowling_economy', 'team1_recent_runs', 'team1_recent_wickets', 'team1_recent_runs_conceded', 'team2_batting_experience', 'team2_batting_form', 'team2_batting_strike_rate', 'team2_bowling_experience', 'team2_bowling_form', 'team2_bowling_economy', 'team2_recent_runs', 'team2_recent_wickets', 'team2_recent_runs_conceded', 'team1_batting_vs_team2_bowling', 'team2_batting_vs_team1_bowling', 'team1_matchup_count', 'team2_matchup_count']

Categorical features:
['toss_winner', 'toss_decision']

Total features expected by preprocessor:
25


In [112]:
# ==========================================
# STEP 57 - CUSTOM TEAM + ML PREDICTION
# ==========================================

def predict_custom_match_ml(
    team1_players,
    team2_players,
    toss_winner="Team 1",
    toss_decision="field",
    season=2026
):

    # --------------------------------------
    # 1. Build custom teams
    # --------------------------------------

    team1 = build_custom_team(
        team1_players,
        "Team 1"
    )

    team2 = build_custom_team(
        team2_players,
        "Team 2"
    )

    # --------------------------------------
    # 2. Matchup statistics
    # --------------------------------------

    matchup = calculate_team_matchup(
        team1_players,
        team2_players
    )

    # --------------------------------------
    # 3. Helper function
    # --------------------------------------

    def avg(players, column):

        data = player_strength_df[
            player_strength_df["player"].isin(players)
        ]

        if column not in data.columns:
            return 0.0

        if len(data) == 0:
            return 0.0

        return float(
            data[column].mean()
        )

    # --------------------------------------
    # 4. Team 1 features
    # --------------------------------------

    team1_batting_experience = avg(
        team1_players,
        "batting_experience"
    )

    team1_batting_form = avg(
        team1_players,
        "batting_form"
    )

    team1_batting_strike_rate = avg(
        team1_players,
        "batting_strike_rate"
    )

    team1_bowling_experience = avg(
        team1_players,
        "bowling_experience"
    )

    team1_bowling_form = avg(
        team1_players,
        "bowling_form"
    )

    team1_bowling_economy = avg(
        team1_players,
        "bowling_economy"
    )

    team1_recent_runs = avg(
        team1_players,
        "last_5_runs"
    )

    team1_recent_wickets = avg(
        team1_players,
        "last_5_wickets"
    )

    team1_recent_runs_conceded = avg(
        team1_players,
        "last_5_runs_conceded"
    )

    # --------------------------------------
    # 5. Team 2 features
    # --------------------------------------

    team2_batting_experience = avg(
        team2_players,
        "batting_experience"
    )

    team2_batting_form = avg(
        team2_players,
        "batting_form"
    )

    team2_batting_strike_rate = avg(
        team2_players,
        "batting_strike_rate"
    )

    team2_bowling_experience = avg(
        team2_players,
        "bowling_experience"
    )

    team2_bowling_form = avg(
        team2_players,
        "bowling_form"
    )

    team2_bowling_economy = avg(
        team2_players,
        "bowling_economy"
    )

    team2_recent_runs = avg(
        team2_players,
        "last_5_runs"
    )

    team2_recent_wickets = avg(
        team2_players,
        "last_5_wickets"
    )

    team2_recent_runs_conceded = avg(
        team2_players,
        "last_5_runs_conceded"
    )

    # --------------------------------------
    # 6. Build exact 25 model features
    # --------------------------------------

    ml_input = pd.DataFrame([{

        "season": season,

        "toss_winner": toss_winner,

        "toss_decision": toss_decision,

        "team1_batting_experience":
            team1_batting_experience,

        "team1_batting_form":
            team1_batting_form,

        "team1_batting_strike_rate":
            team1_batting_strike_rate,

        "team1_bowling_experience":
            team1_bowling_experience,

        "team1_bowling_form":
            team1_bowling_form,

        "team1_bowling_economy":
            team1_bowling_economy,

        "team1_recent_runs":
            team1_recent_runs,

        "team1_recent_wickets":
            team1_recent_wickets,

        "team1_recent_runs_conceded":
            team1_recent_runs_conceded,

        "team2_batting_experience":
            team2_batting_experience,

        "team2_batting_form":
            team2_batting_form,

        "team2_batting_strike_rate":
            team2_batting_strike_rate,

        "team2_bowling_experience":
            team2_bowling_experience,

        "team2_bowling_form":
            team2_bowling_form,

        "team2_bowling_economy":
            team2_bowling_economy,

        "team2_recent_runs":
            team2_recent_runs,

        "team2_recent_wickets":
            team2_recent_wickets,

        "team2_recent_runs_conceded":
            team2_recent_runs_conceded,

        "team1_batting_vs_team2_bowling":
            matchup[
                "team1_vs_team2_strength"
            ],

        "team2_batting_vs_team1_bowling":
            matchup[
                "team2_vs_team1_strength"
            ],

        "team1_matchup_count":
            matchup[
                "team1_matchup_count"
            ],

        "team2_matchup_count":
            matchup[
                "team2_matchup_count"
            ]
    }])

    # --------------------------------------
    # 7. Make prediction
    # --------------------------------------

    prediction = model.predict(
        ml_input
    )[0]

    probabilities = model.predict_proba(
        ml_input
    )[0]

    # --------------------------------------
    # 8. Convert prediction
    # --------------------------------------

    team1_probability = probabilities[1]

    team2_probability = probabilities[0]

    if prediction == 1:

        predicted_winner = "Team 1"

    else:

        predicted_winner = "Team 2"

    # --------------------------------------
    # 9. Display
    # --------------------------------------

    print("======================================")
    print("VERSUS ML PREDICTION")
    print("======================================")

    print()

    print("TEAM 1")
    print("------")

    for player in team1_players:
        print("-", player)

    print()

    print("TEAM 2")
    print("------")

    for player in team2_players:
        print("-", player)

    print()

    print("Toss winner:", toss_winner)
    print("Toss decision:", toss_decision)

    print()

    print(
        "Team 1 probability:",
        round(team1_probability * 100, 2),
        "%"
    )

    print(
        "Team 2 probability:",
        round(team2_probability * 100, 2),
        "%"
    )

    print()

    print(
        "PREDICTED WINNER:",
        predicted_winner
    )

    print("======================================")

    return {
        "team1_probability":
            team1_probability,

        "team2_probability":
            team2_probability,

        "predicted_winner":
            predicted_winner,

        "features":
            ml_input
    }


# ==========================================
# TEST
# ==========================================

ml_result = predict_custom_match_ml(
    team1_players,
    team2_players,
    toss_winner="Team 1",
    toss_decision="field",
    season=2026
)

VERSUS ML PREDICTION

TEAM 1
------
- DA Warner
- S Dhawan
- Yuvraj Singh
- DJ Hooda
- B Kumar

TEAM 2
------
- CH Gayle
- Mandeep Singh
- SR Watson
- TS Mills
- YS Chahal

Toss winner: Team 1
Toss decision: field

Team 1 probability: 0.0 %
Team 2 probability: 100.0 %

PREDICTED WINNER: Team 2


In [113]:
# ==========================================
# STEP 58 - CUSTOM FEATURE SCALE CHECK
# ==========================================

print("======================================")
print("CUSTOM FEATURE SCALE CHECK")
print("======================================")


# Get the numeric features used during training
preprocessor = model.named_steps["preprocessor"]

numeric_features = preprocessor.transformers_[0][2]


# ------------------------------------------
# Training data ranges
# ------------------------------------------

print("\nTRAINING DATA RANGES")
print("--------------------------------------")

for col in numeric_features:

    if col in X_train.columns:

        minimum = X_train[col].min()
        maximum = X_train[col].max()
        mean = X_train[col].mean()

        print(
            f"{col:45} "
            f"min={minimum:.2f} "
            f"max={maximum:.2f} "
            f"mean={mean:.2f}"
        )


# ------------------------------------------
# Custom input values
# ------------------------------------------

print("\nCUSTOM TEAM VALUES")
print("--------------------------------------")

for col in numeric_features:

    if col in ml_result["features"].columns:

        value = ml_result["features"].iloc[0][col]

        print(
            f"{col:45} "
            f"value={value:.2f}"
        )


# ------------------------------------------
# Probability
# ------------------------------------------

print("\n======================================")
print("CURRENT MODEL PROBABILITY")
print("======================================")

print(
    "Team 1:",
    round(
        ml_result["team1_probability"] * 100,
        4
    ),
    "%"
)

print(
    "Team 2:",
    round(
        ml_result["team2_probability"] * 100,
        4
    ),
    "%"
)

CUSTOM FEATURE SCALE CHECK

TRAINING DATA RANGES
--------------------------------------
season                                        min=2007.00 max=2023.00 mean=2015.24
team1_batting_experience                      min=0.00 max=27252.00 mean=10316.71
team1_batting_form                            min=0.00 max=234.00 mean=143.54
team1_batting_strike_rate                     min=0.00 max=195.00 mean=125.54
team1_bowling_experience                      min=0.00 max=27255.00 mean=10318.36
team1_bowling_form                            min=0.00 max=1312.00 mean=479.03
team1_bowling_economy                         min=0.00 max=10.75 mean=7.78
team1_recent_runs                             min=0.00 max=234.00 mean=146.78
team1_recent_wickets                          min=0.00 max=9.00 mean=5.25
team1_recent_runs_conceded                    min=0.00 max=215.00 mean=152.24
team2_batting_experience                      min=0.00 max=27612.00 mean=10200.54
team2_batting_form                         

In [114]:
# ==========================================
# STEP 59 - FIX CUSTOM ML FEATURES
# ==========================================

def custom_team_raw_stats(players):

    data = player_strength_df[
        player_strength_df["player"].isin(players)
    ].copy()

    if len(data) == 0:
        raise ValueError("No players found.")

    # --------------------------------------
    # Raw aggregate features
    # --------------------------------------

    batting_experience = data[
        "batting_experience"
    ].sum()

    batting_form = data[
        "batting_form"
    ].sum()

    bowling_experience = data[
        "bowling_experience"
    ].sum()

    bowling_form = data[
        "bowling_form"
    ].sum()


    # --------------------------------------
    # Strike rate
    # Use actual total runs / balls
    # --------------------------------------

    total_runs = data[
        "total_runs"
    ].sum()

    total_balls = data[
        "total_balls_faced"
    ].sum()

    if total_balls > 0:
        batting_strike_rate = (
            total_runs / total_balls
        ) * 100
    else:
        batting_strike_rate = 0


    # --------------------------------------
    # Bowling economy
    # --------------------------------------

    total_conceded = data[
        "total_runs_conceded"
    ].sum()

    total_bowling_balls = data[
        "total_balls_bowled"
    ].sum()

    if total_bowling_balls > 0:
        bowling_economy = (
            total_conceded
            / total_bowling_balls
        ) * 6
    else:
        bowling_economy = 0


    # --------------------------------------
    # Recent runs / wickets
    # --------------------------------------

    recent_runs = data[
        "last_5_runs"
    ].sum()

    recent_wickets = data[
        "last_5_wickets"
    ].sum()

    recent_runs_conceded = data[
        "last_5_runs_conceded"
    ].sum()


    return {
        "batting_experience":
            batting_experience,

        "batting_form":
            batting_form,

        "batting_strike_rate":
            batting_strike_rate,

        "bowling_experience":
            bowling_experience,

        "bowling_form":
            bowling_form,

        "bowling_economy":
            bowling_economy,

        "recent_runs":
            recent_runs,

        "recent_wickets":
            recent_wickets,

        "recent_runs_conceded":
            recent_runs_conceded
    }


# ==========================================
# CREATE CORRECT CUSTOM FEATURES
# ==========================================

def create_fixed_custom_features(
    team1_players,
    team2_players,
    toss_winner="Team 1",
    toss_decision="field"
):

    t1 = custom_team_raw_stats(
        team1_players
    )

    t2 = custom_team_raw_stats(
        team2_players
    )

    matchup = calculate_team_matchup(
        team1_players,
        team2_players
    )


    # --------------------------------------
    # IMPORTANT:
    # Model was trained through 2023.
    # --------------------------------------

    season = 2023


    features = pd.DataFrame([{

        "season": season,

        "toss_winner": toss_winner,

        "toss_decision": toss_decision,


        "team1_batting_experience":
            t1["batting_experience"],

        "team1_batting_form":
            t1["batting_form"],

        "team1_batting_strike_rate":
            t1["batting_strike_rate"],

        "team1_bowling_experience":
            t1["bowling_experience"],

        "team1_bowling_form":
            t1["bowling_form"],

        "team1_bowling_economy":
            t1["bowling_economy"],

        "team1_recent_runs":
            t1["recent_runs"],

        "team1_recent_wickets":
            t1["recent_wickets"],

        "team1_recent_runs_conceded":
            t1["recent_runs_conceded"],


        "team2_batting_experience":
            t2["batting_experience"],

        "team2_batting_form":
            t2["batting_form"],

        "team2_batting_strike_rate":
            t2["batting_strike_rate"],

        "team2_bowling_experience":
            t2["bowling_experience"],

        "team2_bowling_form":
            t2["bowling_form"],

        "team2_bowling_economy":
            t2["bowling_economy"],

        "team2_recent_runs":
            t2["recent_runs"],

        "team2_recent_wickets":
            t2["recent_wickets"],

        "team2_recent_runs_conceded":
            t2["recent_runs_conceded"],


        "team1_batting_vs_team2_bowling":
            matchup[
                "team1_vs_team2_strength"
            ],

        "team2_batting_vs_team1_bowling":
            matchup[
                "team2_vs_team1_strength"
            ],

        "team1_matchup_count":
            matchup[
                "team1_matchup_count"
            ],

        "team2_matchup_count":
            matchup[
                "team2_matchup_count"
            ]

    }])

    return features


# ==========================================
# TEST
# ==========================================

fixed_features = create_fixed_custom_features(
    team1_players,
    team2_players,
    toss_winner="Team 1",
    toss_decision="field"
)


print("======================================")
print("FIXED CUSTOM FEATURES")
print("======================================")

print("Shape:", fixed_features.shape)

display(fixed_features)

FIXED CUSTOM FEATURES
Shape: (1, 25)


,season,toss_winner,toss_decision,team1_batting_experience,team1_batting_form,team1_batting_strike_rate,team1_bowling_experience,team1_bowling_form,team1_bowling_economy,team1_recent_runs,...,team2_bowling_experience,team2_bowling_form,team2_bowling_economy,team2_recent_runs,team2_recent_wickets,team2_recent_runs_conceded,team1_batting_vs_team2_bowling,team2_batting_vs_team1_bowling,team1_matchup_count,team2_matchup_count
0,2023,Team 1,field,17922.0,77.6,131.512975,274.0,9.0,7.736815,77.6,...,352.0,11.0,8.066365,57.6,11.0,286.0,47.22,63.89,79,45


In [115]:
# ==========================================
# STEP 60 - FINAL ML PREDICTION
# ==========================================

# Make sure columns are in exactly
# the same order as training

model_features = (
    numeric_features +
    categorical_features
)

fixed_features = fixed_features[
    model_features
]


# ------------------------------------------
# Prediction
# ------------------------------------------

prediction = model.predict(
    fixed_features
)[0]

probabilities = model.predict_proba(
    fixed_features
)[0]


# ------------------------------------------
# IMPORTANT:
# Check which class corresponds to which value
# ------------------------------------------

classes = model.named_steps[
    "classifier"
].classes_

print("Model classes:", classes)


# Probability for class 1 = Team 1 wins
team1_probability = probabilities[
    list(classes).index(1)
]

# Probability for class 0 = Team 2 wins
team2_probability = probabilities[
    list(classes).index(0)
]


# ------------------------------------------
# Winner
# ------------------------------------------

if prediction == 1:
    predicted_winner = "Team 1"
else:
    predicted_winner = "Team 2"


# ------------------------------------------
# Display
# ------------------------------------------

print()
print("======================================")
print("FINAL VERSUS ML PREDICTION")
print("======================================")

print()

print(
    "Team 1:",
    ", ".join(team1_players)
)

print(
    "Team 2:",
    ", ".join(team2_players)
)

print()

print(
    "Team 1 probability:",
    round(
        team1_probability * 100,
        2
    ),
    "%"
)

print(
    "Team 2 probability:",
    round(
        team2_probability * 100,
        2
    ),
    "%"
)

print()

print(
    "Predicted winner:",
    predicted_winner
)

print("======================================")

Model classes: [0 1]

FINAL VERSUS ML PREDICTION

Team 1: DA Warner, S Dhawan, Yuvraj Singh, DJ Hooda, B Kumar
Team 2: CH Gayle, Mandeep Singh, SR Watson, TS Mills, YS Chahal

Team 1 probability: 0.01 %
Team 2 probability: 99.99 %

Predicted winner: Team 2


In [116]:
# ==========================================
# STEP 61 - PLAYER BASED MATCH DATASET
# ==========================================

import pandas as pd
import numpy as np

print("======================================")
print("BUILDING PLAYER-BASED MATCH DATASET")
print("======================================")


# ------------------------------------------
# Make sure dates are correct
# ------------------------------------------

player_df["date"] = pd.to_datetime(
    player_df["date"]
)

player_df = player_df.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)


# ------------------------------------------
# Get players who played in each match
# ------------------------------------------

match_players = (
    player_df
    .groupby(
        ["match_id", "team"]
    )["player"]
    .apply(list)
    .reset_index()
)


print(
    "Match-team rows:",
    len(match_players)
)


# ------------------------------------------
# Get match information
# ------------------------------------------

match_base = (
    match_info_df[
        [
            "match_id",
            "date",
            "season",
            "team_1",
            "team_2",
            "winner"
        ]
    ]
    .drop_duplicates("match_id")
    .copy()
)


# ------------------------------------------
# Convert season
# ------------------------------------------

def convert_season(value):

    value = str(value)

    if "/" in value:
        return int(
            value.split("/")[0]
        )

    return int(float(value))


match_base["season_num"] = (
    match_base["season"]
    .apply(convert_season)
)


# ------------------------------------------
# Create player lists for Team 1
# and Team 2
# ------------------------------------------

team1_players_map = (
    match_players
    .rename(
        columns={
            "team": "team_1",
            "player": "team1_players"
        }
    )
)

team2_players_map = (
    match_players
    .rename(
        columns={
            "team": "team_2",
            "player": "team2_players"
        }
    )
)


# ------------------------------------------
# Merge Team 1 players
# ------------------------------------------

custom_match_df = match_base.merge(
    team1_players_map[
        [
            "match_id",
            "team_1",
            "team1_players"
        ]
    ],
    on=["match_id", "team_1"],
    how="left"
)


# ------------------------------------------
# Merge Team 2 players
# ------------------------------------------

custom_match_df = custom_match_df.merge(
    team2_players_map[
        [
            "match_id",
            "team_2",
            "team2_players"
        ]
    ],
    on=["match_id", "team_2"],
    how="left"
)


# ------------------------------------------
# Target
# ------------------------------------------

custom_match_df["team1_win"] = (
    custom_match_df["winner"]
    ==
    custom_match_df["team_1"]
).astype(int)


# ------------------------------------------
# Remove matches without player lists
# ------------------------------------------

custom_match_df = custom_match_df[
    custom_match_df["team1_players"].notna()
    &
    custom_match_df["team2_players"].notna()
].copy()


# ------------------------------------------
# Display
# ------------------------------------------

print()
print("======================================")
print("PLAYER-BASED MATCH DATASET")
print("======================================")

print(
    "Shape:",
    custom_match_df.shape
)

print(
    "Matches:",
    len(custom_match_df)
)

print()

display(
    custom_match_df[
        [
            "match_id",
            "date",
            "team_1",
            "team_2",
            "team1_players",
            "team2_players",
            "winner",
            "team1_win"
        ]
    ].head(10)
)

BUILDING PLAYER-BASED MATCH DATASET
Match-team rows: 2486

PLAYER-BASED MATCH DATASET
Shape: (1243, 10)
Matches: 1243



,match_id,date,team_1,team_2,team1_players,team2_players,winner,team1_win
0,1082591,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,"[A Nehra, B Kumar, BCJ Cutting, Bipul Sharma, ...","[A Choudhary, CH Gayle, KM Jadhav, Mandeep Sin...",Sunrisers Hyderabad,1
1,1082592,2017-04-06,Rising Pune Supergiant,Mumbai Indians,"[A Zampa, AB Dinda, AM Rahane, BA Stokes, DL C...","[AT Rayudu, HH Pandya, JC Buttler, JJ Bumrah, ...",Rising Pune Supergiant,1
2,1082593,2017-04-07,Gujarat Lions,Kolkata Knight Riders,"[AJ Finch, BB McCullum, DR Smith, DS Kulkarni,...","[CA Lynn, CR Woakes, G Gambhir, Kuldeep Yadav,...",Kolkata Knight Riders,0
3,1082594,2017-04-08,Kings XI Punjab,Rising Pune Supergiant,"[AR Patel, DA Miller, GJ Maxwell, HM Amla, M V...","[AB Dinda, AM Rahane, BA Stokes, DT Christian,...",Kings XI Punjab,1
4,1082595,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,"[B Stanlake, CH Gayle, Iqbal Abdulla, KM Jadha...","[A Mishra, AP Tare, CH Morris, CR Brathwaite, ...",Royal Challengers Bangalore,1
5,1082596,2017-04-09,Sunrisers Hyderabad,Gujarat Lions,"[A Nehra, B Kumar, BCJ Cutting, Bipul Sharma, ...","[AJ Finch, BB McCullum, Basil Thampi, DR Smith...",Sunrisers Hyderabad,1
6,1082597,2017-04-09,Mumbai Indians,Kolkata Knight Riders,"[HH Pandya, Harbhajan Singh, JC Buttler, JJ Bu...","[AS Rajpoot, CA Lynn, CR Woakes, G Gambhir, Ku...",Mumbai Indians,1
7,1082598,2017-04-10,Kings XI Punjab,Royal Challengers Bangalore,"[AR Patel, DA Miller, GJ Maxwell, HM Amla, M V...","[AB de Villiers, B Stanlake, Iqbal Abdulla, KM...",Kings XI Punjab,1
8,1082599,2017-04-11,Rising Pune Supergiant,Delhi Daredevils,"[A Zampa, AB Dinda, AM Rahane, BA Stokes, DL C...","[A Mishra, AP Tare, CH Morris, CJ Anderson, KK...",Delhi Daredevils,0
9,1082600,2017-04-12,Mumbai Indians,Sunrisers Hyderabad,"[HH Pandya, Harbhajan Singh, JC Buttler, JJ Bu...","[A Nehra, B Kumar, BCJ Cutting, DA Warner, DJ ...",Mumbai Indians,1


In [117]:
# ==========================================
# STEP 62 - PLAYER COMPOSITION FEATURES
# ==========================================

print("======================================")
print("CREATING PLAYER COMPOSITION FEATURES")
print("======================================")


# ------------------------------------------
# Helper: get player statistics
# ------------------------------------------

player_lookup = (
    player_strength_df
    .drop_duplicates("player")
    .set_index("player")
)


def team_features(players):

    rows = []

    for player in players:

        if player in player_lookup.index:

            rows.append(
                player_lookup.loc[player]
            )

    if len(rows) == 0:

        return {
            "batting_experience": 0,
            "batting_form": 0,
            "batting_strike_rate": 0,
            "bowling_experience": 0,
            "bowling_form": 0,
            "bowling_economy": 0,
            "career_runs": 0,
            "career_wickets": 0
        }


    data = pd.DataFrame(rows)


    return {

        "batting_experience":
            data["batting_experience"].sum(),

        "batting_form":
            data["batting_form"].sum(),

        "batting_strike_rate":
            data["batting_strike_rate"].mean(),

        "bowling_experience":
            data["bowling_experience"].sum(),

        "bowling_form":
            data["bowling_form"].sum(),

        "bowling_economy":
            data["bowling_economy"].mean(),

        "career_runs":
            data["total_runs"].sum(),

        "career_wickets":
            data["total_wickets"].sum()
    }


# ------------------------------------------
# Build features
# ------------------------------------------

feature_rows = []


for _, row in custom_match_df.iterrows():

    t1 = team_features(
        row["team1_players"]
    )

    t2 = team_features(
        row["team2_players"]
    )


    feature_rows.append({

        "match_id":
            row["match_id"],

        "date":
            row["date"],

        "season":
            row["season_num"],


        # Team 1
        "team1_batting_experience":
            t1["batting_experience"],

        "team1_batting_form":
            t1["batting_form"],

        "team1_batting_strike_rate":
            t1["batting_strike_rate"],

        "team1_bowling_experience":
            t1["bowling_experience"],

        "team1_bowling_form":
            t1["bowling_form"],

        "team1_bowling_economy":
            t1["bowling_economy"],

        "team1_career_runs":
            t1["career_runs"],

        "team1_career_wickets":
            t1["career_wickets"],


        # Team 2
        "team2_batting_experience":
            t2["batting_experience"],

        "team2_batting_form":
            t2["batting_form"],

        "team2_batting_strike_rate":
            t2["batting_strike_rate"],

        "team2_bowling_experience":
            t2["bowling_experience"],

        "team2_bowling_form":
            t2["bowling_form"],

        "team2_bowling_economy":
            t2["bowling_economy"],

        "team2_career_runs":
            t2["career_runs"],

        "team2_career_wickets":
            t2["career_wickets"],


        # Target
        "team1_win":
            row["team1_win"]
    })


player_ml_df = pd.DataFrame(
    feature_rows
)


print()
print("======================================")
print("PLAYER COMPOSITION ML DATASET")
print("======================================")

print(
    "Shape:",
    player_ml_df.shape
)

print(
    "Rows:",
    len(player_ml_df)
)

print(
    "Features:",
    len(
        player_ml_df.columns
    )
)

print()

display(
    player_ml_df.head()
)

CREATING PLAYER COMPOSITION FEATURES

PLAYER COMPOSITION ML DATASET
Shape: (1243, 20)
Rows: 1243
Features: 20



,match_id,date,season,team1_batting_experience,team1_batting_form,team1_batting_strike_rate,team1_bowling_experience,team1_bowling_form,team1_bowling_economy,team1_career_runs,team1_career_wickets,team2_batting_experience,team2_batting_form,team2_batting_strike_rate,team2_bowling_experience,team2_bowling_form,team2_bowling_economy,team2_career_runs,team2_career_wickets,team1_win
0,1082591,2017-04-05,2017,21551.0,107.6,128.968268,626.0,28.0,7.677037,21596,630,14444.0,119.85,117.802984,428.0,19.0,7.879972,14500,431,1
1,1082592,2017-04-06,2017,19083.0,121.6,112.429704,374.0,27.0,7.297714,19253,379,30819.0,214.60,129.996358,589.0,20.0,6.114838,30915,591,1
2,1082593,2017-04-07,2017,18790.0,112.0,113.777919,316.0,14.0,6.069555,18923,318,25000.0,89.80,125.226484,722.0,26.0,5.074570,25149,730,0
3,1082594,2017-04-08,2017,15075.0,143.0,118.822282,590.0,19.0,5.501443,15273,591,19514.0,122.20,114.369253,363.0,24.0,7.228173,19685,365,1
4,1082595,2017-04-08,2017,13187.0,94.0,123.172744,455.0,25.0,6.982861,13245,457,13510.0,122.00,124.426485,516.0,28.0,4.393459,13592,520,1


In [118]:
# ==========================================
# STEP 63 - LEAKAGE-FREE PLAYER FEATURES
# ==========================================

print("======================================")
print("BUILDING LEAKAGE-FREE PLAYER FEATURES")
print("======================================")


# Make sure dates are datetime
player_df["date"] = pd.to_datetime(player_df["date"])

player_df = player_df.sort_values(
    ["player", "date", "match_id"]
).copy()


# ------------------------------------------
# Features that must exist before the match
# ------------------------------------------

pre_match_cols = [
    "batting_experience",
    "batting_form",
    "batting_strike_rate",
    "bowling_experience",
    "bowling_form",
    "bowling_economy",
    "previous_runs",
    "previous_wickets",
    "previous_balls_faced",
    "previous_strike_rate",
    "previous_runs_conceded",
    "previous_balls_bowled",
    "previous_economy",
    "last_5_runs",
    "last_5_wickets",
    "last_5_strike_rate",
    "last_5_runs_conceded",
    "last_5_economy"
]


# Keep only columns that actually exist
pre_match_cols = [
    c for c in pre_match_cols
    if c in player_df.columns
]


print(
    "Available pre-match features:",
    len(pre_match_cols)
)


# ------------------------------------------
# Build lookup:
#
# player + match_id
#        ↓
# features known BEFORE match
# ------------------------------------------

prematch_player_df = player_df[
    [
        "match_id",
        "date",
        "player"
    ] + pre_match_cols
].copy()


# ------------------------------------------
# Create team feature function
# ------------------------------------------

def calculate_prematch_team_features(
    players,
    match_date
):

    data = prematch_player_df[
        prematch_player_df["player"].isin(players)
        &
        (
            prematch_player_df["date"]
            < match_date
        )
    ].copy()


    if len(data) == 0:

        return {
            col: 0.0
            for col in pre_match_cols
        }


    # Latest available record for each player
    data = (
        data
        .sort_values("date")
        .groupby("player")
        .tail(1)
    )


    result = {}


    for col in pre_match_cols:

        result[col] = (
            pd.to_numeric(
                data[col],
                errors="coerce"
            )
            .fillna(0)
            .mean()
        )


    return result


# ------------------------------------------
# Test on first match
# ------------------------------------------

first_match = custom_match_df.iloc[0]

match_date = pd.to_datetime(
    first_match["date"]
)


t1_test = calculate_prematch_team_features(
    first_match["team1_players"],
    match_date
)

t2_test = calculate_prematch_team_features(
    first_match["team2_players"],
    match_date
)


print()
print("======================================")
print("FIRST MATCH PRE-MATCH CHECK")
print("======================================")

print(
    "Match:",
    first_match["match_id"]
)

print(
    "Date:",
    match_date
)

print()

print("Team 1:")
print(t1_test)

print()

print("Team 2:")
print(t2_test)

BUILDING LEAKAGE-FREE PLAYER FEATURES
Available pre-match features: 18

FIRST MATCH PRE-MATCH CHECK
Match: 1082591
Date: 2017-04-05 00:00:00

Team 1:
{'batting_experience': np.float64(1148.5), 'batting_form': np.float64(15.45), 'batting_strike_rate': np.float64(124.16577086125167), 'bowling_experience': np.float64(28.0), 'bowling_form': np.float64(2.8), 'bowling_economy': np.float64(6.117531744563236), 'previous_runs': np.float64(1148.5), 'previous_wickets': np.float64(28.0), 'previous_balls_faced': np.float64(892.4), 'previous_strike_rate': np.float64(124.16577086125167), 'previous_runs_conceded': np.float64(699.0), 'previous_balls_bowled': np.float64(558.0), 'previous_economy': np.float64(6.117531744563236), 'last_5_runs': np.float64(15.45), 'last_5_wickets': np.float64(2.8), 'last_5_strike_rate': np.float64(135.64049662376078), 'last_5_runs_conceded': np.float64(56.7), 'last_5_economy': np.float64(5.376067100395459)}

Team 2:
{'batting_experience': np.float64(922.3333333333334), 'ba

In [119]:
# ==========================================
# STEP 64 - FINAL PLAYER ML DATASET
# ==========================================

print("======================================")
print("BUILDING FINAL PLAYER ML DATASET")
print("======================================")


rows = []


for _, match in custom_match_df.iterrows():

    match_date = pd.to_datetime(
        match["date"]
    )

    team1 = calculate_prematch_team_features(
        match["team1_players"],
        match_date
    )

    team2 = calculate_prematch_team_features(
        match["team2_players"],
        match_date
    )


    rows.append({

        "match_id":
            match["match_id"],

        "date":
            match_date,

        "season":
            match["season_num"],


        # -----------------------------
        # Team 1
        # -----------------------------

        "team1_batting_experience":
            team1["batting_experience"],

        "team1_batting_form":
            team1["batting_form"],

        "team1_batting_strike_rate":
            team1["batting_strike_rate"],

        "team1_bowling_experience":
            team1["bowling_experience"],

        "team1_bowling_form":
            team1["bowling_form"],

        "team1_bowling_economy":
            team1["bowling_economy"],

        "team1_recent_runs":
            team1["last_5_runs"],

        "team1_recent_wickets":
            team1["last_5_wickets"],

        "team1_recent_strike_rate":
            team1["last_5_strike_rate"],

        "team1_recent_runs_conceded":
            team1["last_5_runs_conceded"],

        "team1_recent_economy":
            team1["last_5_economy"],


        # -----------------------------
        # Team 2
        # -----------------------------

        "team2_batting_experience":
            team2["batting_experience"],

        "team2_batting_form":
            team2["batting_form"],

        "team2_batting_strike_rate":
            team2["batting_strike_rate"],

        "team2_bowling_experience":
            team2["bowling_experience"],

        "team2_bowling_form":
            team2["bowling_form"],

        "team2_bowling_economy":
            team2["bowling_economy"],

        "team2_recent_runs":
            team2["last_5_runs"],

        "team2_recent_wickets":
            team2["last_5_wickets"],

        "team2_recent_strike_rate":
            team2["last_5_strike_rate"],

        "team2_recent_runs_conceded":
            team2["last_5_runs_conceded"],

        "team2_recent_economy":
            team2["last_5_economy"],


        # -----------------------------
        # Target
        # -----------------------------

        "team1_win":
            match["team1_win"]
    })


player_ml_df = pd.DataFrame(rows)


# ------------------------------------------
# Sort chronologically
# ------------------------------------------

player_ml_df = (
    player_ml_df
    .sort_values(
        ["date", "match_id"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------
# Remove date/id from model features
# ------------------------------------------

player_feature_cols = [
    col
    for col in player_ml_df.columns
    if col not in [
        "match_id",
        "date",
        "team1_win"
    ]
]


X_player = player_ml_df[
    player_feature_cols
].copy()

y_player = player_ml_df[
    "team1_win"
].copy()


# ------------------------------------------
# Chronological split
# ------------------------------------------

split_index = int(
    len(player_ml_df) * 0.80
)


X_player_train = X_player.iloc[
    :split_index
].copy()

X_player_test = X_player.iloc[
    split_index:
].copy()


y_player_train = y_player.iloc[
    :split_index
].copy()

y_player_test = y_player.iloc[
    split_index:
].copy()


print()
print("======================================")
print("FINAL PLAYER ML DATASET")
print("======================================")

print(
    "Dataset:",
    player_ml_df.shape
)

print(
    "Features:",
    len(player_feature_cols)
)

print(
    "Training:",
    len(X_player_train)
)

print(
    "Testing:",
    len(X_player_test)
)

print()

print(
    "Training period:",
    player_ml_df.iloc[0]["date"],
    "to",
    player_ml_df.iloc[split_index - 1]["date"]
)

print(
    "Testing period:",
    player_ml_df.iloc[split_index]["date"],
    "to",
    player_ml_df.iloc[-1]["date"]
)

print()

print(
    "Target distribution:"
)

print(
    y_player.value_counts()
)


print()
print("Feature columns:")

for col in player_feature_cols:
    print("-", col)

BUILDING FINAL PLAYER ML DATASET

FINAL PLAYER ML DATASET
Dataset: (1243, 26)
Features: 23
Training: 994
Testing: 249

Training period: 2008-04-18 00:00:00 to 2023-05-02 00:00:00
Testing period: 2023-05-03 00:00:00 to 2026-05-31 00:00:00

Target distribution:
team1_win
0    635
1    608
Name: count, dtype: int64

Feature columns:
- season
- team1_batting_experience
- team1_batting_form
- team1_batting_strike_rate
- team1_bowling_experience
- team1_bowling_form
- team1_bowling_economy
- team1_recent_runs
- team1_recent_wickets
- team1_recent_strike_rate
- team1_recent_runs_conceded
- team1_recent_economy
- team2_batting_experience
- team2_batting_form
- team2_batting_strike_rate
- team2_bowling_experience
- team2_bowling_form
- team2_bowling_economy
- team2_recent_runs
- team2_recent_wickets
- team2_recent_strike_rate
- team2_recent_runs_conceded
- team2_recent_economy


In [120]:
# ==========================================
# STEP 65 - TRAIN PLAYER BASED MODEL
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("======================================")
print("TRAINING PLAYER-BASED MODEL")
print("======================================")


# ------------------------------------------
# Preprocessor
# ------------------------------------------

player_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------
# Model
# ------------------------------------------

player_model = Pipeline(
    steps=[
        (
            "preprocessor",
            player_preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                C=0.1
            )
        )
    ]
)


# ------------------------------------------
# Train
# ------------------------------------------

print("Training...")

player_model.fit(
    X_player_train,
    y_player_train
)

print("Training complete!")


# ------------------------------------------
# Predictions
# ------------------------------------------

player_pred = player_model.predict(
    X_player_test
)

player_prob = player_model.predict_proba(
    X_player_test
)[:, 1]


# ------------------------------------------
# Metrics
# ------------------------------------------

accuracy = accuracy_score(
    y_player_test,
    player_pred
)

precision = precision_score(
    y_player_test,
    player_pred,
    zero_division=0
)

recall = recall_score(
    y_player_test,
    player_pred,
    zero_division=0
)

f1 = f1_score(
    y_player_test,
    player_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_player_test,
    player_prob
)


# ------------------------------------------
# Results
# ------------------------------------------

print()
print("======================================")
print("PLAYER-BASED MODEL RESULTS")
print("======================================")

print(
    "Accuracy :",
    round(accuracy, 4)
)

print(
    "Precision:",
    round(precision, 4)
)

print(
    "Recall   :",
    round(recall, 4)
)

print(
    "F1 Score :",
    round(f1, 4)
)

print(
    "ROC-AUC  :",
    round(roc_auc, 4)
)


print()
print("======================================")
print("CLASSIFICATION REPORT")
print("======================================")

print(
    classification_report(
        y_player_test,
        player_pred,
        zero_division=0
    )
)


print()
print("======================================")
print("CONFUSION MATRIX")
print("======================================")

print(
    confusion_matrix(
        y_player_test,
        player_pred
    )
)

TRAINING PLAYER-BASED MODEL
Training...
Training complete!

PLAYER-BASED MODEL RESULTS
Accuracy : 0.5181
Precision: 0.4466
Recall   : 0.422
F1 Score : 0.434
ROC-AUC  : 0.4763

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.57      0.59      0.58       140
           1       0.45      0.42      0.43       109

    accuracy                           0.52       249
   macro avg       0.51      0.51      0.51       249
weighted avg       0.52      0.52      0.52       249


CONFUSION MATRIX
[[83 57]
 [63 46]]


In [121]:
# ==========================================
# STEP 66 - CALIBRATED CUSTOM TEAM PREDICTOR
# ==========================================

import numpy as np


def predict_custom_team(
    team1_players,
    team2_players
):

    # --------------------------------------
    # 1. Calculate team strengths
    # --------------------------------------

    team1 = build_custom_team(
        team1_players,
        "Team 1"
    )

    team2 = build_custom_team(
        team2_players,
        "Team 2"
    )


    # --------------------------------------
    # 2. Extract overall strengths
    # --------------------------------------

    team1_strength = float(
        team1["overall_strength"]
    )

    team2_strength = float(
        team2["overall_strength"]
    )


    # --------------------------------------
    # 3. Calculate matchup strength
    # --------------------------------------

    matchup = calculate_team_matchup(
        team1_players,
        team2_players
    )


    team1_matchup = float(
        matchup["team1_vs_team2_strength"]
    )

    team2_matchup = float(
        matchup["team2_vs_team1_strength"]
    )


    # --------------------------------------
    # 4. Normalize matchup
    # --------------------------------------

    total_matchup = (
        team1_matchup +
        team2_matchup
    )

    if total_matchup > 0:

        matchup_ratio = (
            team1_matchup /
            total_matchup
        )

    else:

        matchup_ratio = 0.5


    # --------------------------------------
    # 5. Normalize team strength
    # --------------------------------------

    total_strength = (
        team1_strength +
        team2_strength
    )

    if total_strength > 0:

        strength_ratio = (
            team1_strength /
            total_strength
        )

    else:

        strength_ratio = 0.5


    # --------------------------------------
    # 6. Combine strength + matchup
    #
    # 60% overall team strength
    # 40% matchup
    # --------------------------------------

    team1_score = (
        0.60 * strength_ratio +
        0.40 * matchup_ratio
    )


    # --------------------------------------
    # 7. Convert to probability
    # --------------------------------------

    team1_probability = (
        team1_score * 100
    )

    team2_probability = (
        100 -
        team1_probability
    )


    # --------------------------------------
    # 8. Keep probabilities reasonable
    # --------------------------------------

    team1_probability = np.clip(
        team1_probability,
        5,
        95
    )

    team2_probability = (
        100 -
        team1_probability
    )


    # --------------------------------------
    # 9. Winner
    # --------------------------------------

    if team1_probability >= team2_probability:

        winner = "Team 1"

    else:

        winner = "Team 2"


    # --------------------------------------
    # 10. Display
    # --------------------------------------

    print()
    print("======================================")
    print("VERSUS CUSTOM TEAM PREDICTION")
    print("======================================")

    print()

    print("TEAM 1")
    print("------")

    for player in team1_players:
        print("-", player)

    print()

    print(
        "Overall strength:",
        round(team1_strength, 2)
    )

    print(
        "Matchup strength:",
        round(team1_matchup, 2)
    )


    print()
    print("TEAM 2")
    print("------")

    for player in team2_players:
        print("-", player)

    print()

    print(
        "Overall strength:",
        round(team2_strength, 2)
    )

    print(
        "Matchup strength:",
        round(team2_matchup, 2)
    )


    print()
    print("======================================")

    print(
        "Team 1 win probability:",
        round(team1_probability, 2),
        "%"
    )

    print(
        "Team 2 win probability:",
        round(team2_probability, 2),
        "%"
    )

    print(
        "Predicted winner:",
        winner
    )

    print("======================================")


    return {
        "team1_probability":
            team1_probability,

        "team2_probability":
            team2_probability,

        "winner":
            winner,

        "team1_strength":
            team1_strength,

        "team2_strength":
            team2_strength,

        "team1_matchup":
            team1_matchup,

        "team2_matchup":
            team2_matchup
    }


# ==========================================
# TEST WITH CURRENT TEAMS
# ==========================================

custom_prediction = predict_custom_team(
    team1_players,
    team2_players
)


VERSUS CUSTOM TEAM PREDICTION

TEAM 1
------
- DA Warner
- S Dhawan
- Yuvraj Singh
- DJ Hooda
- B Kumar

Overall strength: 40.54
Matchup strength: 47.22

TEAM 2
------
- CH Gayle
- Mandeep Singh
- SR Watson
- TS Mills
- YS Chahal

Overall strength: 35.18
Matchup strength: 63.89

Team 1 win probability: 49.12 %
Team 2 win probability: 50.88 %
Predicted winner: Team 2


In [122]:
# ==========================================
# STEP 67 - CUSTOM TEAM DIFFERENCE FEATURES
# ==========================================

print("======================================")
print("CREATING CUSTOM TEAM DIFFERENCE FEATURES")
print("======================================")


# ------------------------------------------
# Base features
# ------------------------------------------

base_features = [
    "batting_experience",
    "batting_form",
    "batting_strike_rate",
    "bowling_experience",
    "bowling_form",
    "bowling_economy",
    "recent_runs",
    "recent_wickets",
    "recent_strike_rate",
    "recent_runs_conceded",
    "recent_economy"
]


# ------------------------------------------
# Create difference features
# ------------------------------------------

custom_diff_df = player_ml_df[
    ["match_id", "date", "season", "team1_win"]
].copy()


for feature in base_features:

    team1_col = f"team1_{feature}"
    team2_col = f"team2_{feature}"

    if (
        team1_col in player_ml_df.columns
        and
        team2_col in player_ml_df.columns
    ):

        # Difference
        custom_diff_df[
            f"{feature}_diff"
        ] = (
            player_ml_df[team1_col]
            -
            player_ml_df[team2_col]
        )

        # Absolute difference
        custom_diff_df[
            f"{feature}_abs_diff"
        ] = np.abs(
            player_ml_df[team1_col]
            -
            player_ml_df[team2_col]
        )


# ------------------------------------------
# Strength ratios
# ------------------------------------------

def safe_ratio(a, b):

    denominator = np.abs(b)

    if denominator == 0:

        return 0.0

    return a / denominator


for feature in [
    "batting_experience",
    "batting_form",
    "batting_strike_rate",
    "bowling_experience",
    "bowling_form",
    "recent_runs",
    "recent_wickets",
    "recent_strike_rate"
]:

    team1_col = f"team1_{feature}"
    team2_col = f"team2_{feature}"

    if (
        team1_col in player_ml_df.columns
        and
        team2_col in player_ml_df.columns
    ):

        custom_diff_df[
            f"{feature}_ratio"
        ] = (

            player_ml_df[team1_col]
            /
            (
                player_ml_df[team2_col]
                .abs()
                + 1e-6
            )
        )


# ------------------------------------------
# Remove invalid values
# ------------------------------------------

custom_diff_df = custom_diff_df.replace(
    [np.inf, -np.inf],
    np.nan
)


custom_diff_df = custom_diff_df.fillna(0)


# ------------------------------------------
# Feature matrix
# ------------------------------------------

custom_feature_cols = [
    col
    for col in custom_diff_df.columns
    if col not in [
        "match_id",
        "date",
        "team1_win"
    ]
]


X_custom = custom_diff_df[
    custom_feature_cols
].copy()

y_custom = custom_diff_df[
    "team1_win"
].copy()


# ------------------------------------------
# Chronological split
# ------------------------------------------

split_index = int(
    len(custom_diff_df) * 0.80
)


X_custom_train = X_custom.iloc[
    :split_index
].copy()

X_custom_test = X_custom.iloc[
    split_index:
].copy()


y_custom_train = y_custom.iloc[
    :split_index
].copy()

y_custom_test = y_custom.iloc[
    split_index:
].copy()


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("CUSTOM DIFFERENCE DATASET")
print("======================================")

print(
    "Shape:",
    custom_diff_df.shape
)

print(
    "Training:",
    X_custom_train.shape
)

print(
    "Testing:",
    X_custom_test.shape
)

print(
    "Number of features:",
    len(custom_feature_cols)
)

print()

print("Features:")

for col in custom_feature_cols:
    print("-", col)

CREATING CUSTOM TEAM DIFFERENCE FEATURES

CUSTOM DIFFERENCE DATASET
Shape: (1243, 34)
Training: (994, 31)
Testing: (249, 31)
Number of features: 31

Features:
- season
- batting_experience_diff
- batting_experience_abs_diff
- batting_form_diff
- batting_form_abs_diff
- batting_strike_rate_diff
- batting_strike_rate_abs_diff
- bowling_experience_diff
- bowling_experience_abs_diff
- bowling_form_diff
- bowling_form_abs_diff
- bowling_economy_diff
- bowling_economy_abs_diff
- recent_runs_diff
- recent_runs_abs_diff
- recent_wickets_diff
- recent_wickets_abs_diff
- recent_strike_rate_diff
- recent_strike_rate_abs_diff
- recent_runs_conceded_diff
- recent_runs_conceded_abs_diff
- recent_economy_diff
- recent_economy_abs_diff
- batting_experience_ratio
- batting_form_ratio
- batting_strike_rate_ratio
- bowling_experience_ratio
- bowling_form_ratio
- recent_runs_ratio
- recent_wickets_ratio
- recent_strike_rate_ratio


In [123]:
# ==========================================
# STEP 68 - COMPARE CUSTOM TEAM MODELS
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import xgboost as xgb


# ==========================================
# Preprocessor
# ==========================================

preprocessor_custom = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ==========================================
# MODELS
# ==========================================

models_custom = {

    "Logistic Regression":
        Pipeline(
            steps=[
                (
                    "preprocessor",
                    preprocessor_custom
                ),
                (
                    "classifier",
                    LogisticRegression(
                        max_iter=5000,
                        C=0.1
                    )
                )
            ]
        ),


    "Random Forest":
        Pipeline(
            steps=[
                (
                    "preprocessor",
                    preprocessor_custom
                ),
                (
                    "classifier",
                    RandomForestClassifier(
                        n_estimators=500,
                        max_depth=6,
                        min_samples_leaf=5,
                        random_state=42,
                        n_jobs=-1
                    )
                )
            ]
        ),


    "XGBoost":
        Pipeline(
            steps=[
                (
                    "preprocessor",
                    preprocessor_custom
                ),
                (
                    "classifier",
                    xgb.XGBClassifier(
                        n_estimators=300,
                        max_depth=3,
                        learning_rate=0.03,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        objective="binary:logistic",
                        eval_metric="logloss",
                        random_state=42
                    )
                )
            ]
        )
}


# ==========================================
# TRAIN + TEST
# ==========================================

results_custom = []


for name, model_custom in models_custom.items():

    print()
    print("Training:", name)

    model_custom.fit(
        X_custom_train,
        y_custom_train
    )

    pred = model_custom.predict(
        X_custom_test
    )

    prob = model_custom.predict_proba(
        X_custom_test
    )[:, 1]


    accuracy = accuracy_score(
        y_custom_test,
        pred
    )

    precision = precision_score(
        y_custom_test,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_custom_test,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_custom_test,
        pred,
        zero_division=0
    )

    auc = roc_auc_score(
        y_custom_test,
        prob
    )


    results_custom.append({

        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": auc
    })


# ==========================================
# RESULTS
# ==========================================

results_custom_df = pd.DataFrame(
    results_custom
).sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)


print()
print("======================================")
print("CUSTOM TEAM MODEL COMPARISON")
print("======================================")

print(
    results_custom_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ==========================================
# BEST MODEL
# ==========================================

best_custom_model_name = (
    results_custom_df.iloc[0]["Model"]
)

best_custom_model = models_custom[
    best_custom_model_name
]


print()
print("======================================")
print("BEST CUSTOM TEAM MODEL")
print("======================================")

print(
    "Best model:",
    best_custom_model_name
)

print(
    "Accuracy:",
    round(
        results_custom_df.iloc[0]["Accuracy"],
        4
    )
)

print(
    "ROC-AUC:",
    round(
        results_custom_df.iloc[0]["ROC-AUC"],
        4
    )
)


Training: Logistic Regression

Training: Random Forest

Training: XGBoost

CUSTOM TEAM MODEL COMPARISON
              Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression    0.5301     0.4375  0.2569 0.3237   0.5375
            XGBoost    0.5020     0.4400  0.5046 0.4701   0.5074
      Random Forest    0.5060     0.4271  0.3761 0.4000   0.5014

BEST CUSTOM TEAM MODEL
Best model: Logistic Regression
Accuracy: 0.5301
ROC-AUC: 0.5375


In [124]:
# ==========================================
# STEP 69 - PLAYER IMPACT RATING
# ==========================================

import numpy as np
import pandas as pd

print("======================================")
print("CREATING PLAYER IMPACT RATINGS")
print("======================================")


# ------------------------------------------
# Start from player-level historical data
# ------------------------------------------

rating_df = player_df.copy()

rating_df["date"] = pd.to_datetime(
    rating_df["date"]
)


# ------------------------------------------
# Numeric conversion
# ------------------------------------------

rating_columns = [
    "runs",
    "balls_faced",
    "wickets",
    "runs_conceded",
    "balls_bowled",
    "previous_runs",
    "previous_wickets",
    "previous_strike_rate",
    "previous_economy",
    "last_5_runs",
    "last_5_wickets",
    "last_5_strike_rate",
    "last_5_economy"
]


for col in rating_columns:

    if col in rating_df.columns:

        rating_df[col] = pd.to_numeric(
            rating_df[col],
            errors="coerce"
        ).fillna(0)


# ------------------------------------------
# Batting impact
#
# Recent batting is more important than
# career batting.
# ------------------------------------------

rating_df["batting_impact"] = (

    0.40 *
    rating_df["batting_form"]

    +

    0.30 *
    rating_df["recent_strike_rate"]

    +

    0.20 *
    rating_df["batting_strike_rate"]

    +

    0.10 *
    np.log1p(
        rating_df["batting_experience"]
    )
)


# ------------------------------------------
# Bowling impact
# ------------------------------------------

rating_df["bowling_impact"] = (

    0.40 *
    rating_df["bowling_form"]

    +

    0.30 *
    rating_df["recent_bowling_economy"]

    +

    0.20 *
    rating_df["bowling_economy"].clip(
        upper=12
    )

    +

    0.10 *
    np.log1p(
        rating_df["bowling_experience"]
    )
)


# ------------------------------------------
# Normalize components
# ------------------------------------------

def normalize_series(series):

    series = pd.to_numeric(
        series,
        errors="coerce"
    ).fillna(0)

    low = series.quantile(0.05)
    high = series.quantile(0.95)

    if high == low:

        return pd.Series(
            50.0,
            index=series.index
        )

    normalized = (
        (series - low)
        /
        (high - low)
    ) * 100

    return normalized.clip(0, 100)


rating_df["batting_rating"] = (
    normalize_series(
        rating_df["batting_impact"]
    )
)


# Economy is LOWER = BETTER
bowling_economy_rating = (
    100 -
    normalize_series(
        rating_df["recent_bowling_economy"]
    )
)

rating_df["bowling_rating"] = (
    bowling_economy_rating
)


# ------------------------------------------
# Overall player impact
# ------------------------------------------

rating_df["player_impact"] = (

    0.55 *
    rating_df["batting_rating"]

    +

    0.45 *
    rating_df["bowling_rating"]
)


# ------------------------------------------
# Keep latest pre-match rating per player
# ------------------------------------------

player_rating_df = rating_df[
    [
        "match_id",
        "date",
        "player",
        "batting_rating",
        "bowling_rating",
        "player_impact"
    ]
].copy()


player_rating_df = (
    player_rating_df
    .sort_values(
        ["player", "date", "match_id"]
    )
)


# ------------------------------------------
# Display
# ------------------------------------------

print()
print("======================================")
print("PLAYER IMPACT RATING CREATED")
print("======================================")

print(
    "Players:",
    player_rating_df["player"].nunique()
)

print(
    "Rows:",
    len(player_rating_df)
)

print()

display(
    player_rating_df.head(20)
)

CREATING PLAYER IMPACT RATINGS

PLAYER IMPACT RATING CREATED
Players: 811
Rows: 27909



,match_id,date,player,batting_rating,bowling_rating,player_impact
6160,548341,2012-04-26,A Ashish Reddy,0.000000,100.000000,45.000000
6270,548346,2012-04-29,A Ashish Reddy,0.000000,28.269866,12.721440
6314,548348,2012-05-01,A Ashish Reddy,52.303159,37.478464,45.632047
6402,548352,2012-05-04,A Ashish Reddy,51.635684,26.639635,40.387462
6490,548356,2012-05-06,A Ashish Reddy,51.626456,26.931617,40.513778
6556,548359,2012-05-08,A Ashish Reddy,51.366140,24.919832,39.465302
6600,548329,2012-05-10,A Ashish Reddy,52.052170,20.781555,37.980393
6886,548373,2012-05-18,A Ashish Reddy,51.251200,9.141830,32.301983
6952,548376,2012-05-20,A Ashish Reddy,68.910064,12.258675,43.416939
7128,598000,2013-04-05,A Ashish Reddy,67.588172,18.182816,45.355762


In [125]:
# ==========================================
# STEP 70 - TRUE PRE-MATCH PLAYER RATINGS
# ==========================================

print("======================================")
print("CREATING TRUE PRE-MATCH RATINGS")
print("======================================")


prematch_rating_df = (
    player_rating_df
    .sort_values(
        ["player", "date", "match_id"]
    )
    .copy()
)


# ------------------------------------------
# Shift rating by one match
# ------------------------------------------

rating_columns = [
    "batting_rating",
    "bowling_rating",
    "player_impact"
]


for col in rating_columns:

    prematch_rating_df[
        f"previous_{col}"
    ] = (
        prematch_rating_df
        .groupby("player")[col]
        .shift(1)
    )


# ------------------------------------------
# Missing = player had no previous IPL data
# ------------------------------------------

for col in rating_columns:

    prematch_rating_df[
        f"previous_{col}"
    ] = (
        prematch_rating_df[
            f"previous_{col}"
        ]
        .fillna(50)
    )


# ------------------------------------------
# Keep required columns
# ------------------------------------------

prematch_rating_df = prematch_rating_df[
    [
        "match_id",
        "date",
        "player",
        "previous_batting_rating",
        "previous_bowling_rating",
        "previous_player_impact"
    ]
].copy()


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("TRUE PRE-MATCH RATINGS")
print("======================================")

print(
    "Rows:",
    len(prematch_rating_df)
)

print(
    "Players:",
    prematch_rating_df["player"].nunique()
)

print()

display(
    prematch_rating_df.head(20)
)

CREATING TRUE PRE-MATCH RATINGS

TRUE PRE-MATCH RATINGS
Rows: 27909
Players: 811



,match_id,date,player,previous_batting_rating,previous_bowling_rating,previous_player_impact
6160,548341,2012-04-26,A Ashish Reddy,50.000000,50.000000,50.000000
6270,548346,2012-04-29,A Ashish Reddy,0.000000,100.000000,45.000000
6314,548348,2012-05-01,A Ashish Reddy,0.000000,28.269866,12.721440
6402,548352,2012-05-04,A Ashish Reddy,52.303159,37.478464,45.632047
6490,548356,2012-05-06,A Ashish Reddy,51.635684,26.639635,40.387462
6556,548359,2012-05-08,A Ashish Reddy,51.626456,26.931617,40.513778
6600,548329,2012-05-10,A Ashish Reddy,51.366140,24.919832,39.465302
6886,548373,2012-05-18,A Ashish Reddy,52.052170,20.781555,37.980393
6952,548376,2012-05-20,A Ashish Reddy,51.251200,9.141830,32.301983
7128,598000,2013-04-05,A Ashish Reddy,68.910064,12.258675,43.416939


In [126]:
# ==========================================
# STEP 71 - BUILD PRE-MATCH TEAM RATINGS
# ==========================================

print("======================================")
print("BUILDING PRE-MATCH TEAM RATINGS")
print("======================================")


# ------------------------------------------
# Create quick lookup
# ------------------------------------------

rating_lookup = (
    prematch_rating_df
    .set_index(["match_id", "player"])
)


# ------------------------------------------
# Function to calculate team ratings
# ------------------------------------------

def get_team_ratings(match_id, players):

    ratings = []

    for player in players:

        key = (match_id, player)

        if key in rating_lookup.index:

            row = rating_lookup.loc[key]

            ratings.append({
                "batting":
                    float(
                        row["previous_batting_rating"]
                    ),

                "bowling":
                    float(
                        row["previous_bowling_rating"]
                    ),

                "impact":
                    float(
                        row["previous_player_impact"]
                    )
            })


    # No data
    if len(ratings) == 0:

        return {
            "batting": 50.0,
            "bowling": 50.0,
            "impact": 50.0
        }


    ratings_df = pd.DataFrame(
        ratings
    )


    return {
        "batting":
            ratings_df["batting"].mean(),

        "bowling":
            ratings_df["bowling"].mean(),

        "impact":
            ratings_df["impact"].mean()
    }


# ------------------------------------------
# Build historical team dataset
# ------------------------------------------

team_rating_rows = []


for _, match in custom_match_df.iterrows():

    t1 = get_team_ratings(
        match["match_id"],
        match["team1_players"]
    )

    t2 = get_team_ratings(
        match["match_id"],
        match["team2_players"]
    )


    team_rating_rows.append({

        "match_id":
            match["match_id"],

        "date":
            match["date"],

        "team1_batting_rating":
            t1["batting"],

        "team1_bowling_rating":
            t1["bowling"],

        "team1_impact":
            t1["impact"],


        "team2_batting_rating":
            t2["batting"],

        "team2_bowling_rating":
            t2["bowling"],

        "team2_impact":
            t2["impact"],


        "team1_win":
            match["team1_win"]
    })


team_rating_df = pd.DataFrame(
    team_rating_rows
)


# ------------------------------------------
# Difference features
# ------------------------------------------

team_rating_df[
    "batting_difference"
] = (
    team_rating_df[
        "team1_batting_rating"
    ]
    -
    team_rating_df[
        "team2_batting_rating"
    ]
)


team_rating_df[
    "bowling_difference"
] = (
    team_rating_df[
        "team1_bowling_rating"
    ]
    -
    team_rating_df[
        "team2_bowling_rating"
    ]
)


team_rating_df[
    "impact_difference"
] = (
    team_rating_df[
        "team1_impact"
    ]
    -
    team_rating_df[
        "team2_impact"
    ]
)


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("PRE-MATCH TEAM RATINGS")
print("======================================")

print(
    "Shape:",
    team_rating_df.shape
)

print()

display(
    team_rating_df.head(10)
)

BUILDING PRE-MATCH TEAM RATINGS

PRE-MATCH TEAM RATINGS
Shape: (1243, 12)



,match_id,date,team1_batting_rating,team1_bowling_rating,team1_impact,team2_batting_rating,team2_bowling_rating,team2_impact,team1_win,batting_difference,bowling_difference,impact_difference
0,1082591,2017-04-05,70.363931,51.633407,61.935195,54.266916,57.879128,55.892412,1,16.097015,-6.245721,6.042784
1,1082592,2017-04-06,47.595270,65.807359,55.790710,66.289836,57.911478,62.519575,1,-18.694566,7.895882,-6.728865
2,1082593,2017-04-07,54.666626,54.969340,54.802847,48.099474,65.942016,56.128618,0,6.567151,-10.972676,-1.325771
3,1082594,2017-04-08,65.944334,59.363417,62.982921,55.824446,69.243900,61.863200,1,10.119888,-9.880483,1.119721
4,1082595,2017-04-08,56.469446,47.796677,52.566700,54.827121,61.799078,57.964502,1,1.642325,-14.002401,-5.397801
5,1082596,2017-04-09,67.109211,55.745359,61.995478,51.919271,57.104692,54.252710,1,15.189941,-1.359333,7.742768
6,1082597,2017-04-09,64.230972,51.851869,58.660375,43.851250,68.511226,54.948239,1,20.379722,-16.659358,3.712136
7,1082598,2017-04-10,58.499781,54.700590,56.790145,58.331739,58.328418,58.330245,1,0.168042,-3.627828,-1.540099
8,1082599,2017-04-11,50.694002,56.341005,53.235153,53.712820,60.356568,56.702507,0,-3.018819,-4.015563,-3.467354
9,1082600,2017-04-12,66.872826,51.287110,59.859254,51.849463,61.479887,56.183154,1,15.023363,-10.192777,3.676100


In [127]:
# ==========================================
# STEP 72 - TEST PLAYER IMPACT MODEL
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("======================================")
print("TESTING PLAYER IMPACT MODEL")
print("======================================")


# ------------------------------------------
# Features
# ------------------------------------------

impact_features = [
    "batting_difference",
    "bowling_difference",
    "impact_difference"
]


X_impact = team_rating_df[
    impact_features
].copy()

y_impact = team_rating_df[
    "team1_win"
].copy()


# ------------------------------------------
# Chronological split
# ------------------------------------------

split_index = int(
    len(team_rating_df) * 0.80
)


X_train_impact = X_impact.iloc[
    :split_index
]

X_test_impact = X_impact.iloc[
    split_index:
]

y_train_impact = y_impact.iloc[
    :split_index
]

y_test_impact = y_impact.iloc[
    split_index:
]


# ------------------------------------------
# Logistic Regression
# ------------------------------------------

impact_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                C=0.1
            )
        )
    ]
)


# ------------------------------------------
# Train
# ------------------------------------------

impact_model.fit(
    X_train_impact,
    y_train_impact
)


# ------------------------------------------
# Predict
# ------------------------------------------

impact_pred = impact_model.predict(
    X_test_impact
)

impact_prob = impact_model.predict_proba(
    X_test_impact
)[:, 1]


# ------------------------------------------
# Metrics
# ------------------------------------------

accuracy = accuracy_score(
    y_test_impact,
    impact_pred
)

precision = precision_score(
    y_test_impact,
    impact_pred,
    zero_division=0
)

recall = recall_score(
    y_test_impact,
    impact_pred,
    zero_division=0
)

f1 = f1_score(
    y_test_impact,
    impact_pred,
    zero_division=0
)

auc = roc_auc_score(
    y_test_impact,
    impact_prob
)


# ------------------------------------------
# Results
# ------------------------------------------

print()
print("======================================")
print("PLAYER IMPACT MODEL RESULTS")
print("======================================")

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

print(
    f"ROC-AUC  : {auc:.4f}"
)


print()
print("Test matches:", len(y_test_impact))

TESTING PLAYER IMPACT MODEL

PLAYER IMPACT MODEL RESULTS
Accuracy : 0.4538
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000
ROC-AUC  : 0.5169

Test matches: 249


In [128]:
# ==========================================
# STEP 73 - PLAYER COMPOSITION FEATURES
# ==========================================

print("======================================")
print("CREATING PLAYER COMPOSITION FEATURES")
print("======================================")


# ------------------------------------------
# Helper function
# ------------------------------------------

def calculate_team_composition(players):

    rows = []

    for player in players:

        player_rows = prematch_rating_df[
            prematch_rating_df["player"] == player
        ]

        if len(player_rows) == 0:
            continue

        # Use the latest available rating
        row = player_rows.sort_values(
            ["date", "match_id"]
        ).iloc[-1]

        rows.append({
            "batting":
                float(row["previous_batting_rating"]),

            "bowling":
                float(row["previous_bowling_rating"]),

            "impact":
                float(row["previous_player_impact"])
        })


    # No player data
    if len(rows) == 0:

        return {
            "batting_mean": 50,
            "batting_max": 50,
            "batting_median": 50,
            "bowling_mean": 50,
            "bowling_max": 50,
            "bowling_median": 50,
            "impact_mean": 50,
            "impact_max": 50,
            "impact_median": 50,
            "impact_std": 0,
            "balance": 0
        }


    df = pd.DataFrame(rows)


    # --------------------------------------
    # Composition statistics
    # --------------------------------------

    batting_mean = df["batting"].mean()
    bowling_mean = df["bowling"].mean()

    return {

        "batting_mean":
            batting_mean,

        "batting_max":
            df["batting"].max(),

        "batting_median":
            df["batting"].median(),

        "bowling_mean":
            bowling_mean,

        "bowling_max":
            df["bowling"].max(),

        "bowling_median":
            df["bowling"].median(),

        "impact_mean":
            df["impact"].mean(),

        "impact_max":
            df["impact"].max(),

        "impact_median":
            df["impact"].median(),

        "impact_std":
            df["impact"].std()
            if len(df) > 1 else 0,

        "balance":
            batting_mean - bowling_mean
    }


# ------------------------------------------
# Build dataset
# ------------------------------------------

composition_rows = []


for _, row in custom_match_df.iterrows():

    team1 = calculate_team_composition(
        row["team1_players"]
    )

    team2 = calculate_team_composition(
        row["team2_players"]
    )


    result = {

        "match_id":
            row["match_id"],

        "date":
            row["date"],

        "team1_win":
            row["team1_win"]
    }


    # --------------------------------------
    # Team 1 / Team 2 features
    # --------------------------------------

    for key, value in team1.items():

        result[
            "team1_" + key
        ] = value


    for key, value in team2.items():

        result[
            "team2_" + key
        ] = value


    # --------------------------------------
    # Difference features
    # --------------------------------------

    for key in team1.keys():

        result[
            key + "_difference"
        ] = (
            team1[key]
            -
            team2[key]
        )


    composition_rows.append(result)


composition_df = pd.DataFrame(
    composition_rows
)


# ------------------------------------------
# Clean
# ------------------------------------------

composition_df = composition_df.replace(
    [np.inf, -np.inf],
    np.nan
)

composition_df = composition_df.fillna(0)


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("PLAYER COMPOSITION DATASET")
print("======================================")

print(
    "Shape:",
    composition_df.shape
)

print(
    "Features:",
    len(
        [
            c for c in composition_df.columns
            if c not in [
                "match_id",
                "date",
                "team1_win"
            ]
        ]
    )
)

print()

display(
    composition_df.head(10)
)

CREATING PLAYER COMPOSITION FEATURES

PLAYER COMPOSITION DATASET
Shape: (1243, 36)
Features: 33



,match_id,date,team1_win,team1_batting_mean,team1_batting_max,team1_batting_median,team1_bowling_mean,team1_bowling_max,team1_bowling_median,team1_impact_mean,...,batting_max_difference,batting_median_difference,bowling_mean_difference,bowling_max_difference,bowling_median_difference,impact_mean_difference,impact_max_difference,impact_median_difference,impact_std_difference,balance_difference
0,1082591,2017-04-05,1,68.347876,100.000000,74.351409,56.340363,100.0,34.674342,62.944495,...,0.000000,8.104509,-7.556860,0.0,-65.325658,-0.764326,-9.920566,-11.307911,-5.917379,12.350062
1,1082592,2017-04-06,1,60.986122,90.802560,65.230078,48.415752,100.0,32.254873,55.329456,...,-5.711140,-6.264309,0.189193,0.0,9.647913,-4.930524,-13.837340,-3.201514,-2.546084,-9.308578
2,1082593,2017-04-07,0,58.856168,96.396321,69.881535,52.918331,100.0,22.665949,56.184142,...,10.925400,15.624064,-2.702148,0.0,-19.600040,-1.347099,6.008970,-3.155886,5.900098,2.463726
3,1082594,2017-04-08,1,61.560925,98.840751,67.816343,45.003325,100.0,17.673369,54.110005,...,8.038191,2.586265,-5.980508,0.0,-14.581505,-2.751596,10.944408,0.659088,6.808701,5.870750
4,1082595,2017-04-08,1,59.590348,100.000000,61.857120,58.906276,100.0,42.615893,59.282515,...,14.355943,-5.996496,-0.929841,0.0,7.301103,-1.164684,0.096453,8.665611,1.421794,-0.426986
5,1082596,2017-04-09,1,68.347876,100.000000,74.351409,56.340363,100.0,34.674342,62.944495,...,3.603679,4.469874,0.733339,0.0,-15.325658,4.444886,-7.938542,9.824476,-9.475564,6.748266
6,1082597,2017-04-09,1,66.528224,96.513700,71.494387,39.942708,100.0,19.775508,54.564741,...,11.042778,18.218546,-13.153179,0.0,-22.490482,0.075923,5.896296,4.221231,-1.168840,24.052912
7,1082598,2017-04-10,1,56.949959,98.840751,66.945515,43.396655,100.0,16.314843,50.850972,...,-1.159249,5.088394,-15.509620,0.0,-26.301049,-8.334675,3.356107,-14.492803,5.004602,13.045356
8,1082599,2017-04-11,0,62.192925,90.802560,67.245161,56.566904,100.0,33.016713,59.661216,...,5.813253,4.670491,-2.274109,0.0,-2.298077,1.381192,-3.720609,4.340400,-2.333066,6.646001
9,1082600,2017-04-12,1,66.528224,96.513700,71.494387,39.942708,100.0,19.775508,54.564741,...,-3.486300,-2.857022,-20.836749,0.0,-80.224492,-7.597861,7.825868,-20.616430,0.374681,24.070705


In [129]:
# ==========================================
# STEP 74 - LEAKAGE-FREE COMPOSITION FEATURES
# ==========================================

print("======================================")
print("BUILDING LEAKAGE-FREE COMPOSITION FEATURES")
print("======================================")

# Make sure dates are datetime
prematch_rating_df["date"] = pd.to_datetime(
    prematch_rating_df["date"]
)

custom_match_df["date"] = pd.to_datetime(
    custom_match_df["date"]
)


# ------------------------------------------
# Sort ratings chronologically
# ------------------------------------------

ratings_sorted = (
    prematch_rating_df
    .sort_values(["player", "date", "match_id"])
    .copy()
)


# ------------------------------------------
# Function:
# Get latest rating BEFORE match date
# ------------------------------------------

def get_player_rating_before(
    player,
    match_date
):

    rows = ratings_sorted[
        (ratings_sorted["player"] == player)
        &
        (ratings_sorted["date"] < match_date)
    ]

    if len(rows) == 0:

        return {
            "batting": 50.0,
            "bowling": 50.0,
            "impact": 50.0
        }

    row = rows.iloc[-1]

    return {
        "batting":
            float(row["previous_batting_rating"]),

        "bowling":
            float(row["previous_bowling_rating"]),

        "impact":
            float(row["previous_player_impact"])
    }


# ------------------------------------------
# Calculate team composition
# ------------------------------------------

def calculate_leakage_free_team(
    players,
    match_date
):

    ratings = []

    for player in players:

        rating = get_player_rating_before(
            player,
            match_date
        )

        ratings.append(rating)


    df = pd.DataFrame(ratings)


    return {

        "batting_mean":
            df["batting"].mean(),

        "batting_max":
            df["batting"].max(),

        "batting_median":
            df["batting"].median(),

        "bowling_mean":
            df["bowling"].mean(),

        "bowling_max":
            df["bowling"].max(),

        "bowling_median":
            df["bowling"].median(),

        "impact_mean":
            df["impact"].mean(),

        "impact_max":
            df["impact"].max(),

        "impact_median":
            df["impact"].median(),

        "impact_std":
            df["impact"].std()
            if len(df) > 1 else 0,

        "balance":
            df["batting"].mean()
            -
            df["bowling"].mean()
    }


# ------------------------------------------
# Build dataset
# ------------------------------------------

composition_rows = []


for _, match in custom_match_df.iterrows():

    match_date = match["date"]


    team1 = calculate_leakage_free_team(
        match["team1_players"],
        match_date
    )


    team2 = calculate_leakage_free_team(
        match["team2_players"],
        match_date
    )


    result = {

        "match_id":
            match["match_id"],

        "date":
            match_date,

        "team1_win":
            match["team1_win"]
    }


    # Team 1
    for key, value in team1.items():

        result[
            "team1_" + key
        ] = value


    # Team 2
    for key, value in team2.items():

        result[
            "team2_" + key
        ] = value


    # Differences
    for key in team1.keys():

        result[
            key + "_difference"
        ] = (
            team1[key]
            -
            team2[key]
        )


    composition_rows.append(result)


composition_df = pd.DataFrame(
    composition_rows
)


# ------------------------------------------
# Clean
# ------------------------------------------

composition_df = composition_df.replace(
    [np.inf, -np.inf],
    np.nan
)

composition_df = composition_df.fillna(0)


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("LEAKAGE-FREE COMPOSITION DATASET")
print("======================================")

print(
    "Shape:",
    composition_df.shape
)

print(
    "Matches:",
    len(composition_df)
)

print(
    "Features:",
    len([
        c for c in composition_df.columns
        if c not in [
            "match_id",
            "date",
            "team1_win"
        ]
    ])
)

print()

display(
    composition_df.head(10)
)

BUILDING LEAKAGE-FREE COMPOSITION FEATURES

LEAKAGE-FREE COMPOSITION DATASET
Shape: (1243, 36)
Matches: 1243
Features: 33



,match_id,date,team1_win,team1_batting_mean,team1_batting_max,team1_batting_median,team1_bowling_mean,team1_bowling_max,team1_bowling_median,team1_impact_mean,...,batting_max_difference,batting_median_difference,bowling_mean_difference,bowling_max_difference,bowling_median_difference,impact_mean_difference,impact_max_difference,impact_median_difference,impact_std_difference,balance_difference
0,1082591,2017-04-05,1,66.006324,84.705993,72.777848,51.262698,100.0,38.517028,59.371693,...,-7.652999,1.066121,-6.871157,0.0,-11.482972,-2.224698,-4.209150,-2.441710,-5.106124,8.448106
1,1082592,2017-04-06,1,44.325801,75.555811,62.559730,70.029822,100.0,100.000000,55.892610,...,-19.133953,-8.901383,11.506931,0.0,69.168344,-8.478061,-3.147094,-5.381513,8.048586,-36.336350
2,1082593,2017-04-07,0,51.082571,84.689603,50.000000,50.334422,100.0,36.239881,50.745904,...,1.917287,-14.200971,-14.065853,0.0,-13.760119,-7.290072,-1.348679,-16.315854,-2.563555,12.319602
3,1082594,2017-04-08,1,70.878504,100.000000,79.707404,56.506864,100.0,50.000000,64.411266,...,9.131933,13.431259,-8.479159,0.0,0.000000,2.847793,-5.045047,-4.129155,-4.656913,20.594459
4,1082595,2017-04-08,1,54.838880,87.724455,50.000000,47.906365,100.0,50.000000,51.719249,...,-12.275545,-17.251297,-15.696069,0.0,8.921182,-7.294670,-10.618309,-11.092807,-7.718392,15.275270
5,1082596,2017-04-09,1,70.363931,98.487320,72.876360,51.633407,100.0,38.975856,61.935195,...,10.674766,2.212049,-9.129865,0.0,-11.024144,2.637680,5.278611,1.381574,-8.686847,21.395536
6,1082597,2017-04-09,1,62.597083,100.000000,65.462348,53.140909,100.0,37.684446,58.341805,...,12.556251,7.163191,-11.866284,0.0,-17.484220,2.986725,7.317297,-20.371770,-8.574349,27.005470
7,1082598,2017-04-10,1,67.716211,89.710758,76.086871,51.831753,100.0,30.760495,60.568205,...,-10.289242,10.034805,-2.078288,0.0,-19.239505,5.250491,-8.063188,5.690002,0.774923,13.325052
8,1082599,2017-04-11,0,43.375750,86.062744,50.000000,65.476880,100.0,50.000000,53.321258,...,-7.033532,-17.134903,4.318249,0.0,9.775112,-4.009856,-2.093454,-14.648328,-0.858003,-15.142009
9,1082600,2017-04-12,1,64.230972,100.000000,71.062106,51.851869,100.0,35.820406,58.660375,...,2.361296,10.369442,-7.619888,0.0,-3.657293,0.999712,-8.343548,-1.485631,-0.388758,15.671999


In [130]:
# ==========================================
# STEP 75 - COMBINE CUSTOM TEAM FEATURES
# ==========================================

print("======================================")
print("COMBINING CUSTOM TEAM FEATURES")
print("======================================")


# ------------------------------------------
# Remove duplicate metadata/target columns
# ------------------------------------------

composition_features = composition_df.drop(
    columns=[
        "match_id",
        "date",
        "team1_win"
    ],
    errors="ignore"
).copy()


original_features = custom_diff_df.drop(
    columns=[
        "match_id",
        "date",
        "team1_win"
    ],
    errors="ignore"
).copy()


# ------------------------------------------
# Rename duplicate columns if necessary
# ------------------------------------------

duplicate_columns = (
    set(original_features.columns)
    &
    set(composition_features.columns)
)

print(
    "Duplicate feature names:",
    len(duplicate_columns)
)


# Remove duplicates from composition
composition_features = composition_features.drop(
    columns=list(duplicate_columns),
    errors="ignore"
)


# ------------------------------------------
# Combine
# ------------------------------------------

combined_custom_df = pd.concat(
    [
        original_features.reset_index(drop=True),
        composition_features.reset_index(drop=True)
    ],
    axis=1
)


# ------------------------------------------
# Target
# ------------------------------------------

y_combined = custom_diff_df[
    "team1_win"
].reset_index(drop=True)


# ------------------------------------------
# Clean
# ------------------------------------------

combined_custom_df = combined_custom_df.replace(
    [np.inf, -np.inf],
    np.nan
)

combined_custom_df = combined_custom_df.fillna(0)


# ------------------------------------------
# Chronological split
# ------------------------------------------

split_index = int(
    len(combined_custom_df) * 0.80
)


X_combined_train = combined_custom_df.iloc[
    :split_index
].copy()

X_combined_test = combined_custom_df.iloc[
    split_index:
].copy()


y_combined_train = y_combined.iloc[
    :split_index
].copy()

y_combined_test = y_combined.iloc[
    split_index:
].copy()


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("COMBINED CUSTOM TEAM DATASET")
print("======================================")

print(
    "Total features:",
    combined_custom_df.shape[1]
)

print(
    "Training rows:",
    len(X_combined_train)
)

print(
    "Testing rows:",
    len(X_combined_test)
)

print(
    "Training features:",
    X_combined_train.shape
)

print(
    "Testing features:",
    X_combined_test.shape
)

COMBINING CUSTOM TEAM FEATURES
Duplicate feature names: 0

COMBINED CUSTOM TEAM DATASET
Total features: 64
Training rows: 994
Testing rows: 249
Training features: (994, 64)
Testing features: (249, 64)


In [131]:
# ==========================================
# STEP 76 - FINAL LEAKAGE AUDIT
# ==========================================

print("======================================")
print("FINAL CUSTOM MODEL LEAKAGE AUDIT")
print("======================================")


# ------------------------------------------
# Check NaN / infinite values
# ------------------------------------------

nan_count = (
    combined_custom_df.isna()
    .sum()
    .sum()
)

inf_count = np.isinf(
    combined_custom_df.select_dtypes(
        include=np.number
    )
).sum().sum()


print(
    "NaN values:",
    nan_count
)

print(
    "Infinite values:",
    inf_count
)


# ------------------------------------------
# Check target is NOT inside X
# ------------------------------------------

print(
    "Target included:",
    "team1_win" in combined_custom_df.columns
)


# ------------------------------------------
# Check feature uniqueness
# ------------------------------------------

print(
    "Total features:",
    combined_custom_df.shape[1]
)

print(
    "Duplicate column names:",
    combined_custom_df.columns.duplicated().sum()
)


# ------------------------------------------
# Check constant features
# ------------------------------------------

constant_features = [
    col
    for col in combined_custom_df.columns
    if combined_custom_df[col].nunique() <= 1
]


print(
    "Constant features:",
    len(constant_features)
)

if constant_features:

    print(
        constant_features
    )


# ------------------------------------------
# Check correlation with target
# ------------------------------------------

audit_df = combined_custom_df.copy()

audit_df["team1_win"] = y_combined.values


numeric_audit = audit_df.select_dtypes(
    include=np.number
)


correlations = (
    numeric_audit
    .corr()["team1_win"]
    .drop("team1_win")
    .abs()
    .sort_values(
        ascending=False
    )
)


print()
print("Top 15 feature correlations:")

print(
    correlations.head(15)
)


# ------------------------------------------
# Final summary
# ------------------------------------------

print()
print("======================================")
print("LEAKAGE AUDIT SUMMARY")
print("======================================")

if (
    nan_count == 0
    and
    inf_count == 0
    and
    "team1_win" not in combined_custom_df.columns
):

    print(
        "✓ Basic dataset checks passed"
    )

else:

    print(
        "⚠ Dataset requires cleaning"
    )

FINAL CUSTOM MODEL LEAKAGE AUDIT
NaN values: 0
Infinite values: 0
Target included: False
Total features: 64
Duplicate column names: 0
Constant features: 1
['bowling_max_difference']

Top 15 feature correlations:
season                       0.074751
batting_experience_ratio     0.073745
bowling_experience_diff      0.070099
team2_impact_std             0.066053
batting_experience_diff      0.065360
team1_batting_max            0.062141
bowling_experience_ratio     0.057182
bowling_economy_abs_diff     0.056361
recent_runs_abs_diff         0.051739
batting_form_abs_diff        0.051739
recent_wickets_diff          0.051405
bowling_form_diff            0.051405
batting_strike_rate_diff     0.050155
batting_strike_rate_ratio    0.049133
recent_runs_diff             0.038563
Name: team1_win, dtype: float64

LEAKAGE AUDIT SUMMARY
✓ Basic dataset checks passed


In [132]:
# ==========================================
# STEP 77 - TRAIN COMBINED CUSTOM MODELS
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import xgboost as xgb


print("======================================")
print("TRAINING COMBINED CUSTOM MODELS")
print("======================================")


# ------------------------------------------
# Remove constant features
# ------------------------------------------

constant_features = [
    col
    for col in X_combined_train.columns
    if X_combined_train[col].nunique() <= 1
]


X_combined_train_clean = (
    X_combined_train.drop(
        columns=constant_features
    )
)

X_combined_test_clean = (
    X_combined_test.drop(
        columns=constant_features
    )
)


print(
    "Removed constant features:",
    constant_features
)

print(
    "Final feature count:",
    X_combined_train_clean.shape[1]
)


# ==========================================
# Models
# ==========================================

combined_models = {

    "Logistic Regression":

        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    C=0.1,
                    max_iter=5000
                )
            )
        ]),


    "Random Forest":

        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=5,
                    min_samples_leaf=8,
                    max_features="sqrt",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]),


    "XGBoost":

        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "classifier",
                xgb.XGBClassifier(
                    n_estimators=250,
                    max_depth=2,
                    learning_rate=0.03,
                    min_child_weight=8,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_alpha=0.5,
                    reg_lambda=2.0,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=42
                )
            )
        ])
}


# ==========================================
# Train + evaluate
# ==========================================

combined_results = []


for name, model in combined_models.items():

    print()
    print("Training:", name)

    model.fit(
        X_combined_train_clean,
        y_combined_train
    )


    predictions = model.predict(
        X_combined_test_clean
    )


    probabilities = model.predict_proba(
        X_combined_test_clean
    )[:, 1]


    accuracy = accuracy_score(
        y_combined_test,
        predictions
    )

    precision = precision_score(
        y_combined_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_combined_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_combined_test,
        predictions,
        zero_division=0
    )

    auc = roc_auc_score(
        y_combined_test,
        probabilities
    )


    combined_results.append({

        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": auc
    })


# ==========================================
# Results
# ==========================================

combined_results_df = pd.DataFrame(
    combined_results
).sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)


print()
print("======================================")
print("COMBINED MODEL RESULTS")
print("======================================")

print(
    combined_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ==========================================
# Best model
# ==========================================

best_combined_name = (
    combined_results_df.iloc[0]["Model"]
)

print()
print("======================================")
print("BEST COMBINED MODEL")
print("======================================")

print(
    "Best model:",
    best_combined_name
)

print(
    "Accuracy:",
    f"{combined_results_df.iloc[0]['Accuracy']:.4f}"
)

print(
    "ROC-AUC:",
    f"{combined_results_df.iloc[0]['ROC-AUC']:.4f}"
)

TRAINING COMBINED CUSTOM MODELS
Removed constant features: ['bowling_max_difference']
Final feature count: 63

Training: Logistic Regression

Training: Random Forest

Training: XGBoost

COMBINED MODEL RESULTS
              Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression    0.5301     0.4474  0.3119 0.3676   0.5296
            XGBoost    0.5542     0.4900  0.4495 0.4689   0.5284
      Random Forest    0.5181     0.4444  0.4037 0.4231   0.5204

BEST COMBINED MODEL
Best model: Logistic Regression
Accuracy: 0.5301
ROC-AUC: 0.5296


In [133]:
# ==========================================
# STEP 78 - ROLE-AWARE PLAYER PROFILES
# ==========================================

print("======================================")
print("BUILDING ROLE-AWARE PLAYER PROFILES")
print("======================================")

# Use the original player-level dataset
role_df = player_df.copy()

# Numeric conversion
numeric_cols = [
    "runs",
    "balls_faced",
    "wickets",
    "balls_bowled",
    "runs_conceded",
    "batting_experience",
    "bowling_experience"
]

for col in numeric_cols:
    if col in role_df.columns:
        role_df[col] = pd.to_numeric(
            role_df[col],
            errors="coerce"
        ).fillna(0)


# ==========================================
# Aggregate historical workload
# ==========================================

player_profile = (
    role_df
    .groupby("player")
    .agg(
        total_runs=("runs", "sum"),
        total_balls_faced=("balls_faced", "sum"),
        total_wickets=("wickets", "sum"),
        total_balls_bowled=("balls_bowled", "sum"),
        total_runs_conceded=("runs_conceded", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)


# ==========================================
# Batting / bowling workload
# ==========================================

player_profile["batting_workload"] = (
    player_profile["total_balls_faced"]
    /
    player_profile["matches"].replace(0, np.nan)
).fillna(0)


player_profile["bowling_workload"] = (
    player_profile["total_balls_bowled"]
    /
    player_profile["matches"].replace(0, np.nan)
).fillna(0)


# ==========================================
# Wickets per match
# ==========================================

player_profile["wickets_per_match"] = (
    player_profile["total_wickets"]
    /
    player_profile["matches"].replace(0, np.nan)
).fillna(0)


# ==========================================
# Batting / bowling participation
# ==========================================

player_profile["batting_ratio"] = (
    player_profile["batting_workload"]
    /
    (
        player_profile["batting_workload"]
        +
        player_profile["bowling_workload"]
        +
        1e-6
    )
)


player_profile["bowling_ratio"] = (
    player_profile["bowling_workload"]
    /
    (
        player_profile["batting_workload"]
        +
        player_profile["bowling_workload"]
        +
        1e-6
    )
)


# ==========================================
# Automatic role classification
# ==========================================

def classify_role(row):

    batting = row["batting_workload"]
    bowling = row["bowling_workload"]

    # Very little historical activity
    if batting == 0 and bowling == 0:
        return "Unknown"

    # Primarily batsman
    if batting > 20 and bowling < 2:
        return "Batter"

    # Primarily bowler
    if bowling >= 2 and batting < 10:
        return "Bowler"

    # Significant contribution in both
    if batting >= 10 and bowling >= 2:
        return "All-rounder"

    # Mostly batting with occasional bowling
    if batting >= 10 and bowling < 2:
        return "Batting All-rounder"

    # Mostly bowling with occasional batting
    if bowling >= 2 and batting >= 5:
        return "Bowling All-rounder"

    return "Utility"


player_profile["role"] = (
    player_profile.apply(
        classify_role,
        axis=1
    )
)


# ==========================================
# Output
# ==========================================

print()
print("======================================")
print("PLAYER ROLE DISTRIBUTION")
print("======================================")

print(
    player_profile["role"]
    .value_counts()
)


print()
print("======================================")
print("PLAYER PROFILES")
print("======================================")

print(
    "Players:",
    len(player_profile)
)

display(
    player_profile.head(20)
)

BUILDING ROLE-AWARE PLAYER PROFILES

PLAYER ROLE DISTRIBUTION
role
Bowler                 446
Batting All-rounder    160
Utility                 92
All-rounder             72
Batter                  36
Unknown                  5
Name: count, dtype: int64

PLAYER PROFILES
Players: 811


,player,total_runs,total_balls_faced,total_wickets,total_balls_bowled,total_runs_conceded,matches,batting_workload,bowling_workload,wickets_per_match,batting_ratio,bowling_ratio,role
0,A Ashish Reddy,280,193,18,262,396,31,6.225806,8.451613,0.580645,0.424176,0.575824,Bowler
1,A Badoni,1178,834,4,41,63,66,12.636364,0.621212,0.060606,0.953143,0.046857,Batting All-rounder
2,A Chandila,4,7,11,234,242,12,0.583333,19.500000,0.916667,0.029046,0.970954,Bowler
3,A Chopra,53,71,0,0,0,7,10.142857,0.000000,0.000000,1.000000,0.000000,Batting All-rounder
4,A Choudhary,25,20,5,101,144,5,4.000000,20.200000,1.000000,0.165289,0.834711,Bowler
5,A Dananjaya,4,5,0,24,47,1,5.000000,24.000000,0.000000,0.172414,0.827586,Bowler
6,A Flintoff,62,53,2,66,105,3,17.666667,22.000000,0.666667,0.445378,0.554622,All-rounder
7,A Kamboj,74,50,31,491,816,25,2.000000,19.640000,1.240000,0.092421,0.907579,Bowler
8,A Kumble,35,47,45,965,1058,42,1.119048,22.976190,1.071429,0.046443,0.953557,Bowler
9,A Manohar,292,235,0,0,0,27,8.703704,0.000000,0.000000,1.000000,0.000000,Utility


In [134]:
# ==========================================
# STEP 79 - CONTINUOUS PLAYER ROLE SCORES
# ==========================================

print("======================================")
print("BUILDING CONTINUOUS ROLE SCORES")
print("======================================")


role_score_df = player_profile.copy()


# ------------------------------------------
# Normalize workload
# ------------------------------------------

def percentile_score(series):

    return (
        series.rank(
            pct=True
        ) * 100
    )


role_score_df["batting_score"] = (
    percentile_score(
        role_score_df["batting_workload"]
    )
)


role_score_df["bowling_score"] = (
    percentile_score(
        role_score_df["bowling_workload"]
    )
)


# ------------------------------------------
# Batting dominance
# ------------------------------------------

role_score_df["batting_role_score"] = (
    role_score_df["batting_score"]
    /
    (
        role_score_df["batting_score"]
        +
        role_score_df["bowling_score"]
        +
        1e-6
    )
)


# ------------------------------------------
# Bowling dominance
# ------------------------------------------

role_score_df["bowling_role_score"] = (
    role_score_df["bowling_score"]
    /
    (
        role_score_df["batting_score"]
        +
        role_score_df["bowling_score"]
        +
        1e-6
    )
)


# ------------------------------------------
# Overall activity
# ------------------------------------------

role_score_df["activity_score"] = (
    role_score_df["batting_score"]
    +
    role_score_df["bowling_score"]
) / 2


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("ROLE SCORES CREATED")
print("======================================")

print(
    "Players:",
    len(role_score_df)
)

display(
    role_score_df[
        [
            "player",
            "batting_workload",
            "bowling_workload",
            "batting_score",
            "bowling_score",
            "batting_role_score",
            "bowling_role_score",
            "activity_score"
        ]
    ]
    .sort_values(
        "activity_score",
        ascending=False
    )
    .head(20)
)

BUILDING CONTINUOUS ROLE SCORES

ROLE SCORES CREATED
Players: 811


,player,batting_workload,bowling_workload,batting_score,bowling_score,batting_role_score,bowling_role_score,activity_score
507,P Ray Barman,24.000000,24.000000,97.842170,98.335388,0.498743,0.501257,98.088779
236,GR Napier,16.000000,24.000000,85.758323,98.335388,0.465841,0.534159,92.046856
6,A Flintoff,17.666667,22.000000,89.087546,87.792848,0.503660,0.496340,88.440197
101,Azhar Mahmood,13.173913,23.347826,77.928483,96.177559,0.447592,0.552408,87.053021
28,AA Noffke,10.000000,24.000000,67.385943,98.335388,0.406622,0.593378,82.860666
175,DAJ Bracewell,9.000000,24.000000,64.488286,98.335388,0.396062,0.603938,81.411837
294,JH Kallis,22.581633,17.775510,97.040691,61.282367,0.612928,0.387072,79.161529
39,AD Mascarenhas,6.000000,23.692308,53.390875,96.547472,0.356086,0.643914,74.969174
297,JJ van der Wath,5.333333,24.000000,51.294698,98.335388,0.342810,0.657190,74.815043
5,A Dananjaya,5.000000,24.000000,50.184957,98.335388,0.337900,0.662100,74.260173


In [135]:
# ==========================================
# STEP 80 - PERFORMANCE + ROLE FEATURES
# ==========================================

print("======================================")
print("BUILDING PERFORMANCE + ROLE FEATURES")
print("======================================")


# ------------------------------------------
# Start from player profiles
# ------------------------------------------

performance_df = role_score_df.copy()


# ------------------------------------------
# Merge performance statistics
# ------------------------------------------

performance_stats = (
    role_df
    .groupby("player")
    .agg(
        batting_form=("batting_form", "mean"),
        batting_strike_rate=("batting_strike_rate", "mean"),
        recent_strike_rate=("recent_strike_rate", "mean"),
        bowling_form=("bowling_form", "mean"),
        bowling_economy=("bowling_economy", "mean"),
        recent_bowling_economy=(
            "recent_bowling_economy",
            "mean"
        )
    )
    .reset_index()
)


performance_df = performance_df.merge(
    performance_stats,
    on="player",
    how="left"
)


# ------------------------------------------
# Clean
# ------------------------------------------

performance_columns = [
    "batting_form",
    "batting_strike_rate",
    "recent_strike_rate",
    "bowling_form",
    "bowling_economy",
    "recent_bowling_economy"
]

for col in performance_columns:

    performance_df[col] = pd.to_numeric(
        performance_df[col],
        errors="coerce"
    ).fillna(0)


# ------------------------------------------
# Normalize performance metrics
# ------------------------------------------

performance_df["batting_form_score"] = (
    percentile_score(
        performance_df["batting_form"]
    )
)

performance_df["strike_rate_score"] = (
    percentile_score(
        performance_df["batting_strike_rate"]
    )
)

performance_df["recent_form_score"] = (
    percentile_score(
        performance_df["recent_strike_rate"]
    )
)

performance_df["bowling_form_score"] = (
    percentile_score(
        performance_df["bowling_form"]
    )
)


# Economy is LOWER = better
performance_df["economy_score"] = (
    100
    -
    percentile_score(
        performance_df["bowling_economy"]
    )
)


performance_df["recent_economy_score"] = (
    100
    -
    percentile_score(
        performance_df["recent_bowling_economy"]
    )
)


# ------------------------------------------
# Composite batting performance
# ------------------------------------------

performance_df["batting_performance"] = (
    0.40 *
    performance_df["batting_form_score"]
    +
    0.30 *
    performance_df["strike_rate_score"]
    +
    0.30 *
    performance_df["recent_form_score"]
)


# ------------------------------------------
# Composite bowling performance
# ------------------------------------------

performance_df["bowling_performance"] = (
    0.40 *
    performance_df["bowling_form_score"]
    +
    0.30 *
    performance_df["economy_score"]
    +
    0.30 *
    performance_df["recent_economy_score"]
)


# ------------------------------------------
# Role-aware overall score
# ------------------------------------------

performance_df["role_aware_score"] = (
    performance_df["batting_role_score"]
    *
    performance_df["batting_performance"]
    +
    performance_df["bowling_role_score"]
    *
    performance_df["bowling_performance"]
)


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("ROLE-AWARE PERFORMANCE FEATURES")
print("======================================")

print(
    "Players:",
    len(performance_df)
)

display(
    performance_df[
        [
            "player",
            "batting_performance",
            "bowling_performance",
            "batting_role_score",
            "bowling_role_score",
            "role_aware_score"
        ]
    ]
    .sort_values(
        "role_aware_score",
        ascending=False
    )
    .head(20)
)

BUILDING PERFORMANCE + ROLE FEATURES

ROLE-AWARE PERFORMANCE FEATURES
Players: 811


,player,batting_performance,bowling_performance,batting_role_score,bowling_role_score,role_aware_score
767,V Suryavanshi,99.753391,58.175092,0.865791,0.134209,94.173207
539,Priyansh Arya,98.150432,58.175092,0.852665,0.147335,92.260647
10,A Mhatre,97.509248,58.175092,0.862814,0.137186,92.113143
274,J Fraser-McGurk,98.434032,58.175092,0.839151,0.160849,91.958432
135,C Connolly,96.991369,58.175092,0.868935,0.131065,91.903903
244,H Klaasen,97.016030,58.175092,0.866553,0.133447,91.832826
516,PD Salt,97.077682,58.175092,0.861029,0.138971,91.671348
220,FH Allen,97.509248,58.175092,0.850983,0.149017,91.647784
569,RD Rickelton,96.128237,58.175092,0.865020,0.134980,91.005324
285,JC Buttler,94.549938,58.175092,0.869950,0.130050,89.819396


In [136]:
# ==========================================
# STEP 81 - TIME-AWARE PLAYER PERFORMANCE
# ==========================================

print("======================================")
print("BUILDING TIME-AWARE PLAYER PERFORMANCE")
print("======================================")


# ------------------------------------------
# Prepare historical player data
# ------------------------------------------

time_df = player_df.copy()

time_df["date"] = pd.to_datetime(
    time_df["date"]
)


numeric_cols = [
    "runs",
    "balls_faced",
    "wickets",
    "runs_conceded",
    "balls_bowled",
    "previous_runs",
    "previous_wickets",
    "previous_strike_rate",
    "previous_economy",
    "last_5_runs",
    "last_5_wickets",
    "last_5_strike_rate",
    "last_5_economy"
]


for col in numeric_cols:

    if col in time_df.columns:

        time_df[col] = pd.to_numeric(
            time_df[col],
            errors="coerce"
        ).fillna(0)


# ------------------------------------------
# Sort chronologically
# ------------------------------------------

time_df = (
    time_df
    .sort_values(
        ["player", "date", "match_id"]
    )
    .copy()
)


# ------------------------------------------
# Shift match performance
# ------------------------------------------

time_df["prev_runs"] = (
    time_df
    .groupby("player")["runs"]
    .shift(1)
    .fillna(0)
)

time_df["prev_wickets"] = (
    time_df
    .groupby("player")["wickets"]
    .shift(1)
    .fillna(0)
)

time_df["prev_balls_faced"] = (
    time_df
    .groupby("player")["balls_faced"]
    .shift(1)
    .fillna(0)
)

time_df["prev_balls_bowled"] = (
    time_df
    .groupby("player")["balls_bowled"]
    .shift(1)
    .fillna(0)
)

time_df["prev_runs_conceded"] = (
    time_df
    .groupby("player")["runs_conceded"]
    .shift(1)
    .fillna(0)
)


# ------------------------------------------
# Rolling recent performance
# ------------------------------------------

time_df["rolling_runs"] = (
    time_df
    .groupby("player")["runs"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            5,
            min_periods=1
        )
        .mean()
    )
    .fillna(0)
)


time_df["rolling_wickets"] = (
    time_df
    .groupby("player")["wickets"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            5,
            min_periods=1
        )
        .mean()
    )
    .fillna(0)
)


time_df["rolling_balls_faced"] = (
    time_df
    .groupby("player")["balls_faced"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            5,
            min_periods=1
        )
        .sum()
    )
    .fillna(0)
)


time_df["rolling_balls_bowled"] = (
    time_df
    .groupby("player")["balls_bowled"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            5,
            min_periods=1
        )
        .sum()
    )
    .fillna(0)


)


time_df["rolling_runs_conceded"] = (
    time_df
    .groupby("player")["runs_conceded"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            5,
            min_periods=1
        )
        .sum()
    )
    .fillna(0)
)


# ------------------------------------------
# Strike rate
# ------------------------------------------

time_df["rolling_strike_rate"] = np.where(

    time_df["rolling_balls_faced"] > 0,

    (
        time_df["rolling_runs"]
        /
        time_df["rolling_balls_faced"]
    ) * 100,

    0
)


# ------------------------------------------
# Bowling economy
# ------------------------------------------

time_df["rolling_economy"] = np.where(

    time_df["rolling_balls_bowled"] > 0,

    (
        time_df["rolling_runs_conceded"]
        /
        time_df["rolling_balls_bowled"]
    ) * 6,

    0
)


# ------------------------------------------
# Experience before match
# ------------------------------------------

time_df["previous_matches"] = (
    time_df
    .groupby("player")
    .cumcount()
)


# ------------------------------------------
# Player performance score
# ------------------------------------------

# Batting:
# runs + strike rate + recent runs

time_df["batting_score"] = (

    0.45 *
    np.minimum(
        time_df["rolling_runs"],
        250
    )

    +

    0.30 *
    np.minimum(
        time_df["rolling_strike_rate"],
        200
    )

    +

    0.25 *
    np.minimum(
        time_df["previous_runs"],
        250
    )
)


# Bowling:
# wickets + economy + recent wickets

economy_quality = np.where(

    time_df["rolling_economy"] > 0,

    100 -
    np.minimum(
        time_df["rolling_economy"] * 10,
        100
    ),

    0
)


time_df["bowling_score"] = (

    0.45 *
    np.minimum(
        time_df["rolling_wickets"] * 20,
        100
    )

    +

    0.30 *
    economy_quality

    +

    0.25 *
    np.minimum(
        time_df["previous_wickets"] * 20,
        100
    )
)


# ------------------------------------------
# Normalize within historical dataset
# ------------------------------------------

time_df["batting_score"] = (
    time_df["batting_score"]
    .clip(0, 100)
)


time_df["bowling_score"] = (
    time_df["bowling_score"]
    .clip(0, 100)
)


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("TIME-AWARE PERFORMANCE CREATED")
print("======================================")

print(
    "Rows:",
    len(time_df)
)

print(
    "Players:",
    time_df["player"].nunique()
)

print()

display(
    time_df[
        [
            "date",
            "player",
            "previous_matches",
            "batting_score",
            "bowling_score",
            "rolling_runs",
            "rolling_wickets",
            "rolling_strike_rate",
            "rolling_economy"
        ]
    ]
    .head(20)
)

BUILDING TIME-AWARE PLAYER PERFORMANCE

TIME-AWARE PERFORMANCE CREATED
Rows: 27909
Players: 811



,date,player,previous_matches,batting_score,bowling_score,rolling_runs,rolling_wickets,rolling_strike_rate,rolling_economy
6160,2012-04-26,A Ashish Reddy,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6270,2012-04-29,A Ashish Reddy,1,0.000000,34.000000,0.000000,2.000000,0.000000,8.000000
6314,2012-05-01,A Ashish Reddy,2,19.750000,37.581081,5.000000,1.500000,50.000000,6.972973
6402,2012-05-04,A Ashish Reddy,3,14.000000,37.454545,3.333333,1.333333,33.333333,8.181818
6490,2012-05-06,A Ashish Reddy,4,12.212500,41.802239,3.250000,1.250000,25.000000,8.149254
6556,2012-05-08,A Ashish Reddy,5,10.420000,40.679121,2.600000,1.200000,20.000000,8.373626
6600,2012-05-10,A Ashish Reddy,6,13.140000,39.294505,4.200000,1.200000,20.000000,8.835165
6886,2012-05-18,A Ashish Reddy,7,12.240000,34.000000,2.200000,1.000000,20.000000,10.133333
6952,2012-05-20,A Ashish Reddy,8,18.040000,32.842857,4.200000,0.800000,28.000000,9.785714
7128,2013-04-05,A Ashish Reddy,9,18.980000,38.425000,4.400000,1.200000,27.500000,9.125000


In [137]:
# ==========================================
# STEP 82 - TIME-AWARE TEAM FEATURES
# ==========================================

print("======================================")
print("BUILDING TIME-AWARE TEAM FEATURES")
print("======================================")


# ------------------------------------------
# Prepare lookup
# ------------------------------------------

time_df = time_df.sort_values(
    ["player", "date", "match_id"]
).copy()


# ------------------------------------------
# Get player state BEFORE a match
# ------------------------------------------

def get_player_state(player, match_date):

    rows = time_df[
        (time_df["player"] == player) &
        (time_df["date"] < match_date)
    ]

    if len(rows) == 0:

        return {
            "batting": 50.0,
            "bowling": 50.0,
            "runs": 0.0,
            "wickets": 0.0,
            "strike_rate": 0.0,
            "economy": 0.0
        }

    row = rows.iloc[-1]

    return {
        "batting":
            float(row["batting_score"]),

        "bowling":
            float(row["bowling_score"]),

        "runs":
            float(row["rolling_runs"]),

        "wickets":
            float(row["rolling_wickets"]),

        "strike_rate":
            float(row["rolling_strike_rate"]),

        "economy":
            float(row["rolling_economy"])
    }


# ------------------------------------------
# Team state
# ------------------------------------------

def get_team_state(players, match_date):

    states = []

    for player in players:

        states.append(
            get_player_state(
                player,
                match_date
            )
        )

    df = pd.DataFrame(states)


    return {

        # Batting
        "batting_mean":
            df["batting"].mean(),

        "batting_max":
            df["batting"].max(),

        "batting_median":
            df["batting"].median(),

        # Bowling
        "bowling_mean":
            df["bowling"].mean(),

        "bowling_max":
            df["bowling"].max(),

        "bowling_median":
            df["bowling"].median(),

        # Recent form
        "recent_runs":
            df["runs"].mean(),

        "recent_wickets":
            df["wickets"].mean(),

        "recent_strike_rate":
            df["strike_rate"].mean(),

        "recent_economy":
            df["economy"].mean(),

        # Depth
        "batting_depth":
            (df["batting"] >= 60).sum(),

        "bowling_depth":
            (df["bowling"] >= 60).sum(),

        # Overall
        "overall":
            (
                df["batting"].mean()
                +
                df["bowling"].mean()
            ) / 2
    }


# ------------------------------------------
# Build match dataset
# ------------------------------------------

time_team_rows = []


for _, match in custom_match_df.iterrows():

    match_date = match["date"]


    t1 = get_team_state(
        match["team1_players"],
        match_date
    )

    t2 = get_team_state(
        match["team2_players"],
        match_date
    )


    result = {

        "match_id":
            match["match_id"],

        "date":
            match_date,

        "team1_win":
            match["team1_win"]
    }


    # --------------------------------------
    # Team features
    # --------------------------------------

    for key, value in t1.items():

        result[
            "team1_" + key
        ] = value


    for key, value in t2.items():

        result[
            "team2_" + key
        ] = value


    # --------------------------------------
    # Difference features
    # --------------------------------------

    for key in t1.keys():

        result[
            key + "_diff"
        ] = (
            t1[key]
            -
            t2[key]
        )


    time_team_rows.append(result)


time_team_df = pd.DataFrame(
    time_team_rows
)


# ------------------------------------------
# Clean
# ------------------------------------------

time_team_df = time_team_df.replace(
    [np.inf, -np.inf],
    np.nan
)

time_team_df = time_team_df.fillna(0)


# ------------------------------------------
# Output
# ------------------------------------------

print()
print("======================================")
print("TIME-AWARE TEAM DATASET")
print("======================================")

print(
    "Shape:",
    time_team_df.shape
)

print(
    "Matches:",
    len(time_team_df)
)

print(
    "Features:",
    len([
        c for c in time_team_df.columns
        if c not in [
            "match_id",
            "date",
            "team1_win"
        ]
    ])
)

print()

display(
    time_team_df.head(10)
)

BUILDING TIME-AWARE TEAM FEATURES

TIME-AWARE TEAM DATASET
Shape: (1243, 42)
Matches: 1243
Features: 39



,match_id,date,team1_win,team1_batting_mean,team1_batting_max,team1_batting_median,team1_bowling_mean,team1_bowling_max,team1_bowling_median,team1_recent_runs,...,bowling_mean_diff,bowling_max_diff,bowling_median_diff,recent_runs_diff,recent_wickets_diff,recent_strike_rate_diff,recent_economy_diff,batting_depth_diff,bowling_depth_diff,overall_diff
0,1082591,2017-04-05,1,61.154482,94.219799,71.962857,30.537999,50.000000,34.600000,14.045455,...,-0.719682,-1.968421,-7.285714,6.854545,0.018182,8.485339,1.203530,1,0,5.330232
1,1082592,2017-04-06,1,48.753202,91.341096,72.253846,22.803152,60.596154,10.500000,11.072727,...,-1.410807,14.462821,-14.500000,-2.466667,0.122727,-9.328520,-1.737307,0,1,-2.750591
2,1082593,2017-04-07,0,53.355490,85.167865,64.680000,28.628023,50.000000,32.200000,10.763636,...,5.857090,0.000000,-1.925000,0.136364,0.136364,2.292931,1.223773,0,0,4.348621
3,1082594,2017-04-08,1,54.085671,87.073529,54.553721,20.560726,50.000000,20.000000,13.818182,...,0.930478,0.000000,15.000000,1.709091,0.036364,4.528166,1.101962,-2,0,-0.561899
4,1082595,2017-04-08,1,52.631268,91.886667,61.014615,28.375129,50.000000,35.517647,8.127273,...,8.538495,3.566667,16.517647,-3.727273,-0.060606,-2.855938,0.547774,0,0,5.104493
5,1082596,2017-04-09,1,57.757595,96.392041,71.290000,25.811136,47.350000,31.830769,14.636364,...,-1.023792,-2.650000,-2.319231,4.709091,0.072727,9.633963,1.165506,0,0,1.254107
6,1082597,2017-04-09,1,55.430365,82.590000,72.212500,27.841036,51.550000,27.300000,13.000000,...,10.522238,3.550000,27.300000,2.127273,0.036364,7.290554,2.207739,1,0,12.469702
7,1082598,2017-04-10,1,49.237623,82.240000,61.500526,18.588842,43.442105,20.000000,11.018182,...,0.095265,-4.926316,-0.750000,-0.654545,-0.036364,-15.883714,0.404609,-1,0,-1.095834
8,1082599,2017-04-11,0,47.189702,85.829362,59.700000,23.222193,56.305882,17.000000,10.327273,...,4.822750,10.941176,-0.500000,-3.727273,0.263636,10.127322,-0.059098,-2,0,-1.624072
9,1082600,2017-04-12,1,57.334373,83.575570,72.082162,28.270257,52.750000,33.100000,13.036364,...,2.049511,0.900000,2.100000,0.381818,-0.236364,4.451177,1.791340,1,0,6.573796


In [138]:
# ==========================================
# STEP 83 - EXPERIENCE / RELIABILITY FEATURES
# ==========================================

print("======================================")
print("ADDING EXPERIENCE & RELIABILITY")
print("======================================")


# ------------------------------------------
# Get player experience before each match
# ------------------------------------------

def get_player_experience(player, match_date):

    rows = time_df[
        (time_df["player"] == player) &
        (time_df["date"] < match_date)
    ]

    return len(rows)


def get_team_experience(players, match_date):

    experiences = []

    for player in players:

        experiences.append(
            get_player_experience(
                player,
                match_date
            )
        )

    if len(experiences) == 0:

        return {
            "experience_mean": 0,
            "experience_median": 0,
            "experienced_players": 0,
            "new_players": 5
        }

    return {

        "experience_mean":
            np.mean(experiences),

        "experience_median":
            np.median(experiences),

        "experienced_players":
            sum(
                x >= 5
                for x in experiences
            ),

        "new_players":
            sum(
                x == 0
                for x in experiences
            )
    }


# ------------------------------------------
# Add to existing team dataset
# ------------------------------------------

experience_rows = []


for _, match in custom_match_df.iterrows():

    match_date = match["date"]


    t1 = get_team_experience(
        match["team1_players"],
        match_date
    )

    t2 = get_team_experience(
        match["team2_players"],
        match_date
    )


    row = {
        "team1_experience_mean":
            t1["experience_mean"],

        "team2_experience_mean":
            t2["experience_mean"],

        "team1_experience_median":
            t1["experience_median"],

        "team2_experience_median":
            t2["experience_median"],

        "team1_experienced_players":
            t1["experienced_players"],

        "team2_experienced_players":
            t2["experienced_players"],

        "team1_new_players":
            t1["new_players"],

        "team2_new_players":
            t2["new_players"],

        "experience_mean_diff":
            (
                t1["experience_mean"]
                -
                t2["experience_mean"]
            ),

        "experienced_players_diff":
            (
                t1["experienced_players"]
                -
                t2["experienced_players"]
            ),

        "new_players_diff":
            (
                t1["new_players"]
                -
                t2["new_players"]
            )
    }

    experience_rows.append(row)


experience_df = pd.DataFrame(
    experience_rows
)


# ------------------------------------------
# Combine
# ------------------------------------------

final_time_team_df = pd.concat(
    [
        time_team_df.reset_index(drop=True),
        experience_df.reset_index(drop=True)
    ],
    axis=1
)


# ------------------------------------------
# Clean
# ------------------------------------------

final_time_team_df = (
    final_time_team_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)


print()
print("======================================")
print("FINAL TIME-AWARE TEAM FEATURES")
print("======================================")

print(
    "Shape:",
    final_time_team_df.shape
)

print(
    "Total features:",
    len([
        c for c in final_time_team_df.columns
        if c not in [
            "match_id",
            "date",
            "team1_win"
        ]
    ])
)

display(
    final_time_team_df.head(10)
)

ADDING EXPERIENCE & RELIABILITY

FINAL TIME-AWARE TEAM FEATURES
Shape: (1243, 53)
Total features: 50


,match_id,date,team1_win,team1_batting_mean,team1_batting_max,team1_batting_median,team1_bowling_mean,team1_bowling_max,team1_bowling_median,team1_recent_runs,...,team2_experience_mean,team1_experience_median,team2_experience_median,team1_experienced_players,team2_experienced_players,team1_new_players,team2_new_players,experience_mean_diff,experienced_players_diff,new_players_diff
0,1082591,2017-04-05,1,61.154482,94.219799,71.962857,30.537999,50.000000,34.600000,14.045455,...,41.454545,76.0,43.0,10,8,1,2,20.727273,2,-1
1,1082592,2017-04-06,1,48.753202,91.341096,72.253846,22.803152,60.596154,10.500000,11.072727,...,53.909091,54.0,26.0,9,10,1,0,1.727273,-1,1
2,1082593,2017-04-07,0,53.355490,85.167865,64.680000,28.628023,50.000000,32.200000,10.763636,...,67.272727,66.0,66.0,10,9,1,1,4.818182,1,0
3,1082594,2017-04-08,1,54.085671,87.073529,54.553721,20.560726,50.000000,20.000000,13.818182,...,57.909091,43.0,55.0,9,9,1,1,-22.454545,0,0
4,1082595,2017-04-08,1,52.631268,91.886667,61.014615,28.375129,50.000000,35.517647,8.127273,...,39.909091,47.0,39.0,8,10,2,0,4.909091,-2,2
5,1082596,2017-04-09,1,57.757595,96.392041,71.290000,25.811136,47.350000,31.830769,14.636364,...,63.818182,77.0,67.0,10,8,0,2,-0.636364,2,-2
6,1082597,2017-04-09,1,55.430365,82.590000,72.212500,27.841036,51.550000,27.300000,13.000000,...,57.545455,32.0,46.0,11,9,0,0,5.181818,2,0
7,1082598,2017-04-10,1,49.237623,82.240000,61.500526,18.588842,43.442105,20.000000,11.018182,...,48.181818,44.0,48.0,10,8,0,0,-8.636364,2,0
8,1082599,2017-04-11,0,47.189702,85.829362,59.700000,23.222193,56.305882,17.000000,10.327273,...,41.545455,50.0,40.0,8,11,1,0,8.090909,-3,1
9,1082600,2017-04-12,1,57.334373,83.575570,72.082162,28.270257,52.750000,33.100000,13.036364,...,58.909091,33.0,78.0,11,9,0,0,4.818182,2,0


In [139]:
# ==========================================
# STEP 84 - TRAIN TIME-AWARE CUSTOM MODEL
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import xgboost as xgb

print("======================================")
print("TRAINING TIME-AWARE CUSTOM MODELS")
print("======================================")


# ------------------------------------------
# Features / target
# ------------------------------------------

X_time = final_time_team_df.drop(
    columns=[
        "match_id",
        "date",
        "team1_win"
    ],
    errors="ignore"
).copy()

y_time = final_time_team_df[
    "team1_win"
].copy()


# ------------------------------------------
# Clean
# ------------------------------------------

X_time = X_time.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)


# ------------------------------------------
# Chronological split
# ------------------------------------------

split_index = 994

X_train_time = X_time.iloc[
    :split_index
]

X_test_time = X_time.iloc[
    split_index:
]

y_train_time = y_time.iloc[
    :split_index
]

y_test_time = y_time.iloc[
    split_index:
]


print(
    "Training:",
    X_train_time.shape
)

print(
    "Testing:",
    X_test_time.shape
)


# ==========================================
# Models
# ==========================================

time_models = {

    "Logistic Regression":
        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    C=0.1,
                    max_iter=5000
                )
            )
        ]),

    "Random Forest":
        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=5,
                    min_samples_leaf=8,
                    max_features="sqrt",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]),

    "XGBoost":
        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "classifier",
                xgb.XGBClassifier(
                    n_estimators=300,
                    max_depth=2,
                    learning_rate=0.03,
                    min_child_weight=8,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_alpha=0.5,
                    reg_lambda=2.0,
                    eval_metric="logloss",
                    random_state=42
                )
            )
        ])
}


# ==========================================
# Train + evaluate
# ==========================================

time_results = []

trained_time_models = {}


for name, model in time_models.items():

    print()
    print("Training:", name)

    model.fit(
        X_train_time,
        y_train_time
    )

    predictions = model.predict(
        X_test_time
    )

    probabilities = model.predict_proba(
        X_test_time
    )[:, 1]


    trained_time_models[name] = model


    time_results.append({

        "Model": name,

        "Accuracy":
            accuracy_score(
                y_test_time,
                predictions
            ),

        "Precision":
            precision_score(
                y_test_time,
                predictions,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_test_time,
                predictions,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_test_time,
                predictions,
                zero_division=0
            ),

        "ROC-AUC":
            roc_auc_score(
                y_test_time,
                probabilities
            )
    })


# ==========================================
# Results
# ==========================================

time_results_df = pd.DataFrame(
    time_results
).sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)


print()
print("======================================")
print("TIME-AWARE MODEL RESULTS")
print("======================================")

print(
    time_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ==========================================
# Compare baseline
# ==========================================

best_accuracy = (
    time_results_df.iloc[0]["Accuracy"]
)

print()
print("======================================")
print("BASELINE COMPARISON")
print("======================================")

print(
    "Previous best: 55.42%"
)

print(
    f"New best: {best_accuracy * 100:.2f}%"
)

print(
    f"Improvement: {(best_accuracy - 0.5542) * 100:+.2f}%"
)

TRAINING TIME-AWARE CUSTOM MODELS
Training: (994, 50)
Testing: (249, 50)

Training: Logistic Regression

Training: Random Forest

Training: XGBoost

TIME-AWARE MODEL RESULTS
              Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression    0.5542     0.6374  0.4265 0.5110   0.6007
      Random Forest    0.5181     0.6143  0.3162 0.4175   0.5745
            XGBoost    0.4980     0.5775  0.3015 0.3961   0.5608

BASELINE COMPARISON
Previous best: 55.42%
New best: 55.42%
Improvement: +0.00%


In [140]:
# ==========================================
# STEP 84.5 - THRESHOLD ANALYSIS
# ==========================================

from sklearn.metrics import accuracy_score

print("======================================")
print("THRESHOLD ANALYSIS")
print("======================================")


# Get Logistic Regression model
best_time_model = trained_time_models[
    "Logistic Regression"
]


# Test probabilities
test_probabilities = (
    best_time_model
    .predict_proba(X_test_time)[:, 1]
)


threshold_results = []


for threshold in np.arange(
    0.30,
    0.71,
    0.01
):

    predictions = (
        test_probabilities >= threshold
    ).astype(int)


    accuracy = accuracy_score(
        y_test_time,
        predictions
    )


    threshold_results.append({

        "Threshold":
            round(threshold, 2),

        "Accuracy":
            accuracy,

        "Correct":
            int(
                (predictions == y_test_time)
                .sum()
            ),

        "Team1_predictions":
            int(
                predictions.sum()
            ),

        "Team2_predictions":
            int(
                len(predictions)
                -
                predictions.sum()
            )
    })


threshold_df = pd.DataFrame(
    threshold_results
)


# ------------------------------------------
# Best threshold
# ------------------------------------------

best_threshold_row = (
    threshold_df
    .sort_values(
        "Accuracy",
        ascending=False
    )
    .iloc[0]
)


print()
print("======================================")
print("THRESHOLD RESULTS")
print("======================================")

display(
    threshold_df
    .sort_values(
        "Accuracy",
        ascending=False
    )
    .head(15)
)


print()
print("======================================")
print("BEST THRESHOLD")
print("======================================")

print(
    "Threshold:",
    best_threshold_row["Threshold"]
)

print(
    "Accuracy:",
    f"{best_threshold_row['Accuracy'] * 100:.2f}%"
)

print(
    "Correct:",
    int(best_threshold_row["Correct"]),
    "/",
    len(y_test_time)
)

THRESHOLD ANALYSIS

THRESHOLD RESULTS


,Threshold,Accuracy,Correct,Team1_predictions,Team2_predictions
13,0.43,0.614458,153,180,69
12,0.42,0.602410,150,187,62
14,0.44,0.590361,147,168,81
11,0.41,0.590361,147,198,51
16,0.46,0.586345,146,143,106
17,0.47,0.582329,145,132,117
10,0.40,0.578313,144,217,32
9,0.39,0.570281,142,221,28
19,0.49,0.570281,142,105,144
3,0.33,0.562249,140,243,6



BEST THRESHOLD
Threshold: 0.43
Accuracy: 61.45%
Correct: 153 / 249


In [143]:
# Find all pandas DataFrames currently available

df_variables = []

for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        df_variables.append(
            (name, obj.shape, list(obj.columns[:10]))
        )

for name, shape, columns in df_variables:
    print("\n======================================")
    print("Variable:", name)
    print("Shape:", shape)
    print("Columns:", columns)


Variable: __
Shape: (5, 15)
Columns: ['match_id', 'date', 'season', 'venue', 'player', 'team', 'runs', 'balls_faced', 'fours', 'sixes']

Variable: player_df
Shape: (27909, 48)
Columns: ['match_id', 'date', 'season', 'venue', 'player', 'team', 'runs', 'balls_faced', 'fours', 'sixes']

Variable: matchup_df
Shape: (61429, 22)
Columns: ['match_id', 'date', 'season', 'batter', 'bowler', 'balls', 'runs', 'fours', 'sixes', 'dismissals']

Variable: _12
Shape: (5, 15)
Columns: ['match_id', 'date', 'season', 'venue', 'player', 'team', 'runs', 'balls_faced', 'fours', 'sixes']

Variable: match_info_df
Shape: (1243, 9)
Columns: ['match_id', 'date', 'season', 'venue', 'team_1', 'team_2', 'toss_winner', 'toss_decision', 'winner']

Variable: team1_features
Shape: (2486, 13)
Columns: ['match_id', 'team_1', 'team1_batting_experience', 'team1_batting_form', 'team1_batting_strike_rate', 'team1_recent_strike_rate', 'team1_bowling_experience', 'team1_bowling_form', 'team1_bowling_economy', 'team1_recent_bo

In [144]:
print("======================================")
print("MATCHUP DATA CHECK")
print("======================================")

print("Shape:", matchup_df.shape)

print("\nColumns:")
print(matchup_df.columns.tolist())

print("\nSample:")
display(matchup_df.head(10))

MATCHUP DATA CHECK
Shape: (61429, 22)

Columns:
['match_id', 'date', 'season', 'batter', 'bowler', 'balls', 'runs', 'fours', 'sixes', 'dismissals', 'previous_balls', 'previous_runs', 'previous_dismissals', 'previous_fours', 'previous_sixes', 'previous_strike_rate', 'last_5_runs', 'last_5_balls', 'last_5_strike_rate', 'last_5_dismissals', 'matchup_sr', 'matchup_score']

Sample:


,match_id,date,season,batter,bowler,balls,runs,fours,sixes,dismissals,...,previous_dismissals,previous_fours,previous_sixes,previous_strike_rate,last_5_runs,last_5_balls,last_5_strike_rate,last_5_dismissals,matchup_sr,matchup_score
0,598044,2013-05-04,2013,A Ashish Reddy,A Nehra,4,5,1,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
1,829773,2015-05-02,2015,A Ashish Reddy,A Nehra,4,2,0,0,0,...,1.0,1.0,0.0,125.0,5.0,4.0,125.0,1.0,125.0,12.50
2,598000,2013-04-05,2013,A Ashish Reddy,AB Dinda,3,6,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
3,598018,2013-04-17,2013,A Ashish Reddy,AB Dinda,4,3,0,0,0,...,0.0,1.0,0.0,200.0,6.0,3.0,200.0,0.0,200.0,32.00
4,598018,2013-04-17,2013,A Ashish Reddy,AD Mathews,8,13,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
5,829731,2015-04-18,2015,A Ashish Reddy,AD Mathews,4,12,1,1,0,...,0.0,0.0,1.0,162.5,13.0,8.0,162.5,0.0,162.5,42.25
6,980915,2016-04-16,2016,A Ashish Reddy,AD Russell,3,4,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
7,829759,2015-04-27,2015,A Ashish Reddy,Anureet Singh,2,2,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
8,598021,2013-04-19,2013,A Ashish Reddy,Azhar Mahmood,3,2,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
9,598000,2013-04-05,2013,A Ashish Reddy,B Kumar,1,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00


In [146]:
# ==========================================
# STEP 85A - FAST PLAYER MATCHUP FEATURES
# ==========================================

print("======================================")
print("BUILDING FAST PLAYER MATCHUP FEATURES")
print("======================================")


# -------------------------------------------------
# 1. Prepare matchup history
# -------------------------------------------------

mhist = matchup_df[
    [
        "match_id",
        "date",
        "batter",
        "bowler",
        "previous_balls",
        "previous_runs",
        "previous_dismissals",
        "previous_strike_rate",
        "matchup_score"
    ]
].copy()

mhist["date"] = pd.to_datetime(mhist["date"])

mhist = mhist.sort_values(
    ["batter", "bowler", "date"]
)


# -------------------------------------------------
# 2. Expand playing XI combinations
# -------------------------------------------------

match_rows = []

for _, row in custom_match_df.iterrows():

    t1 = row["team1_players"]
    t2 = row["team2_players"]

    match_id = row["match_id"]
    date = pd.to_datetime(row["date"])

    # Team 1 batter vs Team 2 bowler
    for batter in t1:
        for bowler in t2:
            match_rows.append({
                "match_id": match_id,
                "date": date,
                "side": "team1",
                "batter": batter,
                "bowler": bowler
            })

    # Team 2 batter vs Team 1 bowler
    for batter in t2:
        for bowler in t1:
            match_rows.append({
                "match_id": match_id,
                "date": date,
                "side": "team2",
                "batter": batter,
                "bowler": bowler
            })


combos = pd.DataFrame(match_rows)

print("Player combinations:", len(combos))


# -------------------------------------------------
# 3. Merge historical matchup data
# -------------------------------------------------

combos = combos.merge(
    mhist,
    on=["batter", "bowler"],
    how="left",
    suffixes=("", "_history")
)


# -------------------------------------------------
# 4. CRITICAL: leakage protection
# -------------------------------------------------

# Only use history BEFORE the current match
valid_history = (
    combos["date_history"] < combos["date"]
)

combos.loc[
    ~valid_history,
    [
        "previous_balls",
        "previous_runs",
        "previous_dismissals",
        "previous_strike_rate",
        "matchup_score"
    ]
] = np.nan


# -------------------------------------------------
# 5. Use the most recent historical record
# -------------------------------------------------

combos = combos.sort_values(
    ["match_id", "side", "batter", "bowler", "date_history"]
)

combos = (
    combos
    .groupby(
        [
            "match_id",
            "side",
            "batter",
            "bowler"
        ],
        as_index=False
    )
    .last()
)


# -------------------------------------------------
# 6. Aggregate matchup features
# -------------------------------------------------

combos["has_history"] = (
    combos["previous_balls"].notna()
).astype(int)

numeric_cols = [
    "previous_balls",
    "previous_runs",
    "previous_dismissals",
    "previous_strike_rate",
    "matchup_score"
]

combos[numeric_cols] = (
    combos[numeric_cols]
    .fillna(0)
)


def aggregate_side(group):

    return pd.Series({

        "matchup_mean":
            group["matchup_score"].mean(),

        "matchup_max":
            group["matchup_score"].max(),

        "matchup_median":
            group["matchup_score"].median(),

        "matchup_runs":
            group["previous_runs"].sum(),

        "matchup_balls":
            group["previous_balls"].sum(),

        "matchup_dismissals":
            group["previous_dismissals"].sum(),

        "matchup_strike_rate":
            (
                group["previous_runs"].sum()
                /
                group["previous_balls"].sum()
                * 100
                if group["previous_balls"].sum() > 0
                else 0
            ),

        "matchup_history_count":
            group["has_history"].sum(),

        "matchup_history_ratio":
            group["has_history"].mean()
    })


aggregated = (
    combos
    .groupby(
        ["match_id", "side"]
    )
    .apply(
        aggregate_side,
        include_groups=False
    )
    .reset_index()
)


# -------------------------------------------------
# 7. Convert Team 1 / Team 2 to columns
# -------------------------------------------------

t1 = aggregated[
    aggregated["side"] == "team1"
].drop(columns="side")

t2 = aggregated[
    aggregated["side"] == "team2"
].drop(columns="side")


t1 = t1.rename(
    columns={
        c: "team1_" + c
        for c in t1.columns
        if c != "match_id"
    }
)

t2 = t2.rename(
    columns={
        c: "team2_" + c
        for c in t2.columns
        if c != "match_id"
    }
)


# -------------------------------------------------
# 8. Merge
# -------------------------------------------------

player_matchup_df = (
    custom_match_df[
        ["match_id", "date"]
    ]
    .merge(t1, on="match_id", how="left")
    .merge(t2, on="match_id", how="left")
)


# -------------------------------------------------
# 9. Difference features
# -------------------------------------------------

for feature in [
    "matchup_mean",
    "matchup_max",
    "matchup_median",
    "matchup_runs",
    "matchup_balls",
    "matchup_dismissals",
    "matchup_strike_rate",
    "matchup_history_count",
    "matchup_history_ratio"
]:

    player_matchup_df[
        feature + "_diff"
    ] = (
        player_matchup_df[
            "team1_" + feature
        ]
        -
        player_matchup_df[
            "team2_" + feature
        ]
    )


# -------------------------------------------------
# 10. Clean
# -------------------------------------------------

player_matchup_df = (
    player_matchup_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)


print()
print("======================================")
print("PLAYER MATCHUP FEATURES CREATED")
print("======================================")

print(
    "Shape:",
    player_matchup_df.shape
)

print(
    "Matches:",
    len(player_matchup_df)
)

print(
    "Features:",
    len(player_matchup_df.columns) - 2
)

display(
    player_matchup_df.head()
)

BUILDING FAST PLAYER MATCHUP FEATURES
Player combinations: 313736

PLAYER MATCHUP FEATURES CREATED
Shape: (1243, 29)
Matches: 1243
Features: 27


,match_id,date,team1_matchup_mean,team1_matchup_max,team1_matchup_median,team1_matchup_runs,team1_matchup_balls,team1_matchup_dismissals,team1_matchup_strike_rate,team1_matchup_history_count,...,team2_matchup_history_ratio,matchup_mean_diff,matchup_max_diff,matchup_median_diff,matchup_runs_diff,matchup_balls_diff,matchup_dismissals_diff,matchup_strike_rate_diff,matchup_history_count_diff,matchup_history_ratio_diff
0,1082591,2017-04-05,5.760602,157.857143,0.0,292.0,225.0,9.0,129.777778,29.0,...,0.239669,0.553787,24.964286,0.0,16.0,30.0,3.0,-11.760684,0.0,0.000000
1,1082592,2017-04-06,3.778831,168.638298,0.0,207.0,140.0,6.0,147.857143,30.0,...,0.165289,-1.521714,37.292144,0.0,-91.0,-70.0,-3.0,5.952381,10.0,0.082645
2,1082593,2017-04-07,7.137565,202.181818,0.0,467.0,342.0,18.0,136.549708,24.0,...,0.297521,0.167978,98.898799,0.0,67.0,21.0,3.0,11.939116,-12.0,-0.099174
3,1082594,2017-04-08,1.615014,72.000000,0.0,64.0,48.0,3.0,133.333333,15.0,...,0.157025,-1.837625,-3.846154,0.0,-104.0,-103.0,-4.0,22.075055,-4.0,-0.033058
4,1082595,2017-04-08,4.745025,75.058824,0.0,245.0,207.0,9.0,118.357488,23.0,...,0.190083,3.275853,-1.703081,0.0,183.0,151.0,7.0,7.643202,0.0,0.000000


In [147]:
# ==========================================
# STEP 86 - MERGE TIME-AWARE + MATCHUP FEATURES
# ==========================================

print("======================================")
print("BUILDING ENHANCED MATCHUP DATASET")
print("======================================")

# Keep only feature columns from matchup dataset
matchup_features = player_matchup_df.drop(
    columns=["date"],
    errors="ignore"
).copy()

# Merge with our best time-aware dataset
enhanced_df = final_time_team_df.merge(
    matchup_features,
    on="match_id",
    how="inner",
    suffixes=("", "_matchup")
)

print()
print("======================================")
print("ENHANCED DATASET")
print("======================================")

print("Shape:", enhanced_df.shape)

# Separate target
target_col = "team1_win"

feature_cols = [
    c for c in enhanced_df.columns
    if c not in [
        "match_id",
        "date",
        "team1_win"
    ]
]

X_enhanced = enhanced_df[feature_cols].copy()
y_enhanced = enhanced_df[target_col].copy()

print("Features:", len(feature_cols))
print("Target:", target_col)

# Check data quality
print()
print("======================================")
print("DATA QUALITY CHECK")
print("======================================")

print("NaN values:", X_enhanced.isna().sum().sum())
print(
    "Infinite values:",
    np.isinf(
        X_enhanced.select_dtypes(
            include=np.number
        )
    ).sum().sum()
)

print(
    "Duplicate columns:",
    X_enhanced.columns.duplicated().sum()
)

# Remove constant columns
constant_cols = [
    c for c in X_enhanced.columns
    if X_enhanced[c].nunique() <= 1
]

print(
    "Constant features:",
    constant_cols
)

if constant_cols:
    X_enhanced = X_enhanced.drop(
        columns=constant_cols
    )

print()
print("Final feature count:", X_enhanced.shape[1])

display(
    X_enhanced.head()
)

BUILDING ENHANCED MATCHUP DATASET

ENHANCED DATASET
Shape: (1243, 80)
Features: 77
Target: team1_win

DATA QUALITY CHECK
NaN values: 0
Infinite values: 0
Duplicate columns: 0
Constant features: ['team1_matchup_median', 'team2_matchup_median', 'matchup_median_diff']

Final feature count: 74


,team1_batting_mean,team1_batting_max,team1_batting_median,team1_bowling_mean,team1_bowling_max,team1_bowling_median,team1_recent_runs,team1_recent_wickets,team1_recent_strike_rate,team1_recent_economy,...,team2_matchup_history_count,team2_matchup_history_ratio,matchup_mean_diff,matchup_max_diff,matchup_runs_diff,matchup_balls_diff,matchup_dismissals_diff,matchup_strike_rate_diff,matchup_history_count_diff,matchup_history_ratio_diff
0,61.154482,94.219799,71.962857,30.537999,50.000000,34.600000,14.045455,0.527273,25.280090,4.887334,...,29.0,0.239669,0.553787,24.964286,16.0,30.0,3.0,-11.760684,0.0,0.000000
1,48.753202,91.341096,72.253846,22.803152,60.596154,10.500000,11.072727,0.504545,14.689461,3.306525,...,20.0,0.165289,-1.521714,37.292144,-91.0,-70.0,-3.0,5.952381,10.0,0.082645
2,53.355490,85.167865,64.680000,28.628023,50.000000,32.200000,10.763636,0.581818,16.857695,4.515280,...,36.0,0.297521,0.167978,98.898799,67.0,21.0,3.0,11.939116,-12.0,-0.099174
3,54.085671,87.073529,54.553721,20.560726,50.000000,20.000000,13.818182,0.363636,22.967387,4.025212,...,19.0,0.157025,-1.837625,-3.846154,-104.0,-103.0,-4.0,22.075055,-4.0,-0.033058
4,52.631268,91.886667,61.014615,28.375129,50.000000,35.517647,8.127273,0.381818,16.731498,4.808290,...,23.0,0.190083,3.275853,-1.703081,183.0,151.0,7.0,7.643202,0.0,0.000000


In [148]:
# ==========================================
# STEP 87 - TRAIN ENHANCED MATCHUP MODELS
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("======================================")
print("TRAINING ENHANCED MATCHUP MODELS")
print("======================================")


# -------------------------------------------------
# Chronological split
# -------------------------------------------------

enhanced_df = enhanced_df.sort_values(
    "date"
).reset_index(drop=True)

X_enhanced = enhanced_df[
    [
        c for c in enhanced_df.columns
        if c not in [
            "match_id",
            "date",
            "team1_win"
        ]
    ]
]

y_enhanced = enhanced_df["team1_win"]


split_idx = int(
    len(enhanced_df) * 0.80
)

X_enh_train = X_enhanced.iloc[:split_idx]
X_enh_test = X_enhanced.iloc[split_idx:]

y_enh_train = y_enhanced.iloc[:split_idx]
y_enh_test = y_enhanced.iloc[split_idx:]


print("Training:", X_enh_train.shape)
print("Testing :", X_enh_test.shape)


# -------------------------------------------------
# Models
# -------------------------------------------------

models_enhanced = {

    "Logistic Regression":
        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=3000,
                    C=0.1,
                    class_weight="balanced"
                )
            )
        ]),

    "Random Forest":
        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=5,
                    max_features="sqrt",
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]),

    "XGBoost":
        XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42
        )
}


# -------------------------------------------------
# Train + evaluate
# -------------------------------------------------

enhanced_results = []

enhanced_models = {}

for name, model in models_enhanced.items():

    print()
    print("Training:", name)

    model.fit(
        X_enh_train,
        y_enh_train
    )

    enhanced_models[name] = model

    probabilities = model.predict_proba(
        X_enh_test
    )[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score
    )

    enhanced_results.append({

        "Model": name,

        "Accuracy":
            accuracy_score(
                y_enh_test,
                predictions
            ),

        "Precision":
            precision_score(
                y_enh_test,
                predictions,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_enh_test,
                predictions,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_enh_test,
                predictions,
                zero_division=0
            ),

        "ROC-AUC":
            roc_auc_score(
                y_enh_test,
                probabilities
            )
    })


enhanced_results_df = pd.DataFrame(
    enhanced_results
).sort_values(
    "Accuracy",
    ascending=False
)


print()
print("======================================")
print("ENHANCED MODEL RESULTS")
print("======================================")

display(
    enhanced_results_df
)

TRAINING ENHANCED MATCHUP MODELS
Training: (994, 77)
Testing : (249, 77)

Training: Logistic Regression

Training: Random Forest

Training: XGBoost

ENHANCED MODEL RESULTS


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
2,XGBoost,0.554217,0.488636,0.394495,0.436548,0.507667
1,Random Forest,0.546185,0.469697,0.284404,0.354286,0.504849
0,Logistic Regression,0.530120,0.468254,0.541284,0.502128,0.515269


In [149]:
# ==========================================
# STEP 88 - ENHANCED THRESHOLD OPTIMIZATION
# ==========================================

from sklearn.metrics import accuracy_score

print("======================================")
print("OPTIMIZING ENHANCED MODEL THRESHOLDS")
print("======================================")

threshold_results = []

for name, model in enhanced_models.items():

    probs = model.predict_proba(
        X_enh_test
    )[:, 1]

    for threshold in np.arange(
        0.25, 0.76, 0.01
    ):

        preds = (
            probs >= threshold
        ).astype(int)

        acc = accuracy_score(
            y_enh_test,
            preds
        )

        threshold_results.append({

            "Model": name,

            "Threshold":
                round(threshold, 2),

            "Accuracy":
                acc,

            "Correct":
                int(
                    acc * len(y_enh_test)
                ),

            "Team1_predictions":
                int(preds.sum()),

            "Team2_predictions":
                int(
                    len(preds) - preds.sum()
                )
        })


enhanced_threshold_df = pd.DataFrame(
    threshold_results
).sort_values(
    "Accuracy",
    ascending=False
)


print()
print("======================================")
print("TOP 20 THRESHOLDS")
print("======================================")

display(
    enhanced_threshold_df.head(20)
)


best_enhanced = (
    enhanced_threshold_df.iloc[0]
)

print()
print("======================================")
print("BEST ENHANCED THRESHOLD")
print("======================================")

print(
    "Model:",
    best_enhanced["Model"]
)

print(
    "Threshold:",
    best_enhanced["Threshold"]
)

print(
    "Accuracy:",
    f'{best_enhanced["Accuracy"] * 100:.2f}%'
)

print(
    "Correct:",
    f'{best_enhanced["Correct"]} / {len(y_enh_test)}'
)

print()
print(
    "Previous benchmark: 61.45%"
)

OPTIMIZING ENHANCED MODEL THRESHOLDS

TOP 20 THRESHOLDS


,Model,Threshold,Accuracy,Correct,Team1_predictions,Team2_predictions
144,XGBoost,0.67,0.574297,143,9,240
83,Random Forest,0.57,0.570281,142,6,243
84,Random Forest,0.58,0.570281,142,4,245
145,XGBoost,0.68,0.570281,142,8,241
143,XGBoost,0.66,0.570281,142,10,239
82,Random Forest,0.56,0.570281,142,10,239
85,Random Forest,0.59,0.566265,141,1,248
81,Random Forest,0.55,0.566265,141,15,234
146,XGBoost,0.69,0.566265,141,7,242
88,Random Forest,0.62,0.562249,140,0,249



BEST ENHANCED THRESHOLD
Model: XGBoost
Threshold: 0.67
Accuracy: 57.43%
Correct: 143 / 249

Previous benchmark: 61.45%


In [150]:
# ==========================================
# STEP 89 - TUNE TIME-AWARE LOGISTIC MODEL
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print("======================================")
print("TUNING TIME-AWARE LOGISTIC REGRESSION")
print("======================================")


# Use the ORIGINAL time-aware dataset
X_train = X_train_time.copy()
X_test = X_test_time.copy()

y_train = y_enh_train.copy()
y_test = y_enh_test.copy()


results_tuned = []
tuned_models = {}


for C in [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]:

    for class_weight in [
        None,
        "balanced"
    ]:

        model = Pipeline([

            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "scaler",
                StandardScaler()
            ),

            (
                "classifier",
                LogisticRegression(
                    C=C,
                    class_weight=class_weight,
                    max_iter=5000,
                    solver="liblinear",
                    random_state=42
                )
            )
        ])


        model.fit(
            X_train,
            y_train
        )


        probs = model.predict_proba(
            X_test
        )[:, 1]


        # Default threshold
        preds = (
            probs >= 0.50
        ).astype(int)


        results_tuned.append({

            "C": C,

            "class_weight":
                str(class_weight),

            "Accuracy":
                accuracy_score(
                    y_test,
                    preds
                ),

            "ROC-AUC":
                roc_auc_score(
                    y_test,
                    probs
                )
        })


        tuned_models[
            (C, str(class_weight))
        ] = model


tuned_results_df = pd.DataFrame(
    results_tuned
).sort_values(
    "Accuracy",
    ascending=False
)


print()
print("======================================")
print("TUNING RESULTS")
print("======================================")

display(
    tuned_results_df
)


print()
print("Previous best threshold accuracy: 61.45%")

TUNING TIME-AWARE LOGISTIC REGRESSION

TUNING RESULTS


,C,class_weight,Accuracy,ROC-AUC
3,0.003,balanced,0.461847,0.456291
0,0.001,None,0.457831,0.455963
1,0.001,balanced,0.457831,0.455898
2,0.003,None,0.457831,0.456225
5,0.010,balanced,0.457831,0.460550
4,0.010,None,0.449799,0.460485
6,0.030,None,0.445783,0.460550
8,0.100,None,0.445783,0.463368
9,0.100,balanced,0.441767,0.463303
10,0.300,None,0.441767,0.464810



Previous best threshold accuracy: 61.45%


In [151]:
# ============================================================
# STEP 90 - CLEAN PLAYER MATCHUP MODEL
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("======================================")
print("BUILDING CLEAN PLAYER MATCHUP MODEL")
print("======================================")


# ============================================================
# 1. PREPARE MATCHUP HISTORY
# ============================================================

hist = matchup_df[
    [
        "date",
        "batter",
        "bowler",
        "previous_balls",
        "previous_runs",
        "previous_dismissals",
        "previous_strike_rate",
        "last_5_runs",
        "last_5_balls",
        "last_5_strike_rate",
        "last_5_dismissals",
        "matchup_score"
    ]
].copy()

hist["date"] = pd.to_datetime(hist["date"])

# Sort so the LAST row is the latest historical record
hist = hist.sort_values(
    ["batter", "bowler", "date"]
)

print("Historical matchup rows:", len(hist))


# ============================================================
# 2. CREATE ALL BATTER-BOWLER COMBINATIONS
# ============================================================

rows = []

for _, match in custom_match_df.iterrows():

    match_id = match["match_id"]
    match_date = pd.to_datetime(match["date"])

    team1 = list(match["team1_players"])
    team2 = list(match["team2_players"])


    # --------------------------------------------------------
    # TEAM 1 BATTERS vs TEAM 2 BOWLERS
    # --------------------------------------------------------

    for batter in team1:

        for bowler in team2:

            rows.append({

                "match_id": match_id,
                "match_date": match_date,

                "side": "team1",

                "batter": batter,
                "bowler": bowler
            })


    # --------------------------------------------------------
    # TEAM 2 BATTERS vs TEAM 1 BOWLERS
    # --------------------------------------------------------

    for batter in team2:

        for bowler in team1:

            rows.append({

                "match_id": match_id,
                "match_date": match_date,

                "side": "team2",

                "batter": batter,
                "bowler": bowler
            })


pairs = pd.DataFrame(rows)

print("Total player combinations:", len(pairs))


# ============================================================
# 3. MERGE HISTORICAL MATCHUP DATA
# ============================================================

pairs = pairs.merge(

    hist,

    on=["batter", "bowler"],

    how="left",

    suffixes=("", "_history")
)


# ============================================================
# 4. REMOVE CURRENT/FUTURE INFORMATION
# ============================================================

# VERY IMPORTANT:
#
# Only matchup records BEFORE the current match
# are allowed.

future_or_current = (
    pairs["date"] >= pairs["match_date"]
)

stat_cols = [
    "previous_balls",
    "previous_runs",
    "previous_dismissals",
    "previous_strike_rate",
    "last_5_runs",
    "last_5_balls",
    "last_5_strike_rate",
    "last_5_dismissals",
    "matchup_score"
]

pairs.loc[
    future_or_current,
    stat_cols
] = np.nan


# ============================================================
# 5. KEEP MOST RECENT HISTORICAL RECORD
# ============================================================

pairs = pairs.sort_values(
    [
        "match_id",
        "side",
        "batter",
        "bowler",
        "date"
    ]
)

pairs = (
    pairs
    .groupby(
        [
            "match_id",
            "side",
            "batter",
            "bowler"
        ],
        as_index=False
    )
    .last()
)


# ============================================================
# 6. HISTORY FLAG
# ============================================================

pairs["has_history"] = (
    pairs["previous_balls"].notna()
).astype(int)


# ============================================================
# 7. FILL MISSING HISTORY
# ============================================================

pairs[stat_cols] = (
    pairs[stat_cols]
    .fillna(0)
)


# ============================================================
# 8. AGGREGATE EACH SIDE
# ============================================================

def aggregate_matchups(group):

    balls = group["previous_balls"].sum()
    runs = group["previous_runs"].sum()

    return pd.Series({

        # Overall matchup quality
        "matchup_score_mean":
            group["matchup_score"].mean(),

        "matchup_score_max":
            group["matchup_score"].max(),

        "matchup_score_std":
            group["matchup_score"].std(),

        # Historical performance
        "runs":
            runs,

        "balls":
            balls,

        "dismissals":
            group["previous_dismissals"].sum(),

        "strike_rate":
            (
                runs / balls * 100
                if balls > 0
                else 0
            ),

        # Recent matchup performance
        "last5_runs":
            group["last_5_runs"].mean(),

        "last5_strike_rate":
            group["last_5_strike_rate"].mean(),

        "last5_dismissals":
            group["last_5_dismissals"].mean(),

        # Amount of actual matchup history
        "history_count":
            group["has_history"].sum(),

        "history_ratio":
            group["has_history"].mean()
    })


aggregated = (
    pairs
    .groupby(
        ["match_id", "side"]
    )
    .apply(
        aggregate_matchups,
        include_groups=False
    )
    .reset_index()
)


# ============================================================
# 9. SEPARATE TEAM 1 / TEAM 2
# ============================================================

t1 = aggregated[
    aggregated["side"] == "team1"
].drop(
    columns="side"
)

t2 = aggregated[
    aggregated["side"] == "team2"
].drop(
    columns="side"
)


t1 = t1.rename(
    columns={
        c: "team1_" + c
        for c in t1.columns
        if c != "match_id"
    }
)

t2 = t2.rename(
    columns={
        c: "team2_" + c
        for c in t2.columns
        if c != "match_id"
    }
)


# ============================================================
# 10. BUILD FINAL MATCHUP DATASET
# ============================================================

matchup_model_df = (
    custom_match_df[
        [
            "match_id",
            "date"
        ]
    ]

    .merge(
        t1,
        on="match_id",
        how="left"
    )

    .merge(
        t2,
        on="match_id",
        how="left"
    )
)


# ============================================================
# 11. DIFFERENCE FEATURES
# ============================================================

base_features = [
    "matchup_score_mean",
    "matchup_score_max",
    "matchup_score_std",
    "runs",
    "balls",
    "dismissals",
    "strike_rate",
    "last5_runs",
    "last5_strike_rate",
    "last5_dismissals",
    "history_count",
    "history_ratio"
]


for feature in base_features:

    matchup_model_df[
        feature + "_diff"
    ] = (

        matchup_model_df[
            "team1_" + feature
        ]

        -

        matchup_model_df[
            "team2_" + feature
        ]
    )


# ============================================================
# 12. CLEAN
# ============================================================

matchup_model_df = (
    matchup_model_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)


# ============================================================
# 13. CHECK
# ============================================================

print()
print("======================================")
print("CLEAN MATCHUP DATASET")
print("======================================")

print(
    "Shape:",
    matchup_model_df.shape
)

print(
    "Matches:",
    matchup_model_df["match_id"].nunique()
)

print(
    "NaN:",
    matchup_model_df.isna().sum().sum()
)

print(
    "Infinite:",
    np.isinf(
        matchup_model_df.select_dtypes(
            include=np.number
        )
    ).sum().sum()
)


print()
print("======================================")
print("IMPORTANT MATCHUP FEATURES")
print("======================================")

display(
    matchup_model_df[
        [
            "match_id",

            "team1_matchup_score_mean",
            "team2_matchup_score_mean",

            "matchup_score_mean_diff",

            "team1_strike_rate",
            "team2_strike_rate",

            "strike_rate_diff",

            "team1_history_count",
            "team2_history_count",

            "history_count_diff"
        ]
    ].head(10)
)

BUILDING CLEAN PLAYER MATCHUP MODEL
Historical matchup rows: 61429
Total player combinations: 313736

CLEAN MATCHUP DATASET
Shape: (1243, 38)
Matches: 1243
NaN: 0
Infinite: 0

IMPORTANT MATCHUP FEATURES


,match_id,team1_matchup_score_mean,team2_matchup_score_mean,matchup_score_mean_diff,team1_strike_rate,team2_strike_rate,strike_rate_diff,team1_history_count,team2_history_count,history_count_diff
0,1082591,5.760602,5.206815,0.553787,129.777778,141.538462,-11.760684,29.0,29.0,0.0
1,1082592,3.778831,5.300545,-1.521714,147.857143,141.904762,5.952381,30.0,20.0,10.0
2,1082593,7.137565,6.969587,0.167978,136.549708,124.610592,11.939116,24.0,36.0,-12.0
3,1082594,1.615014,3.452639,-1.837625,133.333333,111.258278,22.075055,15.0,19.0,-4.0
4,1082595,4.745025,1.469172,3.275853,118.357488,110.714286,7.643202,23.0,23.0,0.0
5,1082596,7.050989,5.429941,1.621048,116.873449,109.152542,7.720907,30.0,33.0,-3.0
6,1082597,1.573868,9.940192,-8.366324,96.825397,120.272904,-23.447508,18.0,35.0,-17.0
7,1082598,2.574303,2.875002,-0.300699,138.961039,128.099174,10.861865,25.0,20.0,5.0
8,1082599,5.290176,0.951594,4.338582,134.934498,141.666667,-6.732169,24.0,23.0,1.0
9,1082600,3.741234,8.637833,-4.896598,129.064039,118.475751,10.588289,22.0,35.0,-13.0


In [152]:
# ============================================================
# FAST CLEAN MATCHUP MODEL
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# Target
y = custom_match_df["team1_win"].astype(int)

# Use ONLY the matchup difference features
matchup_features = [
    c for c in matchup_model_df.columns
    if c.endswith("_diff")
]

X = matchup_model_df[matchup_features].copy()

# Safety cleanup
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# Same chronological split
X_train = X.iloc[:994]
X_test  = X.iloc[994:]

y_train = y.iloc[:994]
y_test  = y.iloc[994:]

print("======================================")
print("TRAINING CLEAN MATCHUP MODEL")
print("======================================")

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=2000
    ))
])

model.fit(X_train, y_train)

prob = model.predict_proba(X_test)[:, 1]

# Default threshold
pred = (prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, prob)

print()
print("======================================")
print("MATCHUP MODEL RESULTS")
print("======================================")

print(f"Features : {len(matchup_features)}")
print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"ROC-AUC  : {auc:.4f}")


# ============================================================
# QUICK THRESHOLD SEARCH
# ============================================================

best_acc = 0
best_threshold = 0.5

for threshold in np.arange(0.30, 0.71, 0.01):

    p = (prob >= threshold).astype(int)

    acc = accuracy_score(y_test, p)

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

print()
print("======================================")
print("BEST MATCHUP THRESHOLD")
print("======================================")

print(f"Threshold : {best_threshold:.2f}")
print(f"Accuracy  : {best_acc*100:.2f}%")
print(f"Correct   : {int(best_acc * len(y_test))} / {len(y_test)}")

TRAINING CLEAN MATCHUP MODEL

MATCHUP MODEL RESULTS
Features : 12
Accuracy : 0.4940 (49.40%)
ROC-AUC  : 0.5145

BEST MATCHUP THRESHOLD
Threshold : 0.39
Accuracy  : 55.42%
Correct   : 138 / 249


In [153]:
# ============================================================
# FINAL MODEL: TIME-AWARE + PLAYER MATCHUP
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print("======================================")
print("BUILDING FINAL TIME + MATCHUP MODEL")
print("======================================")

# ------------------------------------------------------------
# 1. TIME-AWARE FEATURES
# ------------------------------------------------------------

time_features = [
    c for c in X_time.columns
]

X_time_clean = (
    X_time[time_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# ------------------------------------------------------------
# 2. MATCHUP FEATURES
# ------------------------------------------------------------

matchup_features = [
    c for c in matchup_model_df.columns
    if c.endswith("_diff")
]

X_matchup_clean = (
    matchup_model_df[matchup_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# ------------------------------------------------------------
# 3. COMBINE
# ------------------------------------------------------------

X_final = pd.concat(
    [
        X_time_clean.reset_index(drop=True),
        X_matchup_clean.reset_index(drop=True)
    ],
    axis=1
)

# Remove duplicate columns if any
X_final = X_final.loc[
    :,
    ~X_final.columns.duplicated()
]

print("Time features   :", len(time_features))
print("Matchup features:", len(matchup_features))
print("Final features  :", X_final.shape[1])

# ------------------------------------------------------------
# 4. CHRONOLOGICAL SPLIT
# ------------------------------------------------------------

X_train_final = X_final.iloc[:994]
X_test_final  = X_final.iloc[994:]

y_train_final = y.iloc[:994]
y_test_final  = y.iloc[994:]

# ------------------------------------------------------------
# 5. MODEL
# ------------------------------------------------------------

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=3000
    ))
])

final_model.fit(
    X_train_final,
    y_train_final
)

prob_final = final_model.predict_proba(
    X_test_final
)[:, 1]

# ------------------------------------------------------------
# 6. DEFAULT
# ------------------------------------------------------------

pred_default = (
    prob_final >= 0.50
).astype(int)

default_acc = accuracy_score(
    y_test_final,
    pred_default
)

auc_final = roc_auc_score(
    y_test_final,
    prob_final
)

# ------------------------------------------------------------
# 7. THRESHOLD SEARCH
# ------------------------------------------------------------

best_acc = 0
best_threshold = 0.50

for threshold in np.arange(
    0.25,
    0.76,
    0.01
):

    pred = (
        prob_final >= threshold
    ).astype(int)

    acc = accuracy_score(
        y_test_final,
        pred
    )

    if acc > best_acc:

        best_acc = acc
        best_threshold = threshold

# ------------------------------------------------------------
# 8. FINAL RESULT
# ------------------------------------------------------------

print()
print("======================================")
print("FINAL MODEL RESULTS")
print("======================================")

print(
    f"Default Accuracy : {default_acc*100:.2f}%"
)

print(
    f"ROC-AUC          : {auc_final:.4f}"
)

print()
print("======================================")
print("BEST THRESHOLD")
print("======================================")

print(
    f"Threshold : {best_threshold:.2f}"
)

print(
    f"Accuracy  : {best_acc*100:.2f}%"
)

print(
    f"Correct   : {int(best_acc * len(y_test_final))} / {len(y_test_final)}"
)

print()
print("Previous best : 61.45%")
print(
    f"Improvement   : {(best_acc - 0.6145)*100:+.2f}%"
)

BUILDING FINAL TIME + MATCHUP MODEL
Time features   : 50
Matchup features: 12
Final features  : 62

FINAL MODEL RESULTS
Default Accuracy : 54.62%
ROC-AUC          : 0.5999

BEST THRESHOLD
Threshold : 0.39
Accuracy  : 59.04%
Correct   : 147 / 249

Previous best : 61.45%
Improvement   : -2.41%


In [154]:
# ============================================================
# RECENCY-WEIGHTED TIME-AWARE LOGISTIC REGRESSION
# ============================================================

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print("======================================")
print("RECENCY-WEIGHTED TIME-AWARE MODEL")
print("======================================")

# Use the existing chronological split
Xtr = X_train_time.copy()
Xte = X_test_time.copy()

ytr = y.iloc[:len(Xtr)].copy()
yte = y.iloc[len(Xtr):len(Xtr) + len(Xte)].copy()

# ------------------------------------------------------------
# RECENCY WEIGHTS
# ------------------------------------------------------------
# Older matches get lower weight.
# Newer matches get higher weight.

n = len(Xtr)

weights = np.linspace(
    0.5,       # oldest training match weight
    1.5,       # newest training match weight
    n
)

# ------------------------------------------------------------
# TRY DIFFERENT WEIGHT STRENGTHS
# ------------------------------------------------------------

best_acc = 0
best_auc = 0
best_strength = None
best_threshold = 0.50

results = []

for strength in [0.0, 0.25, 0.5, 0.75, 1.0]:

    if strength == 0:
        sample_weight = np.ones(n)
    else:
        sample_weight = np.linspace(
            1.0 - strength,
            1.0 + strength,
            n
        )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=0.1,
            max_iter=3000
        ))
    ])

    model.fit(
        Xtr,
        ytr,
        classifier__sample_weight=sample_weight
    )

    probabilities = model.predict_proba(Xte)[:, 1]

    auc = roc_auc_score(
        yte,
        probabilities
    )

    # Threshold optimization
    local_best_acc = 0
    local_threshold = 0.50

    for threshold in np.arange(0.30, 0.71, 0.01):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        acc = accuracy_score(
            yte,
            predictions
        )

        if acc > local_best_acc:
            local_best_acc = acc
            local_threshold = threshold

    results.append([
        strength,
        local_best_acc,
        auc,
        local_threshold
    ])

    if local_best_acc > best_acc:
        best_acc = local_best_acc
        best_auc = auc
        best_strength = strength
        best_threshold = local_threshold


print()
print("======================================")
print("RECENCY WEIGHTING RESULTS")
print("======================================")

for r in results:
    print(
        f"Strength={r[0]:.2f} | "
        f"Accuracy={r[1]*100:.2f}% | "
        f"ROC-AUC={r[2]:.4f} | "
        f"Threshold={r[3]:.2f}"
    )

print()
print("======================================")
print("BEST RECENCY MODEL")
print("======================================")

print(
    f"Best strength : {best_strength:.2f}"
)

print(
    f"Best threshold: {best_threshold:.2f}"
)

print(
    f"Accuracy      : {best_acc*100:.2f}%"
)

print(
    f"ROC-AUC       : {best_auc:.4f}"
)

print(
    f"Correct       : {round(best_acc * len(yte))} / {len(yte)}"
)

print()
print("Previous best : 61.45%")
print(
    f"Improvement   : {(best_acc - 0.6145)*100:+.2f}%"
)

RECENCY-WEIGHTED TIME-AWARE MODEL

RECENCY WEIGHTING RESULTS
Strength=0.00 | Accuracy=61.45% | ROC-AUC=0.6007 | Threshold=0.43
Strength=0.25 | Accuracy=61.45% | ROC-AUC=0.6089 | Threshold=0.43
Strength=0.50 | Accuracy=62.65% | ROC-AUC=0.6182 | Threshold=0.44
Strength=0.75 | Accuracy=62.25% | ROC-AUC=0.6230 | Threshold=0.44
Strength=1.00 | Accuracy=61.04% | ROC-AUC=0.6240 | Threshold=0.44

BEST RECENCY MODEL
Best strength : 0.50
Best threshold: 0.44
Accuracy      : 62.65%
ROC-AUC       : 0.6182
Correct       : 156 / 249

Previous best : 61.45%
Improvement   : +1.20%


In [155]:
# ============================================================
# STEP: ADD LEAKAGE-FREE RECENT WIN RATE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print("======================================")
print("ADDING RECENT WIN-RATE FEATURES")
print("======================================")

# ------------------------------------------------------------
# 1. BUILD MATCH RESULT HISTORY
# ------------------------------------------------------------

matches = match_info_df.copy()

matches["date"] = pd.to_datetime(matches["date"])

# Team-level match results
team_rows = []

for _, r in matches.sort_values("date").iterrows():

    team_rows.append({
        "date": r["date"],
        "team": r["team_1"],
        "win": int(r["winner"] == r["team_1"])
    })

    team_rows.append({
        "date": r["date"],
        "team": r["team_2"],
        "win": int(r["winner"] == r["team_2"])
    })

team_results = pd.DataFrame(team_rows)

team_results = team_results.sort_values(
    ["team", "date"]
)

# ------------------------------------------------------------
# 2. STRICT PRE-MATCH ROLLING WIN RATE
# ------------------------------------------------------------

team_results["previous_matches"] = (
    team_results
    .groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(
        5,
        min_periods=1
    ).count())
)

team_results["recent_wins"] = (
    team_results
    .groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(
        5,
        min_periods=1
    ).sum())
)

team_results["recent_win_rate"] = (
    team_results["recent_wins"] /
    team_results["previous_matches"]
)

# No previous matches = neutral 0.5
team_results["recent_win_rate"] = (
    team_results["recent_win_rate"]
    .fillna(0.5)
)

# ------------------------------------------------------------
# 3. CREATE MATCH-LEVEL FEATURES
# ------------------------------------------------------------

match_sorted = matches.sort_values("date").copy()

t1_winrate = team_results.rename(
    columns={
        "team": "team_1",
        "recent_win_rate": "team1_recent_win_rate"
    }
)[
    ["date", "team_1", "team1_recent_win_rate"]
]

t2_winrate = team_results.rename(
    columns={
        "team": "team_2",
        "recent_win_rate": "team2_recent_win_rate"
    }
)[
    ["date", "team_2", "team2_recent_win_rate"]
]

winrate_df = match_sorted[
    ["match_id", "date", "team_1", "team_2"]
].copy()

# Merge by team/date.
# Because the rolling value is shifted, it is PRE-MATCH.
winrate_df = winrate_df.merge(
    t1_winrate,
    on=["date", "team_1"],
    how="left"
)

winrate_df = winrate_df.merge(
    t2_winrate,
    on=["date", "team_2"],
    how="left"
)

winrate_df["recent_win_rate_diff"] = (
    winrate_df["team1_recent_win_rate"]
    -
    winrate_df["team2_recent_win_rate"]
)

winrate_df = winrate_df.fillna(0.5)

# ------------------------------------------------------------
# 4. ALIGN WITH TIME-AWARE DATA
# ------------------------------------------------------------

base = final_time_team_df.copy()

base["match_id"] = base["match_id"].astype(
    winrate_df["match_id"].dtype
)

combined = base.merge(
    winrate_df[
        [
            "match_id",
            "team1_recent_win_rate",
            "team2_recent_win_rate",
            "recent_win_rate_diff"
        ]
    ],
    on="match_id",
    how="left"
)

combined = combined.sort_values("match_id")

# ------------------------------------------------------------
# 5. FEATURES
# ------------------------------------------------------------

target_col = "team1_win"

exclude = [
    "match_id",
    "date",
    target_col
]

feature_cols = [
    c for c in combined.columns
    if c not in exclude
]

X_winrate = (
    combined[feature_cols]
    .select_dtypes(include=np.number)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y_winrate = combined[target_col].astype(int)

# Chronological split
X_train_wr = X_winrate.iloc[:994]
X_test_wr = X_winrate.iloc[994:]

y_train_wr = y_winrate.iloc[:994]
y_test_wr = y_winrate.iloc[994:]

# ------------------------------------------------------------
# 6. RECENCY WEIGHT = 0.50
# ------------------------------------------------------------

weights = np.linspace(
    0.5,
    1.5,
    len(X_train_wr)
)

model_wr = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=3000
    ))
])

model_wr.fit(
    X_train_wr,
    y_train_wr,
    classifier__sample_weight=weights
)

prob_wr = model_wr.predict_proba(
    X_test_wr
)[:, 1]

auc_wr = roc_auc_score(
    y_test_wr,
    prob_wr
)

# ------------------------------------------------------------
# 7. THRESHOLD SEARCH
# ------------------------------------------------------------

best_acc = 0
best_threshold = 0.5

for threshold in np.arange(
    0.30,
    0.71,
    0.01
):

    pred = (
        prob_wr >= threshold
    ).astype(int)

    acc = accuracy_score(
        y_test_wr,
        pred
    )

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

# ------------------------------------------------------------
# 8. RESULT
# ------------------------------------------------------------

print()
print("======================================")
print("RECENT WIN-RATE RESULTS")
print("======================================")

print(
    "Features:",
    X_winrate.shape[1]
)

print(
    f"ROC-AUC: {auc_wr:.4f}"
)

print(
    f"Best threshold: {best_threshold:.2f}"
)

print(
    f"Accuracy: {best_acc*100:.2f}%"
)

print(
    f"Correct: {int(best_acc * len(y_test_wr))} / {len(y_test_wr)}"
)

print()
print("Previous best: 62.65%")

print(
    f"Improvement: {(best_acc - 0.6265)*100:+.2f}%"
)

ADDING RECENT WIN-RATE FEATURES

RECENT WIN-RATE RESULTS
Features: 53
ROC-AUC: 0.6048
Best threshold: 0.47
Accuracy: 59.44%
Correct: 148 / 249

Previous best: 62.65%
Improvement: -3.21%


In [156]:
# ============================================================
# STEP: LEAKAGE-FREE VENUE FEATURES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print("======================================")
print("BUILDING LEAKAGE-FREE VENUE FEATURES")
print("======================================")

# ------------------------------------------------------------
# 1. PREPARE MATCH DATA
# ------------------------------------------------------------

m = match_info_df.copy()
m["date"] = pd.to_datetime(m["date"])

m = m.sort_values("date").reset_index(drop=True)

# Team 1 win
m["team1_win"] = (
    m["winner"] == m["team_1"]
).astype(int)

# ------------------------------------------------------------
# 2. VENUE HISTORICAL WIN RATE
# ------------------------------------------------------------

# IMPORTANT:
# shift(1) means current match is NOT included.

m["venue_previous_matches"] = (
    m.groupby("venue")
     .cumcount()
)

m["venue_previous_wins"] = (
    m.groupby("venue")["team1_win"]
     .transform(
         lambda x: x.shift(1).fillna(0).cumsum()
     )
)

# Historical Team-1 win rate at venue
m["venue_team1_win_rate"] = np.where(
    m["venue_previous_matches"] > 0,
    m["venue_previous_wins"] /
    m["venue_previous_matches"],
    0.5
)

# ------------------------------------------------------------
# 3. VENUE EXPERIENCE / RELIABILITY
# ------------------------------------------------------------

# More matches = more reliable venue estimate
m["venue_reliability"] = (
    m["venue_previous_matches"] /
    (
        m["venue_previous_matches"] + 10
    )
)

# How different venue is from neutral 50%
m["venue_advantage"] = (
    m["venue_team1_win_rate"] - 0.5
)

# ------------------------------------------------------------
# 4. RECENT VENUE FORM
# ------------------------------------------------------------

m["venue_recent_win_rate"] = (
    m.groupby("venue")["team1_win"]
     .transform(
         lambda x:
         x.shift(1)
          .rolling(10, min_periods=1)
          .mean()
     )
     .fillna(0.5)
)

m["venue_recent_advantage"] = (
    m["venue_recent_win_rate"] - 0.5
)

# ------------------------------------------------------------
# 5. SAVE ONLY PRE-MATCH FEATURES
# ------------------------------------------------------------

venue_features = m[
    [
        "match_id",
        "venue_team1_win_rate",
        "venue_reliability",
        "venue_advantage",
        "venue_recent_win_rate",
        "venue_recent_advantage"
    ]
].copy()

print()
print("Venue features created:", venue_features.shape)

# ------------------------------------------------------------
# 6. MERGE WITH TIME-AWARE DATA
# ------------------------------------------------------------

base = final_time_team_df.copy()

combined_venue = base.merge(
    venue_features,
    on="match_id",
    how="left"
)

combined_venue = (
    combined_venue
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# ------------------------------------------------------------
# 7. BUILD FEATURES
# ------------------------------------------------------------

exclude = [
    "match_id",
    "date",
    "team1_win"
]

feature_cols = [
    c for c in combined_venue.columns
    if c not in exclude
]

X_venue = (
    combined_venue[feature_cols]
    .select_dtypes(include=np.number)
)

y_venue = combined_venue["team1_win"].astype(int)

# Same chronological split
X_train_venue = X_venue.iloc[:994]
X_test_venue = X_venue.iloc[994:]

y_train_venue = y_venue.iloc[:994]
y_test_venue = y_venue.iloc[994:]

# ------------------------------------------------------------
# 8. RECENCY WEIGHTING = 0.50
# ------------------------------------------------------------

weights = np.linspace(
    0.5,
    1.5,
    len(X_train_venue)
)

model_venue = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=3000
    ))
])

model_venue.fit(
    X_train_venue,
    y_train_venue,
    classifier__sample_weight=weights
)

prob_venue = model_venue.predict_proba(
    X_test_venue
)[:, 1]

auc_venue = roc_auc_score(
    y_test_venue,
    prob_venue
)

# ------------------------------------------------------------
# 9. THRESHOLD SEARCH
# ------------------------------------------------------------

best_acc = 0
best_threshold = 0.50

for threshold in np.arange(
    0.30,
    0.71,
    0.01
):

    pred = (
        prob_venue >= threshold
    ).astype(int)

    acc = accuracy_score(
        y_test_venue,
        pred
    )

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

# ------------------------------------------------------------
# 10. RESULT
# ------------------------------------------------------------

print()
print("======================================")
print("VENUE MODEL RESULTS")
print("======================================")

print(
    f"ROC-AUC: {auc_venue:.4f}"
)

print(
    f"Best threshold: {best_threshold:.2f}"
)

print(
    f"Accuracy: {best_acc*100:.2f}%"
)

print(
    f"Correct: {int(best_acc * len(y_test_venue))} / {len(y_test_venue)}"
)

print()
print("Previous best: 62.65%")

print(
    f"Improvement: {(best_acc - 0.6265)*100:+.2f}%"
)

BUILDING LEAKAGE-FREE VENUE FEATURES

Venue features created: (1243, 6)

VENUE MODEL RESULTS
ROC-AUC: 0.5877
Best threshold: 0.46
Accuracy: 59.84%
Correct: 149 / 249

Previous best: 62.65%
Improvement: -2.81%


In [157]:
# ============================================================
# FINAL LOGISTIC REGRESSION TUNING
# SAME 50 FEATURES + RECENCY WEIGHT 0.50
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

print("======================================")
print("TUNING BEST TIME-AWARE MODEL")
print("======================================")

# ------------------------------------------------------------
# USE EXACT SAME DATA THAT GAVE 62.65%
# ------------------------------------------------------------

Xtr = X_train_time.copy()
Xte = X_test_time.copy()

ytr = y.iloc[:len(Xtr)].copy()
yte = y.iloc[len(Xtr):len(Xtr) + len(Xte)].copy()

# Clean
Xtr = Xtr.replace([np.inf, -np.inf], np.nan).fillna(0)
Xte = Xte.replace([np.inf, -np.inf], np.nan).fillna(0)

# EXACT successful recency weighting
weights = np.linspace(
    0.5,
    1.5,
    len(Xtr)
)

# ------------------------------------------------------------
# C VALUES
# ------------------------------------------------------------

C_values = [
    0.0001,
    0.0003,
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]

results = []

best_accuracy = 0
best_auc = 0
best_C = None
best_threshold = None

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

for C in C_values:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=C,
            max_iter=5000,
            solver="liblinear"
        ))
    ])

    model.fit(
        Xtr,
        ytr,
        classifier__sample_weight=weights
    )

    probability = model.predict_proba(Xte)[:, 1]

    auc = roc_auc_score(
        yte,
        probability
    )

    local_best_accuracy = 0
    local_best_threshold = 0.50

    # Same threshold search
    for threshold in np.arange(
        0.30,
        0.71,
        0.01
    ):

        prediction = (
            probability >= threshold
        ).astype(int)

        accuracy = accuracy_score(
            yte,
            prediction
        )

        if accuracy > local_best_accuracy:

            local_best_accuracy = accuracy
            local_best_threshold = threshold

    results.append({
        "C": C,
        "Accuracy": local_best_accuracy,
        "ROC-AUC": auc,
        "Threshold": local_best_threshold
    })

    if local_best_accuracy > best_accuracy:

        best_accuracy = local_best_accuracy
        best_auc = auc
        best_C = C
        best_threshold = local_best_threshold


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)

print()
print("======================================")
print("LOGISTIC TUNING RESULTS")
print("======================================")

print(
    results_df.to_string(index=False)
)

# ------------------------------------------------------------
# BEST
# ------------------------------------------------------------

correct = round(
    best_accuracy * len(yte)
)

print()
print("======================================")
print("BEST TUNED MODEL")
print("======================================")

print(f"C              : {best_C}")
print(f"Threshold      : {best_threshold:.2f}")
print(f"Accuracy       : {best_accuracy*100:.2f}%")
print(f"Correct        : {correct} / {len(yte)}")
print(f"ROC-AUC        : {best_auc:.4f}")

print()
print("Previous best  : 62.65%")
print(
    f"Improvement    : {(best_accuracy - 0.6265)*100:+.2f}%"
)

TUNING BEST TIME-AWARE MODEL

LOGISTIC TUNING RESULTS
      C  Accuracy  ROC-AUC  Threshold
 1.0000  0.630522 0.619014       0.43
 0.3000  0.630522 0.617647       0.44
 0.1000  0.626506 0.617647       0.44
10.0000  0.626506 0.619209       0.43
 3.0000  0.626506 0.619274       0.43
 0.0300  0.598394 0.612897       0.45
 0.0100  0.594378 0.604698       0.47
 0.0030  0.586345 0.597215       0.48
 0.0003  0.578313 0.572358       0.48
 0.0010  0.574297 0.584526       0.46
 0.0001  0.558233 0.570731       0.49

BEST TUNED MODEL
C              : 0.3
Threshold      : 0.44
Accuracy       : 63.05%
Correct        : 157 / 249
ROC-AUC        : 0.6176

Previous best  : 62.65%
Improvement    : +0.40%


In [158]:
# ============================================================
# ENSEMBLE: LOGISTIC + RANDOM FOREST + XGBOOST
# Uses the SAME time-aware features
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

print("======================================")
print("TRAINING FINAL ENSEMBLE")
print("======================================")

Xtr = X_train_time.copy()
Xte = X_test_time.copy()

ytr = y.iloc[:len(Xtr)].copy()
yte = y.iloc[len(Xtr):len(Xtr)+len(Xte)].copy()

Xtr = Xtr.replace([np.inf, -np.inf], np.nan).fillna(0)
Xte = Xte.replace([np.inf, -np.inf], np.nan).fillna(0)

# Recency weighting that worked best
weights = np.linspace(0.5, 1.5, len(Xtr))

# ------------------------------------------------------------
# 1. LOGISTIC
# ------------------------------------------------------------

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=5000
    ))
])

lr.fit(
    Xtr,
    ytr,
    classifier__sample_weight=weights
)

p_lr = lr.predict_proba(Xte)[:, 1]

# ------------------------------------------------------------
# 2. RANDOM FOREST
# ------------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_leaf=8,
    random_state=42,
    class_weight=None
)

rf.fit(
    Xtr,
    ytr,
    sample_weight=weights
)

p_rf = rf.predict_proba(Xte)[:, 1]

# ------------------------------------------------------------
# 3. XGBOOST
# ------------------------------------------------------------

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    reg_alpha=0.5,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(
    Xtr,
    ytr,
    sample_weight=weights
)

p_xgb = xgb.predict_proba(Xte)[:, 1]

# ------------------------------------------------------------
# 4. TEST DIFFERENT ENSEMBLE WEIGHTS
# ------------------------------------------------------------

best_acc = 0
best_config = None
best_threshold = 0.5
best_auc = 0

configs = [
    (1.0, 0.0, 0.0),
    (0.8, 0.2, 0.0),
    (0.7, 0.3, 0.0),
    (0.7, 0.0, 0.3),
    (0.6, 0.2, 0.2),
    (0.6, 0.3, 0.1),
    (0.5, 0.3, 0.2),
    (0.5, 0.2, 0.3),
    (0.4, 0.3, 0.3)
]

for w_lr, w_rf, w_xgb in configs:

    probability = (
        w_lr * p_lr +
        w_rf * p_rf +
        w_xgb * p_xgb
    )

    auc = roc_auc_score(
        yte,
        probability
    )

    for threshold in np.arange(
        0.30,
        0.71,
        0.01
    ):

        pred = (
            probability >= threshold
        ).astype(int)

        acc = accuracy_score(
            yte,
            pred
        )

        if acc > best_acc:

            best_acc = acc
            best_config = (
                w_lr,
                w_rf,
                w_xgb
            )
            best_threshold = threshold
            best_auc = auc

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print()
print("======================================")
print("FINAL ENSEMBLE RESULT")
print("======================================")

print(
    f"LR weight      : {best_config[0]}"
)

print(
    f"RF weight      : {best_config[1]}"
)

print(
    f"XGB weight     : {best_config[2]}"
)

print(
    f"Threshold      : {best_threshold:.2f}"
)

print(
    f"Accuracy       : {best_acc*100:.2f}%"
)

print(
    f"Correct        : {round(best_acc * len(yte))} / {len(yte)}"
)

print(
    f"ROC-AUC        : {best_auc:.4f}"
)

print()
print("Current best: 63.05%")
print(
    f"Improvement: {(best_acc - 0.6305)*100:+.2f}%"
)

TRAINING FINAL ENSEMBLE

FINAL ENSEMBLE RESULT
LR weight      : 1.0
RF weight      : 0.0
XGB weight     : 0.0
Threshold      : 0.43
Accuracy       : 63.05%
Correct        : 157 / 249
ROC-AUC        : 0.6189

Current best: 63.05%
Improvement: +0.00%


In [160]:
# ============================================================
# FIXED FAST 2-WAY PLAYER MATCHUP FEATURES
# ============================================================

import numpy as np
import pandas as pd

print("======================================")
print("BUILDING FAST 2-WAY MATCHUP FEATURES")
print("======================================")

# ------------------------------------------------------------
# 1. PREPARE MATCHUP HISTORY
# ------------------------------------------------------------

m = matchup_df.copy()

m["date"] = pd.to_datetime(m["date"])

# Aggregate historical batter vs bowler performance
pair_stats = (
    m.groupby(["batter", "bowler"], as_index=False)
     .agg(
         balls=("balls", "sum"),
         runs=("runs", "sum"),
         dismissals=("dismissals", "sum")
     )
)

pair_stats["strike_rate"] = np.where(
    pair_stats["balls"] > 0,
    pair_stats["runs"] / pair_stats["balls"] * 100,
    0
)

# Reliability: more balls = more trustworthy
pair_stats["reliability"] = np.minimum(
    pair_stats["balls"] / 30.0,
    1.0
)

# Fast dictionary
pair_lookup = {}

for r in pair_stats.itertuples(index=False):

    batter = str(r.batter)
    bowler = str(r.bowler)

    pair_lookup[(batter, bowler)] = (
        float(r.runs),
        float(r.balls),
        float(r.dismissals),
        float(r.strike_rate),
        float(r.reliability)
    )

print(
    "Historical player combinations:",
    len(pair_lookup)
)

# ------------------------------------------------------------
# 2. BUILD PLAYER LOOKUP
# ------------------------------------------------------------

xi = playing_xi.copy()

xi["player"] = xi["player"].astype(str)
xi["team"] = xi["team"].astype(str)

players_lookup = (
    xi.groupby(
        ["match_id", "team"]
    )["player"]
    .apply(list)
    .to_dict()
)

# ------------------------------------------------------------
# 3. IMPORTANT:
#    FLATTEN ANY NESTED LISTS
# ------------------------------------------------------------

def flatten_players(players):

    result = []

    if players is None:
        return result

    for p in players:

        if isinstance(p, (list, tuple, np.ndarray)):
            for x in p:
                if x is not None:
                    result.append(str(x))
        else:
            result.append(str(p))

    return result


# ------------------------------------------------------------
# 4. MATCHUP CALCULATION
# ------------------------------------------------------------

def matchup_score(
    batters,
    bowlers
):

    batters = flatten_players(batters)
    bowlers = flatten_players(bowlers)

    total_runs = 0
    total_balls = 0
    total_dismissals = 0

    weighted_sr = 0
    total_weight = 0

    history_count = 0

    for batter in batters:

        for bowler in bowlers:

            key = (
                batter,
                bowler
            )

            stats = pair_lookup.get(key)

            if stats is None:
                continue

            runs, balls, dismissals, sr, reliability = stats

            if balls <= 0:
                continue

            history_count += 1

            total_runs += runs
            total_balls += balls
            total_dismissals += dismissals

            weighted_sr += (
                sr * reliability
            )

            total_weight += reliability

    # No historical matchup
    if total_balls == 0:

        return {
            "score": 0.0,
            "strike_rate": 0.0,
            "history_count": 0,
            "runs": 0.0,
            "balls": 0.0,
            "dismissals": 0.0
        }

    overall_sr = (
        total_runs /
        total_balls *
        100
    )

    avg_sr = (
        weighted_sr / total_weight
        if total_weight > 0
        else overall_sr
    )

    # Matchup score
    score = (
        0.55 * avg_sr
        +
        0.25 * overall_sr
        -
        8.0 * total_dismissals
    )

    return {
        "score": score,
        "strike_rate": avg_sr,
        "history_count": history_count,
        "runs": total_runs,
        "balls": total_balls,
        "dismissals": total_dismissals
    }


# ------------------------------------------------------------
# 5. BUILD MATCH FEATURES
# ------------------------------------------------------------

records = []

for row in match_info_df.itertuples(index=False):

    match_id = row.match_id

    team1 = str(row.team_1)
    team2 = str(row.team_2)

    team1_players = players_lookup.get(
        (match_id, team1),
        []
    )

    team2_players = players_lookup.get(
        (match_id, team2),
        []
    )

    # --------------------------------------------------------
    # CORRECT DIRECTION 1
    # TEAM 1 BATTERS vs TEAM 2 BOWLERS
    # --------------------------------------------------------

    t1 = matchup_score(
        team1_players,
        team2_players
    )

    # --------------------------------------------------------
    # CORRECT DIRECTION 2
    # TEAM 2 BATTERS vs TEAM 1 BOWLERS
    # --------------------------------------------------------

    t2 = matchup_score(
        team2_players,
        team1_players
    )

    records.append({

        "match_id": match_id,

        "team1_matchup_score":
            t1["score"],

        "team2_matchup_score":
            t2["score"],

        "matchup_score_diff":
            t1["score"] -
            t2["score"],

        "team1_matchup_sr":
            t1["strike_rate"],

        "team2_matchup_sr":
            t2["strike_rate"],

        "matchup_sr_diff":
            t1["strike_rate"] -
            t2["strike_rate"],

        "team1_matchup_history":
            t1["history_count"],

        "team2_matchup_history":
            t2["history_count"],

        "matchup_history_diff":
            t1["history_count"] -
            t2["history_count"]
    })


matchup_2way_df = pd.DataFrame(records)

print()
print("======================================")
print("2-WAY MATCHUP DATASET CREATED")
print("======================================")

print(
    "Shape:",
    matchup_2way_df.shape
)

print(
    "Features:",
    matchup_2way_df.columns.tolist()
)

print()
print(
    matchup_2way_df.head()
)

BUILDING FAST 2-WAY MATCHUP FEATURES
Historical player combinations: 31370

2-WAY MATCHUP DATASET CREATED
Shape: (1243, 10)
Features: ['match_id', 'team1_matchup_score', 'team2_matchup_score', 'matchup_score_diff', 'team1_matchup_sr', 'team2_matchup_sr', 'matchup_sr_diff', 'team1_matchup_history', 'team2_matchup_history', 'matchup_history_diff']

  match_id  team1_matchup_score  team2_matchup_score  matchup_score_diff  \
0  1082591                  0.0                  0.0                 0.0   
1  1082592                  0.0                  0.0                 0.0   
2  1082593                  0.0                  0.0                 0.0   
3  1082594                  0.0                  0.0                 0.0   
4  1082595                  0.0                  0.0                 0.0   

   team1_matchup_sr  team2_matchup_sr  matchup_sr_diff  team1_matchup_history  \
0               0.0               0.0              0.0                      0   
1               0.0             

In [161]:
# ============================================================
# LEAKAGE-FREE 2-WAY PLAYER MATCHUP FEATURES
# ============================================================

import numpy as np
import pandas as pd
from collections import defaultdict

print("======================================")
print("BUILDING LEAKAGE-FREE MATCHUP FEATURES")
print("======================================")

# ------------------------------------------------------------
# 1. PREPARE MATCHUP DATA
# ------------------------------------------------------------

m = matchup_df.copy()

m["date"] = pd.to_datetime(m["date"])

# Make names consistent
m["batter"] = m["batter"].astype(str).str.strip()
m["bowler"] = m["bowler"].astype(str).str.strip()

m = m.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. PREPARE PLAYING XI
# ------------------------------------------------------------

xi = playing_xi.copy()

xi["player"] = (
    xi["player"]
    .astype(str)
    .str.strip()
)

xi["team"] = (
    xi["team"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 3. MATCH -> TEAM -> PLAYERS
# ------------------------------------------------------------

match_team_players = (
    xi.groupby(
        ["match_id", "team"]
    )["player"]
    .apply(list)
    .to_dict()
)

# ------------------------------------------------------------
# 4. MATCH INFORMATION
# ------------------------------------------------------------

matches = match_info_df.copy()

matches["date"] = pd.to_datetime(
    matches["date"]
)

matches["team_1"] = (
    matches["team_1"]
    .astype(str)
    .str.strip()
)

matches["team_2"] = (
    matches["team_2"]
    .astype(str)
    .str.strip()
)

matches = matches.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. HISTORICAL PAIR STATISTICS
#
# IMPORTANT:
# This dictionary contains ONLY matches already processed.
# Therefore no future leakage.
# ------------------------------------------------------------

pair_stats = defaultdict(
    lambda: {
        "runs": 0.0,
        "balls": 0.0,
        "dismissals": 0.0
    }
)

# ------------------------------------------------------------
# 6. MATCHUP SCORE FUNCTION
# ------------------------------------------------------------

def calculate_matchup(
    batters,
    bowlers
):

    total_runs = 0.0
    total_balls = 0.0
    total_dismissals = 0.0

    weighted_sr = 0.0
    total_weight = 0.0

    history_pairs = 0

    for batter in batters:

        for bowler in bowlers:

            key = (
                batter,
                bowler
            )

            stats = pair_stats.get(key)

            if stats is None:
                continue

            balls = stats["balls"]

            if balls <= 0:
                continue

            runs = stats["runs"]
            dismissals = stats["dismissals"]

            history_pairs += 1

            sr = (
                runs / balls * 100
            )

            # Reliability grows with sample size
            reliability = min(
                balls / 30.0,
                1.0
            )

            total_runs += runs
            total_balls += balls
            total_dismissals += dismissals

            weighted_sr += (
                sr * reliability
            )

            total_weight += reliability

    if total_balls == 0:

        return {
            "score": 0.0,
            "sr": 0.0,
            "balls": 0.0,
            "dismissals": 0.0,
            "history": 0
        }

    overall_sr = (
        total_runs /
        total_balls *
        100
    )

    weighted_sr_value = (
        weighted_sr / total_weight
        if total_weight > 0
        else overall_sr
    )

    # Moderate scoring formula
    score = (
        0.65 * weighted_sr_value
        +
        0.35 * overall_sr
        -
        5.0 * total_dismissals
    )

    return {
        "score": score,
        "sr": weighted_sr_value,
        "balls": total_balls,
        "dismissals": total_dismissals,
        "history": history_pairs
    }


# ------------------------------------------------------------
# 7. PROCESS MATCHES CHRONOLOGICALLY
# ------------------------------------------------------------

records = []

matches_with_history = 0

for row in matches.itertuples(index=False):

    match_id = row.match_id

    team1 = row.team_1
    team2 = row.team_2

    # ----------------------------------------
    # Get actual XI
    # ----------------------------------------

    t1_players = match_team_players.get(
        (match_id, team1),
        []
    )

    t2_players = match_team_players.get(
        (match_id, team2),
        []
    )

    # ----------------------------------------
    # T1 BATTERS vs T2 BOWLERS
    # ----------------------------------------

    t1_matchup = calculate_matchup(
        t1_players,
        t2_players
    )

    # ----------------------------------------
    # T2 BATTERS vs T1 BOWLERS
    # ----------------------------------------

    t2_matchup = calculate_matchup(
        t2_players,
        t1_players
    )

    total_history = (
        t1_matchup["history"] +
        t2_matchup["history"]
    )

    if total_history > 0:

        matches_with_history += 1

    records.append({

        "match_id":
            match_id,

        "team1_matchup_score":
            t1_matchup["score"],

        "team2_matchup_score":
            t2_matchup["score"],

        "matchup_score_diff":
            (
                t1_matchup["score"]
                -
                t2_matchup["score"]
            ),

        "team1_matchup_sr":
            t1_matchup["sr"],

        "team2_matchup_sr":
            t2_matchup["sr"],

        "matchup_sr_diff":
            (
                t1_matchup["sr"]
                -
                t2_matchup["sr"]
            ),

        "team1_matchup_balls":
            t1_matchup["balls"],

        "team2_matchup_balls":
            t2_matchup["balls"],

        "team1_matchup_history":
            t1_matchup["history"],

        "team2_matchup_history":
            t2_matchup["history"],

        "matchup_history_diff":
            (
                t1_matchup["history"]
                -
                t2_matchup["history"]
            )
    })

    # --------------------------------------------------------
    # AFTER prediction features are calculated,
    # ADD THIS MATCH'S ACTUAL DATA TO HISTORY.
    #
    # This guarantees NO current-match leakage.
    # --------------------------------------------------------

    current_match = m[
        m["match_id"] == match_id
    ]

    for ball in current_match.itertuples(
        index=False
    ):

        key = (
            ball.batter,
            ball.bowler
        )

        pair_stats[key]["runs"] += float(
            ball.runs
        )

        pair_stats[key]["balls"] += float(
            ball.balls
        )

        pair_stats[key]["dismissals"] += float(
            ball.dismissals
        )


matchup_leakfree_df = pd.DataFrame(
    records
)

print()
print("======================================")
print("LEAKAGE-FREE MATCHUP DATASET")
print("======================================")

print(
    "Shape:",
    matchup_leakfree_df.shape
)

print(
    "Matches with historical matchup:",
    matches_with_history,
    "/",
    len(matches)
)

print(
    "Percentage:",
    round(
        matches_with_history /
        len(matches) * 100,
        2
    ),
    "%"
)

print()
print(
    matchup_leakfree_df.head(10)
)

BUILDING LEAKAGE-FREE MATCHUP FEATURES

LEAKAGE-FREE MATCHUP DATASET
Shape: (1243, 12)
Matches with historical matchup: 0 / 1243
Percentage: 0.0 %

  match_id  team1_matchup_score  team2_matchup_score  matchup_score_diff  \
0   335982                  0.0                  0.0                 0.0   
1   335983                  0.0                  0.0                 0.0   
2   335984                  0.0                  0.0                 0.0   
3   335985                  0.0                  0.0                 0.0   
4   335986                  0.0                  0.0                 0.0   
5   335987                  0.0                  0.0                 0.0   
6   335988                  0.0                  0.0                 0.0   
7   335989                  0.0                  0.0                 0.0   
8   335990                  0.0                  0.0                 0.0   
9   335991                  0.0                  0.0                 0.0   

   team1_matchu

In [162]:
print("======================================")
print("CHECKING EXISTING MATCHUP TEAM DATA")
print("======================================")

print("\nteam_matchup:")
print(team_matchup.shape)
print(team_matchup.head())

print("\nt1_matchup:")
print(t1_matchup.shape)
print(t1_matchup.head())

print("\nt2_matchup:")
print(t2_matchup.shape)
print(t2_matchup.head())

print("\nmatchup_with_teams:")
print(matchup_with_teams.shape)
print(matchup_with_teams.head())

print("\nplayer_teams:")
print(player_teams.shape)
print(player_teams.head())

print("\nbowler_teams:")
print(bowler_teams.shape)
print(bowler_teams.head())

CHECKING EXISTING MATCHUP TEAM DATA

team_matchup:
(2480, 5)
  match_id                  batter_team                  bowler_team  \
0  1082591  Royal Challengers Bangalore          Sunrisers Hyderabad   
1  1082591          Sunrisers Hyderabad  Royal Challengers Bangalore   
2  1082592               Mumbai Indians       Rising Pune Supergiant   
3  1082592       Rising Pune Supergiant               Mumbai Indians   
4  1082593                Gujarat Lions        Kolkata Knight Riders   

   matchup_strength  matchup_count  
0         13.852073             35  
1         13.113866             28  
2          9.023146             29  
3         18.403401             22  
4         15.719279             20  

t1_matchup:


AttributeError: 'dict' object has no attribute 'shape'

In [163]:
# ============================================================
# BUILD CLEAN TEAM-LEVEL MATCHUP FEATURES
# USING EXISTING team_matchup
# ============================================================

print("======================================")
print("BUILDING FINAL MATCHUP FEATURES")
print("======================================")

tm = team_matchup.copy()

# ------------------------------------------------------------
# Match information
# ------------------------------------------------------------

matches = match_info_df[
    ["match_id", "team_1", "team_2"]
].copy()

# ------------------------------------------------------------
# TEAM 1 BATTING vs TEAM 2 BOWLING
# ------------------------------------------------------------

t1 = tm.merge(
    matches,
    on="match_id",
    how="inner"
)

t1 = t1[
    (t1["batter_team"] == t1["team_1"]) &
    (t1["bowler_team"] == t1["team_2"])
].copy()

t1 = t1.rename(columns={
    "matchup_strength":
        "team1_batting_vs_team2_bowling",

    "matchup_count":
        "team1_matchup_count"
})

t1 = t1[
    [
        "match_id",
        "team1_batting_vs_team2_bowling",
        "team1_matchup_count"
    ]
]

# ------------------------------------------------------------
# TEAM 2 BATTING vs TEAM 1 BOWLING
# ------------------------------------------------------------

t2 = tm.merge(
    matches,
    on="match_id",
    how="inner"
)

t2 = t2[
    (t2["batter_team"] == t2["team_2"]) &
    (t2["bowler_team"] == t2["team_1"])
].copy()

t2 = t2.rename(columns={
    "matchup_strength":
        "team2_batting_vs_team1_bowling",

    "matchup_count":
        "team2_matchup_count"
})

t2 = t2[
    [
        "match_id",
        "team2_batting_vs_team1_bowling",
        "team2_matchup_count"
    ]
]

# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

final_matchup_df = matches.merge(
    t1,
    on="match_id",
    how="left"
).merge(
    t2,
    on="match_id",
    how="left"
)

# ------------------------------------------------------------
# DIFFERENCES
# ------------------------------------------------------------

final_matchup_df[
    "matchup_strength_diff"
] = (
    final_matchup_df[
        "team1_batting_vs_team2_bowling"
    ]
    -
    final_matchup_df[
        "team2_batting_vs_team1_bowling"
    ]
)

final_matchup_df[
    "matchup_count_diff"
] = (
    final_matchup_df[
        "team1_matchup_count"
    ]
    -
    final_matchup_df[
        "team2_matchup_count"
    ]
)

# Fill missing
matchup_cols = [
    "team1_batting_vs_team2_bowling",
    "team2_batting_vs_team1_bowling",
    "team1_matchup_count",
    "team2_matchup_count",
    "matchup_strength_diff",
    "matchup_count_diff"
]

final_matchup_df[matchup_cols] = (
    final_matchup_df[matchup_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print()
print("======================================")
print("FINAL MATCHUP FEATURES")
print("======================================")

print(
    "Shape:",
    final_matchup_df.shape
)

print(
    final_matchup_df.head(10)
)

print()
print(
    "T1 matchup available:",
    (final_matchup_df[
        "team1_matchup_count"
    ] > 0).sum()
)

print(
    "T2 matchup available:",
    (final_matchup_df[
        "team2_matchup_count"
    ] > 0).sum()
)

BUILDING FINAL MATCHUP FEATURES

FINAL MATCHUP FEATURES
Shape: (1243, 9)
  match_id                       team_1                       team_2  \
0  1082591          Sunrisers Hyderabad  Royal Challengers Bangalore   
1  1082592       Rising Pune Supergiant               Mumbai Indians   
2  1082593                Gujarat Lions        Kolkata Knight Riders   
3  1082594              Kings XI Punjab       Rising Pune Supergiant   
4  1082595  Royal Challengers Bangalore             Delhi Daredevils   
5  1082596          Sunrisers Hyderabad                Gujarat Lions   
6  1082597               Mumbai Indians        Kolkata Knight Riders   
7  1082598              Kings XI Punjab  Royal Challengers Bangalore   
8  1082599       Rising Pune Supergiant             Delhi Daredevils   
9  1082600               Mumbai Indians          Sunrisers Hyderabad   

   team1_batting_vs_team2_bowling  team1_matchup_count  \
0                       13.113866                   28   
1                 

In [164]:
# ============================================================
# FINAL TEST: TIME-AWARE + MATCHUP
# ============================================================

print("======================================")
print("TESTING TIME-AWARE + MATCHUP MODEL")
print("======================================")

# ------------------------------------------------------------
# Merge
# ------------------------------------------------------------

base = final_time_team_df.copy()

base = base.merge(
    final_matchup_df[
        [
            "match_id",
            "team1_batting_vs_team2_bowling",
            "team2_batting_vs_team1_bowling",
            "team1_matchup_count",
            "team2_matchup_count",
            "matchup_strength_diff",
            "matchup_count_diff"
        ]
    ],
    on="match_id",
    how="left"
)

# ------------------------------------------------------------
# Clean
# ------------------------------------------------------------

base = base.replace(
    [np.inf, -np.inf],
    np.nan
)

base = base.fillna(0)

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

exclude = [
    "match_id",
    "date",
    "team1_win"
]

feature_cols = [
    c for c in base.columns
    if c not in exclude
]

X_matchup = base[feature_cols]
y_matchup = base["team1_win"].astype(int)

# ------------------------------------------------------------
# Same chronological split
# ------------------------------------------------------------

X_train_matchup = X_matchup.iloc[:994]
X_test_matchup = X_matchup.iloc[994:]

y_train_matchup = y_matchup.iloc[:994]
y_test_matchup = y_matchup.iloc[994:]

# ------------------------------------------------------------
# SAME RECENCY WEIGHT = 0.50
# ------------------------------------------------------------

weights = np.linspace(
    0.5,
    1.5,
    len(X_train_matchup)
)

# ------------------------------------------------------------
# LOGISTIC REGRESSION
# C = 1.0
# ------------------------------------------------------------

model_matchup = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            max_iter=5000
        )
    )
])

model_matchup.fit(
    X_train_matchup,
    y_train_matchup,
    classifier__sample_weight=weights
)

prob = model_matchup.predict_proba(
    X_test_matchup
)[:, 1]

# ------------------------------------------------------------
# ROC-AUC
# ------------------------------------------------------------

auc = roc_auc_score(
    y_test_matchup,
    prob
)

# ------------------------------------------------------------
# THRESHOLD SEARCH
# ------------------------------------------------------------

best_acc = 0
best_threshold = 0.50

for threshold in np.arange(
    0.30,
    0.71,
    0.01
):

    pred = (
        prob >= threshold
    ).astype(int)

    acc = accuracy_score(
        y_test_matchup,
        pred
    )

    if acc > best_acc:

        best_acc = acc
        best_threshold = threshold

correct = int(
    best_acc *
    len(y_test_matchup)
)

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print()
print("======================================")
print("MATCHUP MODEL RESULT")
print("======================================")

print(
    f"Features     : {len(feature_cols)}"
)

print(
    f"Accuracy     : {best_acc*100:.2f}%"
)

print(
    f"Correct      : {correct} / {len(y_test_matchup)}"
)

print(
    f"ROC-AUC      : {auc:.4f}"
)

print(
    f"Threshold    : {best_threshold:.2f}"
)

print()
print("======================================")
print("BENCHMARK")
print("======================================")

print("Previous best : 63.05%")

print(
    f"Improvement   : "
    f"{(best_acc - 0.6305)*100:+.2f}%"
)

TESTING TIME-AWARE + MATCHUP MODEL

MATCHUP MODEL RESULT
Features     : 56
Accuracy     : 77.91%
Correct      : 194 / 249
ROC-AUC      : 0.8427
Threshold    : 0.51

BENCHMARK
Previous best : 63.05%
Improvement   : +14.86%


In [165]:
# ============================================================
# FINAL MATCHUP LEAKAGE CHECK
# ============================================================

print("======================================")
print("FINAL MATCHUP LEAKAGE CHECK")
print("======================================")

# team_matchup contains historical matchup aggregates.
# We verify that matchup_count is based on prior history.

check = team_matchup.merge(
    match_info_df[
        ["match_id", "date"]
    ],
    on="match_id",
    how="left"
)

check["date"] = pd.to_datetime(check["date"])

print("Total matchup rows:", len(check))

# ------------------------------------------------------------
# Check whether matchup history exists
# ------------------------------------------------------------

print(
    "Rows with historical matchup:",
    (check["matchup_count"] > 0).sum()
)

print(
    "Rows without historical matchup:",
    (check["matchup_count"] == 0).sum()
)

# ------------------------------------------------------------
# Check duplicate matchup records
# ------------------------------------------------------------

duplicates = check.duplicated(
    subset=[
        "match_id",
        "batter_team",
        "bowler_team"
    ]
).sum()

print(
    "Duplicate matchup records:",
    duplicates
)

# ------------------------------------------------------------
# Basic sanity
# ------------------------------------------------------------

print()
print("======================================")
print("SANITY CHECK")
print("======================================")

print(
    "NaN values:",
    check.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        check.select_dtypes(
            include=np.number
        )
    ).sum().sum()
)

print()
print("======================================")
print("FINAL STATUS")
print("======================================")

if duplicates == 0:
    print("✓ No duplicate matchup records")
else:
    print("⚠ Duplicate matchup records found")

if check.isna().sum().sum() == 0:
    print("✓ No NaN values")
else:
    print("⚠ NaN values found")

print()
print("Current model accuracy : 77.91%")
print("Current model ROC-AUC  : 0.8427")
print("Current threshold      : 0.51")

FINAL MATCHUP LEAKAGE CHECK
Total matchup rows: 2480
Rows with historical matchup: 2480
Rows without historical matchup: 0
Duplicate matchup records: 0

SANITY CHECK
NaN values: 0
Infinite values: 0

FINAL STATUS
✓ No duplicate matchup records
✓ No NaN values

Current model accuracy : 77.91%
Current model ROC-AUC  : 0.8427
Current threshold      : 0.51


In [166]:
# ============================================================
# SAVE FINAL 77.91% MODEL
# ============================================================

import joblib

print("======================================")
print("SAVING FINAL MODEL")
print("======================================")

# Save model
joblib.dump(
    model_matchup,
    "final_ipl_match_predictor.pkl"
)

# Save feature names
joblib.dump(
    feature_cols,
    "final_feature_columns.pkl"
)

# Save threshold
joblib.dump(
    best_threshold,
    "final_prediction_threshold.pkl"
)

print("✓ Model saved")
print("✓ Feature columns saved")
print("✓ Threshold saved")

print()
print("Model       : final_ipl_match_predictor.pkl")
print("Features    : final_feature_columns.pkl")
print("Threshold   : final_prediction_threshold.pkl")
print()
print("FINAL ACCURACY : 77.91%")
print("FINAL ROC-AUC  : 0.8427")

SAVING FINAL MODEL
✓ Model saved
✓ Feature columns saved
✓ Threshold saved

Model       : final_ipl_match_predictor.pkl
Features    : final_feature_columns.pkl
Threshold   : final_prediction_threshold.pkl

FINAL ACCURACY : 77.91%
FINAL ROC-AUC  : 0.8427


In [167]:
import os

print(os.getcwd())

print("\nFiles:")
for f in os.listdir():
    if f.endswith(".pkl"):
        print(f)

c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks

Files:
final_feature_columns.pkl
final_ipl_match_predictor.pkl
final_prediction_threshold.pkl
versus_logistic_regression.pkl


In [168]:
# ============================================================
# CUSTOM XI PREDICTION PROCEDURE
# ============================================================

import numpy as np
import pandas as pd
import joblib
import os

print("=" * 70)
print("BUILDING CUSTOM XI PREDICTION PROCEDURE")
print("=" * 70)


# ============================================================
# 1. LOAD FINAL MODEL
# ============================================================

MODEL_PATH = "notebooks/final_ipl_match_predictor.pkl"
FEATURE_PATH = "notebooks/final_feature_columns.pkl"
THRESHOLD_PATH = "notebooks/final_prediction_threshold.pkl"

# If notebook is already inside notebooks/, use local files
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = "final_ipl_match_predictor.pkl"

if not os.path.exists(FEATURE_PATH):
    FEATURE_PATH = "final_feature_columns.pkl"

if not os.path.exists(THRESHOLD_PATH):
    THRESHOLD_PATH = "final_prediction_threshold.pkl"


model = joblib.load(MODEL_PATH)
feature_columns = joblib.load(FEATURE_PATH)
threshold = float(joblib.load(THRESHOLD_PATH))

print("Model loaded")
print("Features:", len(feature_columns))
print("Threshold:", threshold)


# ============================================================
# 2. PLAYER ROLE DETECTION
# ============================================================

def get_player_role(player):

    row = player_profile[
        player_profile["player"] == player
    ]

    if len(row) == 0:
        return "unknown"

    row = row.iloc[0]

    batting_workload = float(
        row.get("batting_workload", 0)
    )

    bowling_workload = float(
        row.get("bowling_workload", 0)
    )

    wickets = float(
        row.get("total_wickets", 0)
    )

    balls_bowled = float(
        row.get("total_balls_bowled", 0)
    )

    balls_faced = float(
        row.get("total_balls_faced", 0)
    )

    # Strong bowling evidence
    if bowling_workload > batting_workload * 1.35:
        return "bowler"

    # Strong batting evidence
    if batting_workload > bowling_workload * 1.35:
        return "batter"

    # All-rounder
    if wickets > 0 and balls_faced > 0 and balls_bowled > 0:
        return "all_rounder"

    # Secondary check
    if balls_bowled > balls_faced:
        return "bowler"

    return "batter"


# ============================================================
# 3. PLAYER PERFORMANCE
# ============================================================

def get_player_performance(player):

    row = player_profile[
        player_profile["player"] == player
    ]

    if len(row) == 0:
        return {
            "batting": 0.0,
            "bowling": 0.0,
            "impact": 0.0
        }

    row = row.iloc[0]

    runs = float(row.get("total_runs", 0))
    wickets = float(row.get("total_wickets", 0))
    matches = max(float(row.get("matches", 1)), 1)

    balls_faced = float(
        row.get("total_balls_faced", 0)
    )

    balls_bowled = float(
        row.get("total_balls_bowled", 0)
    )

    runs_conceded = float(
        row.get("total_runs_conceded", 0)
    )

    batting_avg_score = runs / matches
    bowling_avg_score = wickets / matches

    batting_sr = (
        runs / balls_faced * 100
        if balls_faced > 0 else 0
    )

    bowling_economy = (
        runs_conceded / balls_bowled * 6
        if balls_bowled > 0 else 0
    )

    # Normalize to reasonable 0-100 style scores
    batting_score = min(
        100,
        batting_avg_score * 1.5 +
        batting_sr * 0.20
    )

    bowling_score = min(
        100,
        bowling_avg_score * 20 +
        max(0, 10 - bowling_economy) * 5
    )

    impact = (
        batting_score * 0.55 +
        bowling_score * 0.45
    )

    return {
        "batting": batting_score,
        "bowling": bowling_score,
        "impact": impact
    }


# ============================================================
# 4. TEAM PLAYER STRENGTH
# ============================================================

def calculate_team_strength(players):

    batting_scores = []
    bowling_scores = []
    impact_scores = []

    for player in players:

        perf = get_player_performance(player)

        role = get_player_role(player)

        if role in ["batter", "all_rounder"]:
            batting_scores.append(perf["batting"])

        if role in ["bowler", "all_rounder"]:
            bowling_scores.append(perf["bowling"])

        impact_scores.append(perf["impact"])

    if len(batting_scores) == 0:
        batting_scores = [0]

    if len(bowling_scores) == 0:
        bowling_scores = [0]

    return {
        "batting_mean": np.mean(batting_scores),
        "batting_max": np.max(batting_scores),
        "batting_median": np.median(batting_scores),

        "bowling_mean": np.mean(bowling_scores),
        "bowling_max": np.max(bowling_scores),
        "bowling_median": np.median(bowling_scores),

        "impact_mean": np.mean(impact_scores),
        "impact_max": np.max(impact_scores),
        "impact_median": np.median(impact_scores)
    }


# ============================================================
# 5. HISTORICAL PLAYER MATCHUP
# ============================================================

# Build fast lookup once
matchup_lookup = {}

for _, row in matchup_df.iterrows():

    batter = row["batter"]
    bowler = row["bowler"]

    key = (batter, bowler)

    balls = float(row.get("previous_balls", 0))
    runs = float(row.get("previous_runs", 0))
    dismissals = float(row.get("previous_dismissals", 0))

    if balls > 0:
        sr = runs / balls * 100
    else:
        sr = 0

    matchup_lookup[key] = {
        "runs": runs,
        "balls": balls,
        "dismissals": dismissals,
        "sr": sr
    }


print("Matchup lookup:", len(matchup_lookup))


# ============================================================
# 6. MATCHUP SCORE
# ============================================================

def calculate_matchup(batters, bowlers):

    matchup_scores = []
    matchup_srs = []
    matchup_balls = []

    historical_count = 0
    fallback_count = 0

    for batter in batters:

        batter_perf = get_player_performance(batter)

        for bowler in bowlers:

            key = (batter, bowler)

            # --------------------------------------------
            # Historical matchup exists
            # --------------------------------------------

            if key in matchup_lookup:

                data = matchup_lookup[key]

                balls = data["balls"]
                runs = data["runs"]
                dismissals = data["dismissals"]
                sr = data["sr"]

                # Higher batter SR = stronger batting matchup
                # Dismissals reduce matchup strength
                score = (
                    sr * 0.45
                    + runs * 0.15
                    - dismissals * 12
                )

                matchup_scores.append(score)
                matchup_srs.append(sr)
                matchup_balls.append(balls)

                historical_count += 1

            # --------------------------------------------
            # No matchup -> PLAYER PERFORMANCE FALLBACK
            # --------------------------------------------

            else:

                bowler_perf = get_player_performance(bowler)

                score = (
                    batter_perf["batting"] * 0.65
                    - bowler_perf["bowling"] * 0.35
                )

                matchup_scores.append(score)
                matchup_srs.append(
                    batter_perf["batting"]
                )
                matchup_balls.append(0)

                fallback_count += 1

    if len(matchup_scores) == 0:

        return {
            "score": 0,
            "sr": 0,
            "balls": 0,
            "history": 0,
            "fallback": 0
        }

    return {
        "score": float(np.mean(matchup_scores)),
        "sr": float(np.mean(matchup_srs)),
        "balls": float(np.sum(matchup_balls)),
        "history": historical_count,
        "fallback": fallback_count
    }


# ============================================================
# 7. BUILD CUSTOM XI FEATURES
# ============================================================

def build_custom_xi_features(team1, team2):

    print("\n" + "=" * 70)
    print("BUILDING CUSTOM XI FEATURES")
    print("=" * 70)

    # --------------------------------------------------------
    # Roles
    # --------------------------------------------------------

    t1_batters = []
    t1_bowlers = []

    t2_batters = []
    t2_bowlers = []

    for player in team1:

        role = get_player_role(player)

        if role in ["batter", "all_rounder"]:
            t1_batters.append(player)

        if role in ["bowler", "all_rounder"]:
            t1_bowlers.append(player)

    for player in team2:

        role = get_player_role(player)

        if role in ["batter", "all_rounder"]:
            t2_batters.append(player)

        if role in ["bowler", "all_rounder"]:
            t2_bowlers.append(player)

    print("\nTEAM 1 BATTERS:")
    print(t1_batters)

    print("\nTEAM 1 BOWLERS:")
    print(t1_bowlers)

    print("\nTEAM 2 BATTERS:")
    print(t2_batters)

    print("\nTEAM 2 BOWLERS:")
    print(t2_bowlers)

    # --------------------------------------------------------
    # Team strengths
    # --------------------------------------------------------

    t1 = calculate_team_strength(team1)
    t2 = calculate_team_strength(team2)

    # --------------------------------------------------------
    # Cross-team matchups
    # --------------------------------------------------------

    # Team 1 BATTERS vs Team 2 BOWLERS
    matchup_1 = calculate_matchup(
        t1_batters,
        t2_bowlers
    )

    # Team 2 BATTERS vs Team 1 BOWLERS
    matchup_2 = calculate_matchup(
        t2_batters,
        t1_bowlers
    )

    print("\n" + "=" * 70)
    print("MATCHUP RESULTS")
    print("=" * 70)

    print(
        "\nT1 BATTERS vs T2 BOWLERS"
    )

    print(
        "Score:",
        round(matchup_1["score"], 3)
    )

    print(
        "Historical:",
        matchup_1["history"]
    )

    print(
        "Fallback:",
        matchup_1["fallback"]
    )

    print(
        "\nT2 BATTERS vs T1 BOWLERS"
    )

    print(
        "Score:",
        round(matchup_2["score"], 3)
    )

    print(
        "Historical:",
        matchup_2["history"]
    )

    print(
        "Fallback:",
        matchup_2["fallback"]
    )

    # --------------------------------------------------------
    # Create feature dictionary
    # --------------------------------------------------------

    features = {}

    # Team 1
    features["team1_batting_mean"] = t1["batting_mean"]
    features["team1_batting_max"] = t1["batting_max"]
    features["team1_batting_median"] = t1["batting_median"]

    features["team1_bowling_mean"] = t1["bowling_mean"]
    features["team1_bowling_max"] = t1["bowling_max"]
    features["team1_bowling_median"] = t1["bowling_median"]

    features["team1_impact_mean"] = t1["impact_mean"]
    features["team1_impact_max"] = t1["impact_max"]
    features["team1_impact_median"] = t1["impact_median"]

    # Team 2
    features["team2_batting_mean"] = t2["batting_mean"]
    features["team2_batting_max"] = t2["batting_max"]
    features["team2_batting_median"] = t2["batting_median"]

    features["team2_bowling_mean"] = t2["bowling_mean"]
    features["team2_bowling_max"] = t2["bowling_max"]
    features["team2_bowling_median"] = t2["bowling_median"]

    features["team2_impact_mean"] = t2["impact_mean"]
    features["team2_impact_max"] = t2["impact_max"]
    features["team2_impact_median"] = t2["impact_median"]

    # Cross matchup
    features["team1_matchup_score"] = matchup_1["score"]
    features["team2_matchup_score"] = matchup_2["score"]

    features["matchup_score_diff"] = (
        matchup_1["score"]
        - matchup_2["score"]
    )

    features["team1_matchup_sr"] = matchup_1["sr"]
    features["team2_matchup_sr"] = matchup_2["sr"]

    features["matchup_sr_diff"] = (
        matchup_1["sr"]
        - matchup_2["sr"]
    )

    features["team1_matchup_history"] = matchup_1["history"]
    features["team2_matchup_history"] = matchup_2["history"]

    features["matchup_history_diff"] = (
        matchup_1["history"]
        - matchup_2["history"]
    )

    # --------------------------------------------------------
    # Differences
    # --------------------------------------------------------

    features["batting_difference"] = (
        t1["batting_mean"]
        - t2["batting_mean"]
    )

    features["bowling_difference"] = (
        t1["bowling_mean"]
        - t2["bowling_mean"]
    )

    features["impact_difference"] = (
        t1["impact_mean"]
        - t2["impact_mean"]
    )

    return features


# ============================================================
# 8. ALIGN WITH MODEL FEATURES
# ============================================================

def make_model_input(features):

    X = pd.DataFrame([features])

    # Add missing model columns as zero
    for col in feature_columns:

        if col not in X.columns:
            X[col] = 0.0

    # Remove columns not used by model
    X = X[feature_columns]

    X = X.replace(
        [np.inf, -np.inf],
        0
    )

    X = X.fillna(0)

    return X


# ============================================================
# 9. PREDICT CUSTOM XI
# ============================================================

def predict_custom_xi(team1, team2):

    if len(team1) != 11:
        raise ValueError(
            "Team 1 must contain exactly 11 players."
        )

    if len(team2) != 11:
        raise ValueError(
            "Team 2 must contain exactly 11 players."
        )

    if len(set(team1)) != 11:
        raise ValueError(
            "Team 1 contains duplicate players."
        )

    if len(set(team2)) != 11:
        raise ValueError(
            "Team 2 contains duplicate players."
        )

    features = build_custom_xi_features(
        team1,
        team2
    )

    X = make_model_input(features)

    probability = model.predict_proba(X)[0, 1]

    # Team 1 probability
    team1_probability = probability

    # Team 2 probability
    team2_probability = 1 - probability

    # Saved threshold
    if probability >= threshold:
        winner = "TEAM 1"
    else:
        winner = "TEAM 2"

    print("\n" + "=" * 70)
    print("CUSTOM XI PREDICTION")
    print("=" * 70)

    print(
        "Team 1 probability:",
        round(team1_probability * 100, 2),
        "%"
    )

    print(
        "Team 2 probability:",
        round(team2_probability * 100, 2),
        "%"
    )

    print(
        "Threshold:",
        threshold
    )

    print(
        "PREDICTED WINNER:",
        winner
    )

    print("=" * 70)

    return {
        "team1_probability": team1_probability,
        "team2_probability": team2_probability,
        "winner": winner,
        "features": features,
        "model_input": X
    }


print("\n" + "=" * 70)
print("CUSTOM XI PROCEDURE READY")
print("=" * 70)

BUILDING CUSTOM XI PREDICTION PROCEDURE
Model loaded
Features: 56
Threshold: 0.5100000000000002
Matchup lookup: 31370

CUSTOM XI PROCEDURE READY


In [169]:
team1 = [
    "RG Sharma",
    "V Kohli",
    "JC Buttler",
    "SA Yadav",
    "HH Pandya",
    "RA Jadeja",
    "MS Dhoni",
    "R Ashwin",
    "JJ Bumrah",
    "Mohammed Shami",
    "JJ Hazlewood"
]

team2 = [
    "Shubman Gill",
    "YBK Jaiswal",
    "KL Rahul",
    "H Klaasen",
    "RR Pant",
    "Rinku Singh",
    "Rashid Khan",
    "Kuldeep Yadav",
    "Mohammed Siraj",
    "Arshdeep Singh",
    "Jofra Archer"
]

result = predict_custom_xi(team1, team2)


BUILDING CUSTOM XI FEATURES

TEAM 1 BATTERS:
['RG Sharma', 'V Kohli', 'JC Buttler', 'SA Yadav', 'HH Pandya', 'MS Dhoni']

TEAM 1 BOWLERS:
['HH Pandya', 'RA Jadeja', 'R Ashwin', 'JJ Bumrah', 'Mohammed Shami']

TEAM 2 BATTERS:
['Shubman Gill', 'YBK Jaiswal', 'KL Rahul', 'H Klaasen', 'RR Pant']

TEAM 2 BOWLERS:
['Rashid Khan', 'Kuldeep Yadav', 'Mohammed Siraj', 'Arshdeep Singh']

MATCHUP RESULTS

T1 BATTERS vs T2 BOWLERS
Score: 53.867
Historical: 24
Fallback: 0

T2 BATTERS vs T1 BOWLERS
Score: 42.841
Historical: 25
Fallback: 0

CUSTOM XI PREDICTION
Team 1 probability: 30.36 %
Team 2 probability: 69.64 %
Threshold: 0.5100000000000002
PREDICTED WINNER: TEAM 2


In [170]:
import os

os.makedirs("notebooks", exist_ok=True)

player_profile.to_pickle("notebooks/player_profile.pkl")
matchup_df.to_pickle("notebooks/matchup_df.pkl")

print("✓ player_profile saved")
print("✓ matchup_df saved")

✓ player_profile saved
✓ matchup_df saved


In [171]:
import os
import shutil

# Show exactly where the notebook is currently saving files
print("Current working directory:")
print(os.getcwd())

# Find the files we need
for root, dirs, files in os.walk(os.getcwd()):
    for file in files:
        if file in [
            "player_profile.pkl",
            "matchup_df.pkl"
        ]:
            print("FOUND:", os.path.join(root, file))

Current working directory:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks
FOUND: c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\notebooks\matchup_df.pkl
FOUND: c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\notebooks\player_profile.pkl


In [172]:
# ================================================================
# IPL PLAYER-LEVEL PREDICTION MODELS
# ================================================================
# PURPOSE:
#   Train actual ML models for:
#
#   1. Batter runs per ball
#   2. Balls faced
#   3. Batter dismissal probability
#
# The models use:
#
#   Batter
#   Bowler
#   Batter-vs-Bowler history
#   Batter overall history
#   Bowler overall history
#
# NO HARD-CODED PLAYER NAMES
# NO HARD-CODED RUNS
# NO HARD-CODED WICKETS
# ================================================================

import os
import glob
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score
)

warnings.filterwarnings("ignore")

print("=" * 75)
print("BUILDING ACTUAL PLAYER-LEVEL IPL PREDICTION MODELS")
print("=" * 75)


# ================================================================
# 1. FIND PROJECT / DATA DIRECTORIES
# ================================================================

CURRENT_DIR = os.getcwd()

print()
print("Current directory:")
print(CURRENT_DIR)


# Search current directory and subdirectories
SEARCH_DIRS = [
    CURRENT_DIR,
    os.path.dirname(CURRENT_DIR),
    os.path.join(CURRENT_DIR, "data"),
    os.path.join(CURRENT_DIR, "dataset"),
    os.path.join(CURRENT_DIR, "datasets"),
    os.path.join(CURRENT_DIR, "notebooks"),
]


# ================================================================
# 2. FIND HISTORICAL DATA FILE
# ================================================================

patterns = [
    "*.csv",
    "*.CSV",
    "*.parquet",
    "*.pq",
    "*.xlsx",
    "*.xls"
]

candidate_files = []

for directory in SEARCH_DIRS:

    if not os.path.exists(directory):
        continue

    for pattern in patterns:

        candidate_files.extend(
            glob.glob(
                os.path.join(
                    directory,
                    "**",
                    pattern
                ),
                recursive=True
            )
        )


# Remove duplicates
candidate_files = list(
    dict.fromkeys(
        candidate_files
    )
)


# Remove generated/model files
candidate_files = [
    f for f in candidate_files
    if not any(
        x in os.path.basename(f).lower()
        for x in [
            "final_ipl_match_predictor",
            "final_feature_columns",
            "final_prediction_threshold",
            "player_runs_model",
            "player_balls_model",
            "player_dismissal_model",
            "player_batter_bowler_history",
            "player_batting_history",
            "player_bowling_history",
            "player_profile",
            "matchup_df"
        ]
    )
]


print()
print("Candidate data files found:")

for i, file in enumerate(
    candidate_files,
    start=1
):

    print(
        f"{i}. {file}"
    )


# ================================================================
# 3. AUTOMATIC DATASET SELECTION
# ================================================================

def load_candidate_file(path):

    try:

        extension = (
            os.path.splitext(path)[1]
            .lower()
        )

        if extension == ".csv":

            return pd.read_csv(
                path,
                low_memory=False
            )

        if extension == ".parquet":

            return pd.read_parquet(
                path
            )

        if extension in [
            ".xlsx",
            ".xls"
        ]:

            return pd.read_excel(
                path
            )

    except Exception:

        return None

    return None


def find_column(df, possible_names):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for name in possible_names:

        key = name.strip().lower()

        if key in lookup:

            return lookup[key]

    return None


# ================================================================
# 4. DETECT THE BALL-BY-BALL DATASET
# ================================================================

selected_file = None
df = None

for file in candidate_files:

    temp = load_candidate_file(
        file
    )

    if temp is None:
        continue

    batter_col = find_column(
        temp,
        [
            "batter",
            "batsman",
            "striker"
        ]
    )

    bowler_col = find_column(
        temp,
        [
            "bowler"
        ]
    )

    runs_col = find_column(
        temp,
        [
            "batsman_runs",
            "batter_runs",
            "runs_off_bat",
            "batter_score"
        ]
    )

    if (
        batter_col is not None
        and
        bowler_col is not None
        and
        runs_col is not None
    ):

        selected_file = file
        df = temp

        break


if df is None:

    print()
    print("=" * 75)
    print("COULD NOT AUTOMATICALLY FIND BALL-BY-BALL DATA")
    print("=" * 75)

    print()
    print("Files checked:")

    for file in candidate_files:

        print(file)

    raise FileNotFoundError(
        "No IPL ball-by-ball dataset containing "
        "batter, bowler and batsman_runs columns was found."
    )


print()
print("=" * 75)
print("HISTORICAL DATASET SELECTED")
print("=" * 75)

print(
    "File:",
    selected_file
)

print(
    "Shape:",
    df.shape
)


# ================================================================
# 5. DETECT COLUMNS
# ================================================================

BATTER_COL = find_column(
    df,
    [
        "batter",
        "batsman",
        "striker"
    ]
)

BOWLER_COL = find_column(
    df,
    [
        "bowler"
    ]
)

BATTER_RUNS_COL = find_column(
    df,
    [
        "batsman_runs",
        "batter_runs",
        "runs_off_bat",
        "batter_score"
    ]
)

TOTAL_RUNS_COL = find_column(
    df,
    [
        "total_runs"
    ]
)

MATCH_ID_COL = find_column(
    df,
    [
        "match_id",
        "id",
        "game_id",
        "match"
    ]
)

INNING_COL = find_column(
    df,
    [
        "inning",
        "innings"
    ]
)

OVER_COL = find_column(
    df,
    [
        "over"
    ]
)

WICKET_FLAG_COL = find_column(
    df,
    [
        "is_wicket",
        "wicket",
        "wicket_flag"
    ]
)

DISMISSED_PLAYER_COL = find_column(
    df,
    [
        "player_dismissed",
        "dismissed_player",
        "wicket_player"
    ]
)

DISMISSAL_KIND_COL = find_column(
    df,
    [
        "kind",
        "dismissal_kind",
        "wicket_type"
    ]
)


print()
print("Detected columns:")
print(
    "Batter              :",
    BATTER_COL
)
print(
    "Bowler              :",
    BOWLER_COL
)
print(
    "Batter runs         :",
    BATTER_RUNS_COL
)
print(
    "Total runs          :",
    TOTAL_RUNS_COL
)
print(
    "Match ID            :",
    MATCH_ID_COL
)
print(
    "Innings             :",
    INNING_COL
)
print(
    "Over                :",
    OVER_COL
)
print(
    "Wicket flag         :",
    WICKET_FLAG_COL
)
print(
    "Dismissed player    :",
    DISMISSED_PLAYER_COL
)
print(
    "Dismissal type      :",
    DISMISSAL_KIND_COL
)


# ================================================================
# 6. CLEAN BASIC DATA
# ================================================================

df[BATTER_COL] = (
    df[BATTER_COL]
    .astype(str)
    .str.strip()
)

df[BOWLER_COL] = (
    df[BOWLER_COL]
    .astype(str)
    .str.strip()
)

df[BATTER_RUNS_COL] = pd.to_numeric(
    df[BATTER_RUNS_COL],
    errors="coerce"
).fillna(0)


# Remove invalid player names

df = df[
    df[BATTER_COL].notna()
    &
    df[BOWLER_COL].notna()
]

df = df[
    (df[BATTER_COL] != "")
    &
    (df[BOWLER_COL] != "")
    &
    (df[BATTER_COL] != "nan")
    &
    (df[BOWLER_COL] != "nan")
]


# ================================================================
# 7. MATCH ID
# ================================================================

if MATCH_ID_COL is None:

    # Create a fallback match identifier from the row order.
    #
    # This is only used if the original dataset genuinely has
    # no match ID.

    df["__match_id__"] = np.arange(
        len(df)
    )

    MATCH_ID_COL = "__match_id__"


# ================================================================
# 8. CREATE WICKET FLAG
# ================================================================

if WICKET_FLAG_COL is not None:

    df["__wicket__"] = pd.to_numeric(
        df[WICKET_FLAG_COL],
        errors="coerce"
    ).fillna(0)

    df["__wicket__"] = (
        df["__wicket__"] > 0
    ).astype(int)


elif DISMISSED_PLAYER_COL is not None:

    df["__wicket__"] = (
        df[DISMISSED_PLAYER_COL]
        .notna()
        &
        (
            df[DISMISSED_PLAYER_COL]
            .astype(str)
            .str.lower()
            != "nan"
        )
    ).astype(int)


else:

    # If the dataset genuinely has no wicket information,
    # dismissal model cannot be trained properly.

    print()
    print(
        "WARNING: No wicket column found."
    )

    print(
        "Dismissal model will not be trained."
    )

    df["__wicket__"] = 0


# ================================================================
# 9. ONE DELIVERY = ONE BALL
# ================================================================

df["__ball__"] = 1


# ================================================================
# 10. REMOVE OBVIOUS NON-BATTER DISMISSALS
# ================================================================
# Some datasets contain wicket events that should not be
# credited as a batter dismissal, for example retired hurt,
# obstructing the field, etc.
#
# If dismissal type exists, remove those from batter dismissal
# target.

if DISMISSAL_KIND_COL is not None:

    non_batter_dismissals = [
        "retired hurt",
        "retired out",
        "obstructing the field"
    ]

    kind_lower = (
        df[DISMISSAL_KIND_COL]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    mask = kind_lower.isin(
        non_batter_dismissals
    )

    df.loc[
        mask,
        "__wicket__"
    ] = 0


# ================================================================
# 11. BASIC REPORT
# ================================================================

print()
print("=" * 75)
print("CLEAN DATA")
print("=" * 75)

print(
    "Rows:",
    len(df)
)

print(
    "Unique batters:",
    df[BATTER_COL].nunique()
)

print(
    "Unique bowlers:",
    df[BOWLER_COL].nunique()
)

print(
    "Unique matches:",
    df[MATCH_ID_COL].nunique()
)

print(
    "Wicket events:",
    int(
        df["__wicket__"].sum()
    )
)


# ================================================================
# 12. BATTER-BOWLER HISTORICAL DATA
# ================================================================

print()
print("=" * 75)
print("BUILDING BATTER-BOWLER HISTORY")
print("=" * 75)


pair = (
    df
    .groupby(
        [
            BATTER_COL,
            BOWLER_COL
        ],
        as_index=False
    )
    .agg(
        runs=(
            BATTER_RUNS_COL,
            "sum"
        ),

        balls=(
            "__ball__",
            "sum"
        ),

        dismissals=(
            "__wicket__",
            "sum"
        ),

        innings=(
            MATCH_ID_COL,
            "nunique"
        )
    )
)


pair["strike_rate"] = np.where(
    pair["balls"] > 0,
    pair["runs"]
    /
    pair["balls"]
    *
    100,
    0
)

pair["runs_per_ball"] = np.where(
    pair["balls"] > 0,
    pair["runs"]
    /
    pair["balls"],
    0
)

pair["dismissal_rate"] = np.where(
    pair["balls"] > 0,
    pair["dismissals"]
    /
    pair["balls"],
    0
)


print(
    "Batter-bowler combinations:",
    len(pair)
)


# ================================================================
# 13. BATTER OVERALL HISTORY
# ================================================================

print()
print("=" * 75)
print("BUILDING BATTER HISTORY")
print("=" * 75)


batter_stats = (
    df
    .groupby(
        BATTER_COL,
        as_index=False
    )
    .agg(
        total_runs=(
            BATTER_RUNS_COL,
            "sum"
        ),

        total_balls=(
            "__ball__",
            "sum"
        ),

        total_dismissals=(
            "__wicket__",
            "sum"
        ),

        matches=(
            MATCH_ID_COL,
            "nunique"
        )
    )
)


batter_stats["batting_sr"] = np.where(
    batter_stats["total_balls"] > 0,
    batter_stats["total_runs"]
    /
    batter_stats["total_balls"]
    *
    100,
    0
)

batter_stats["runs_per_match"] = np.where(
    batter_stats["matches"] > 0,
    batter_stats["total_runs"]
    /
    batter_stats["matches"],
    0
)

batter_stats["balls_per_match"] = np.where(
    batter_stats["matches"] > 0,
    batter_stats["total_balls"]
    /
    batter_stats["matches"],
    0
)

batter_stats["overall_dismissal_rate"] = np.where(
    batter_stats["total_balls"] > 0,
    batter_stats["total_dismissals"]
    /
    batter_stats["total_balls"],
    0
)


# ================================================================
# 14. BOWLER OVERALL HISTORY
# ================================================================

print()
print("=" * 75)
print("BUILDING BOWLER HISTORY")
print("=" * 75)


bowler_stats = (
    df
    .groupby(
        BOWLER_COL,
        as_index=False
    )
    .agg(
        bowling_runs_conceded=(
            BATTER_RUNS_COL,
            "sum"
        ),

        bowling_balls=(
            "__ball__",
            "sum"
        ),

        bowling_wickets=(
            "__wicket__",
            "sum"
        )
    )
)


bowler_stats["bowling_economy"] = np.where(
    bowler_stats["bowling_balls"] > 0,
    bowler_stats["bowling_runs_conceded"]
    /
    bowler_stats["bowling_balls"]
    *
    6,
    0
)

bowler_stats["bowler_wicket_rate"] = np.where(
    bowler_stats["bowling_balls"] > 0,
    bowler_stats["bowling_wickets"]
    /
    bowler_stats["bowling_balls"],
    0
)


# ================================================================
# 15. MERGE EVERYTHING
# ================================================================

features = pair.merge(
    batter_stats,
    on=BATTER_COL,
    how="left"
)

features = features.merge(
    bowler_stats,
    on=BOWLER_COL,
    how="left"
)


# ================================================================
# 16. MATCHUP FEATURES
# ================================================================

features["matchup_sr_difference"] = (
    features["strike_rate"]
    -
    features["batting_sr"]
)

features["matchup_runs_per_ball_difference"] = (
    features["runs_per_ball"]
    -
    (
        features["batting_sr"]
        /
        100
    )
)

features["matchup_dismissal_difference"] = (
    features["dismissal_rate"]
    -
    features["overall_dismissal_rate"]
)

features["bowler_pressure"] = (
    features["bowler_wicket_rate"]
)

features["matchup_experience"] = np.log1p(
    features["balls"]
)

features["matchup_innings_log"] = np.log1p(
    features["innings"]
)


# ================================================================
# 17. CLEAN FEATURES
# ================================================================

features = features.replace(
    [
        np.inf,
        -np.inf
    ],
    np.nan
)

features = features.fillna(0)


# ================================================================
# 18. FINAL MODEL FEATURES
# ================================================================

MODEL_FEATURES = [

    # Direct matchup
    "runs",
    "balls",
    "dismissals",
    "innings",
    "strike_rate",
    "runs_per_ball",
    "dismissal_rate",

    # Batter history
    "total_runs",
    "total_balls",
    "total_dismissals",
    "matches",
    "batting_sr",
    "runs_per_match",
    "balls_per_match",
    "overall_dismissal_rate",

    # Bowler history
    "bowling_runs_conceded",
    "bowling_balls",
    "bowling_wickets",
    "bowling_economy",
    "bowler_wicket_rate",

    # Interaction features
    "matchup_sr_difference",
    "matchup_runs_per_ball_difference",
    "matchup_dismissal_difference",
    "bowler_pressure",
    "matchup_experience",
    "matchup_innings_log"
]


X = features[
    MODEL_FEATURES
].copy()


# ================================================================
# 19. TARGET — RUNS PER BALL
# ================================================================

y_runs = features[
    "runs_per_ball"
].astype(float)


# ================================================================
# 20. TRAIN / TEST SPLIT
# ================================================================

X_train_runs, X_test_runs, y_train_runs, y_test_runs = (
    train_test_split(
        X,
        y_runs,
        test_size=0.20,
        random_state=42
    )
)


# ================================================================
# 21. RUNS MODEL
# ================================================================

print()
print("=" * 75)
print("TRAINING BATTER RUNS MODEL")
print("=" * 75)


runs_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=14,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)


runs_model.fit(
    X_train_runs,
    y_train_runs
)


runs_pred = runs_model.predict(
    X_test_runs
)


runs_mae = mean_absolute_error(
    y_test_runs,
    runs_pred
)

runs_rmse = np.sqrt(
    mean_squared_error(
        y_test_runs,
        runs_pred
    )
)


print(
    "Runs MAE :",
    round(
        runs_mae,
        5
    )
)

print(
    "Runs RMSE:",
    round(
        runs_rmse,
        5
    )
)


# ================================================================
# 22. TARGET — BALLS
# ================================================================

# Balls are modelled using log transformation because the
# distribution is strongly skewed.

y_balls = np.log1p(
    features["balls"].astype(float)
)


X_train_balls, X_test_balls, y_train_balls, y_test_balls = (
    train_test_split(
        X,
        y_balls,
        test_size=0.20,
        random_state=42
    )
)


# ================================================================
# 23. BALLS MODEL
# ================================================================

print()
print("=" * 75)
print("TRAINING BALLS-FACED MODEL")
print("=" * 75)


balls_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=14,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)


balls_model.fit(
    X_train_balls,
    y_train_balls
)


balls_pred_log = balls_model.predict(
    X_test_balls
)


balls_pred = np.expm1(
    balls_pred_log
)

balls_actual = np.expm1(
    y_test_balls
)


balls_mae = mean_absolute_error(
    balls_actual,
    balls_pred
)


print(
    "Balls MAE:",
    round(
        balls_mae,
        3
    )
)


# ================================================================
# 24. DISMISSAL TARGET
# ================================================================

# IMPORTANT:
#
# This target means:
#
#   Did this batter-bowler relationship historically contain
#   at least one dismissal?
#
# The model will learn from the matchup and overall history.
#
# It is NOT hard-coded to a particular player.

y_dismissal = (
    features["dismissals"]
    >
    0
).astype(int)


print()
print(
    "Dismissal positive rate:",
    round(
        y_dismissal.mean() * 100,
        2
    ),
    "%"
)


# ================================================================
# 25. DISMISSAL TRAIN / TEST
# ================================================================

X_train_dismissal, X_test_dismissal, y_train_dismissal, y_test_dismissal = (
    train_test_split(
        X,
        y_dismissal,
        test_size=0.20,
        random_state=42,
        stratify=y_dismissal
    )
)


# ================================================================
# 26. DISMISSAL MODEL
# ================================================================

print()
print("=" * 75)
print("TRAINING DISMISSAL MODEL")
print("=" * 75)


dismissal_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=14,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


dismissal_model.fit(
    X_train_dismissal,
    y_train_dismissal
)


dismissal_prob = (
    dismissal_model
    .predict_proba(
        X_test_dismissal
    )[:, 1]
)


try:

    dismissal_auc = roc_auc_score(
        y_test_dismissal,
        dismissal_prob
    )

except Exception:

    dismissal_auc = 0.0


print(
    "Dismissal ROC-AUC:",
    round(
        dismissal_auc,
        4
    )
)


# ================================================================
# 27. SAVE MODELS
# ================================================================

# Save directly beside your existing final model files.

SAVE_DIR = CURRENT_DIR

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


RUNS_MODEL_PATH = os.path.join(
    SAVE_DIR,
    "player_runs_model.pkl"
)

BALLS_MODEL_PATH = os.path.join(
    SAVE_DIR,
    "player_balls_model.pkl"
)

DISMISSAL_MODEL_PATH = os.path.join(
    SAVE_DIR,
    "player_dismissal_model.pkl"
)

PLAYER_FEATURES_PATH = os.path.join(
    SAVE_DIR,
    "player_model_features.pkl"
)

MATCHUP_HISTORY_PATH = os.path.join(
    SAVE_DIR,
    "player_batter_bowler_history.pkl"
)

BATTING_HISTORY_PATH = os.path.join(
    SAVE_DIR,
    "player_batting_history.pkl"
)

BOWLING_HISTORY_PATH = os.path.join(
    SAVE_DIR,
    "player_bowling_history.pkl"
)


joblib.dump(
    runs_model,
    RUNS_MODEL_PATH
)

joblib.dump(
    balls_model,
    BALLS_MODEL_PATH
)

joblib.dump(
    dismissal_model,
    DISMISSAL_MODEL_PATH
)

joblib.dump(
    MODEL_FEATURES,
    PLAYER_FEATURES_PATH
)

joblib.dump(
    pair,
    MATCHUP_HISTORY_PATH
)

joblib.dump(
    batter_stats,
    BATTING_HISTORY_PATH
)

joblib.dump(
    bowler_stats,
    BOWLING_HISTORY_PATH
)


# ================================================================
# 28. SAVE A CLEAN PLAYER PROFILE
# ================================================================

player_profile = batter_stats.copy()

PLAYER_PROFILE_PATH = os.path.join(
    SAVE_DIR,
    "player_profile.pkl"
)

player_profile.to_pickle(
    PLAYER_PROFILE_PATH
)


# ================================================================
# 29. SAVE MATCHUP DATA
# ================================================================

MATCHUP_DF_PATH = os.path.join(
    SAVE_DIR,
    "matchup_df.pkl"
)

pair.to_pickle(
    MATCHUP_DF_PATH
)


# ================================================================
# 30. FINAL REPORT
# ================================================================

print()
print("=" * 75)
print("PLAYER-LEVEL MODELS SUCCESSFULLY CREATED")
print("=" * 75)

print()
print("Historical dataset:")
print(
    selected_file
)

print()
print("Historical rows:")
print(
    len(df)
)

print()
print("Batter-bowler combinations:")
print(
    len(pair)
)

print()
print("Unique batters:")
print(
    df[BATTER_COL].nunique()
)

print()
print("Unique bowlers:")
print(
    df[BOWLER_COL].nunique()
)

print()
print(
    "Runs model MAE:",
    round(
        runs_mae,
        5
    )
)

print(
    "Balls model MAE:",
    round(
        balls_mae,
        3
    )
)

print(
    "Dismissal ROC-AUC:",
    round(
        dismissal_auc,
        4
    )
)

print()
print("=" * 75)
print("FILES SAVED")
print("=" * 75)

print(
    RUNS_MODEL_PATH
)

print(
    BALLS_MODEL_PATH
)

print(
    DISMISSAL_MODEL_PATH
)

print(
    PLAYER_FEATURES_PATH
)

print(
    MATCHUP_HISTORY_PATH
)

print(
    BATTING_HISTORY_PATH
)

print(
    BOWLING_HISTORY_PATH
)

print(
    PLAYER_PROFILE_PATH
)

print(
    MATCHUP_DF_PATH
)

print()
print("=" * 75)
print("READY FOR CUSTOM XI PREDICTION")
print("=" * 75)

BUILDING ACTUAL PLAYER-LEVEL IPL PREDICTION MODELS

Current directory:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks

Candidate data files found:
1. c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\data\processed\batter_bowler_matchups.csv
2. c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\data\processed\player_match_stats.csv
3. c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
4. c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
5. c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
6. c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-1.csv
7. c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-2.csv
8. c:\Users\SAI PAVAN KARTHIK\OneDrive\De

FileNotFoundError: No IPL ball-by-ball dataset containing batter, bowler and batsman_runs columns was found.

In [1]:
import os

print("=" * 70)
print("MODEL FILES")
print("=" * 70)

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith((".pkl", ".joblib")):
            print(os.path.join(root, file))

MODEL FILES
.\custom_xi_balls_model.pkl
.\custom_xi_batter_history.pkl
.\custom_xi_bowler_dismissal_history.pkl
.\custom_xi_dismissal_model.pkl
.\custom_xi_model_features.pkl
.\custom_xi_runs_model.pkl
.\custom_xi_training_lookup.pkl
.\final_feature_columns.pkl
.\final_ipl_match_predictor.pkl
.\final_prediction_threshold.pkl
.\versus_logistic_regression.pkl
.\notebooks\matchup_df.pkl
.\notebooks\player_profile.pkl
